# Нейросети

[Исходный Colab](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/ru_sentiment_research_FINAL.ipynb) · [CNN Kim](https://colab.research.google.com/github/SamDerden/SpeechSense/blob/main/%22ru_sentiment_pipeline_adapted_gzip_robust_ipynb%22.ipynb)

Архивный перенос: блоки восстановления сессий сохранены для проверки, запуск подряд пока не подтверждён. Исторические метрики — reports/historical_metrics.csv.

In [ ]:
import pandas as pd

df = pd.read_csv("/content/merged_sentiment_dataset.csv.gz")

print("Размер датасета:", df.shape)
print("\nКолонки:", df.columns.tolist())
print("\nРаспределение классов:")
print(df["sentiment"].value_counts(dropna=False))

print("\nПервые 5 строк:")
display(df.head())

In [ ]:
print("Разделение данных:")
print(df["split"].value_counts(dropna=False))

print("\nКлассы по выборкам:")
display(pd.crosstab(df["split"], df["sentiment"]))

print("\nИсточники данных:")
print(df["source_dataset"].value_counts(dropna=False))

print("\nПропуски в текстах:")
print(df["text"].isna().sum())

print("\nПолные дубли текстов:")
print(df["text"].duplicated().sum())

In [ ]:
# Проверка пересечений и длины текстов

import pandas as pd

train_df = df[df["split"] == "train"].copy()
val_df = df[df["split"] == "validation"].copy()

# Нормализуем текст для проверки возможных дублей
def normalize_for_check(text):
    return " ".join(str(text).lower().split())

train_normalized = set(
    train_df["text"].map(normalize_for_check)
)

val_normalized = set(
    val_df["text"].map(normalize_for_check)
)

overlap = train_normalized & val_normalized

print("Пересечений Train / Validation:", len(overlap))

print("\nДлины текстов (в словах):")
for name, data in [
    ("Train", train_df),
    ("Validation", val_df)
]:
    lengths = data["text"].str.split().str.len()

    print(f"\n{name}:")
    print("Медиана:", lengths.median())
    print("95-й перцентиль:", lengths.quantile(0.95))
    print("Максимум:", lengths.max())

print("\nРаспределение источников по выборкам:")
display(
    pd.crosstab(
        df["source_dataset"],
        df["split"]
    )
)

In [ ]:
import re
import numpy as np

def tokenize_new(text):
    return re.findall(r"[а-яёa-z0-9]+", str(text).lower())

# Проверяем на обучающей выборке
train_lengths = train_df["text"].map(
    lambda text: len(tokenize_new(text))
)

print("Длины после токенизации:")
print(train_lengths.describe(
    percentiles=[0.5, 0.75, 0.90, 0.95, 0.99]
))

print("\nДоля текстов, которые будут обрезаны:")

for max_len in [70, 128, 256, 384, 512]:
    truncated = (train_lengths > max_len).mean() * 100
    print(f"MAX_LEN={max_len}: {truncated:.2f}%")

In [ ]:
# Проверяем пустые после токенизации и экстремально длинные тексты

print("Пустых после токенизации:", (train_lengths == 0).sum())
print("Короче 3 токенов:", (train_lengths < 3).sum())
print("Длиннее 1000 токенов:", (train_lengths > 1000).sum())

print("\nПримеры самых длинных текстов:")

longest_indices = train_lengths.nlargest(3).index

for idx in longest_indices:
    text = train_df.loc[idx, "text"]

    print("\n" + "=" * 60)
    print("Длина:", train_lengths.loc[idx], "токенов")
    print("Источник:", train_df.loc[idx, "source_dataset"])
    print("Текст (первые 700 символов):")
    print(str(text)[:700])

In [ ]:
# Анализ экстремально длинных текстов

long_df = train_df.loc[train_lengths > 1000].copy()
long_df["tokens"] = train_lengths.loc[long_df.index]

print("Всего длинных текстов:", len(long_df))

print("\nРаспределение по классам:")
display(pd.crosstab(
    long_df["source_dataset"],
    long_df["sentiment"],
    margins=True
))

print("\nПримеры из разных классов:")

for sentiment_id, name in [
    (0, "Neutral"),
    (1, "Positive"),
    (2, "Negative")
]:
    examples = long_df[
        long_df["sentiment"] == sentiment_id
    ].head(2)

    print(f"\n{'=' * 60}")
    print(name)

    for _, row in examples.iterrows():
        print(
            f"\nДлина: {row['tokens']} токенов | "
            f"Источник: {row['source_dataset']}"
        )
        print(str(row["text"])[:400])

In [ ]:
# ============================================================
# ШАГ 7. Подготовка данных для новых нейросетей
# ============================================================

import numpy as np
import pandas as pd

MAX_LEN_NEW = 256
SEED_NEW = 42

# Сохраняем оригинальные выборки
train_new = df[df["split"] == "train"].copy()
val_new = df[df["split"] == "validation"].copy()

# Количество токенов после нашей токенизации
train_new["n_tokens"] = train_new["text"].map(
    lambda x: len(tokenize_new(x))
)

val_new["n_tokens"] = val_new["text"].map(
    lambda x: len(tokenize_new(x))
)

# Удаляем только тексты без распознанных токенов
train_new = train_new[
    train_new["n_tokens"] > 0
].copy()

val_new = val_new[
    val_new["n_tokens"] > 0
].copy()

# Проверяем допустимые метки
assert set(train_new["sentiment"].unique()) == {0, 1, 2}
assert set(val_new["sentiment"].unique()) == {0, 1, 2}

# Тексты и метки
X_train_new = train_new["text"].astype(str).tolist()
y_train_new = train_new["sentiment"].astype(int).to_numpy()

X_val_new = val_new["text"].astype(str).tolist()
y_val_new = val_new["sentiment"].astype(int).to_numpy()

CLASS_NAMES_NEW = ["Neutral", "Positive", "Negative"]

print("Train:", len(X_train_new))
print("Validation:", len(X_val_new))
print("MAX_LEN:", MAX_LEN_NEW)

print("\nРаспределение Train:")
print(train_new["sentiment"].value_counts().sort_index())

print("\nРаспределение Validation:")
print(val_new["sentiment"].value_counts().sort_index())

print("\nУдалено пустых Train:",
      (df["split"] == "train").sum() - len(train_new))

print("Удалено пустых Validation:",
      (df["split"] == "validation").sum() - len(val_new))

### 2.2. Повторное обучение нейросетей на объединённом датасете

Подготовка словаря и DataLoader для TextCNN и BiLSTM. Используется MAX_LEN=256, словарь строится только по Train. Метки классов: 0 — Neutral, 1 — Positive, 2 — Negative.

In [ ]:
# ============================================================
# ШАГ 8. Словарь, Dataset и DataLoader
# ============================================================

import random
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

SEED_NEW = 42

random.seed(SEED_NEW)
np.random.seed(SEED_NEW)
torch.manual_seed(SEED_NEW)

DEVICE_NEW = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Устройство:", DEVICE_NEW)

# ------------------------------------------------------------
# 1. Строим словарь ТОЛЬКО на Train
# ------------------------------------------------------------

MIN_FREQ_NEW = 2
PAD_IDX_NEW = 0
UNK_IDX_NEW = 1

word_counts_new = Counter()

for text in X_train_new:
    word_counts_new.update(tokenize_new(text))

word2idx_new = {
    "<PAD>": PAD_IDX_NEW,
    "<UNK>": UNK_IDX_NEW
}

for word, count in word_counts_new.items():
    if count >= MIN_FREQ_NEW:
        word2idx_new[word] = len(word2idx_new)

VOCAB_SIZE_NEW = len(word2idx_new)

print("Размер словаря:", VOCAB_SIZE_NEW)


# ------------------------------------------------------------
# 2. Dataset
# ------------------------------------------------------------

class SentimentDatasetNew(Dataset):

    def __init__(self, texts, labels):
        self.labels = np.asarray(labels, dtype=np.int64)

        # Токенизируем заранее, чтобы ускорить обучение
        self.encoded = []

        for text in texts:
            tokens = tokenize_new(text)[:MAX_LEN_NEW]

            ids = [
                word2idx_new.get(token, UNK_IDX_NEW)
                for token in tokens
            ]

            self.encoded.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], int(self.labels[idx])


def collate_new(batch):
    texts, labels = zip(*batch)

    max_batch_len = min(
        max(len(text) for text in texts),
        MAX_LEN_NEW
    )

    padded = torch.full(
        (len(texts), max_batch_len),
        PAD_IDX_NEW,
        dtype=torch.long
    )

    for i, ids in enumerate(texts):
        padded[i, :len(ids)] = torch.tensor(
            ids, dtype=torch.long
        )

    return (
        padded,
        torch.tensor(labels, dtype=torch.long)
    )


# ------------------------------------------------------------
# 3. Создаём Dataset
# ------------------------------------------------------------

train_dataset_new = SentimentDatasetNew(
    X_train_new,
    y_train_new
)

val_dataset_new = SentimentDatasetNew(
    X_val_new,
    y_val_new
)


# ------------------------------------------------------------
# 4. DataLoader
# ------------------------------------------------------------

BATCH_SIZE_NEW = 64

train_loader_new = DataLoader(
    train_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=True,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)

val_loader_new = DataLoader(
    val_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=False,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)


# ------------------------------------------------------------
# 5. Проверка
# ------------------------------------------------------------

X_batch, y_batch = next(iter(train_loader_new))

print("Train batches:", len(train_loader_new))
print("Validation batches:", len(val_loader_new))

print("Форма текстов:", tuple(X_batch.shape))
print("Форма меток:", tuple(y_batch.shape))

print("Метки в батче:", torch.unique(y_batch).tolist())
print("MAX_LEN:", MAX_LEN_NEW)

In [ ]:
import torch

print("GPU доступен:", torch.cuda.is_available())
print("Устройство:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

### 2.3. TextCNN — базовая модель на объединённом датасете

In [ ]:
# ============================================================
# ШАГ 10. TEXTCNN BASELINE — ОБУЧЕНИЕ
# ============================================================

import copy
import time

import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import f1_score, accuracy_score, classification_report

# Проверяем GPU и подготовленные данные
assert torch.cuda.is_available(), "GPU не подключён!"
assert "train_loader_new" in globals(), "Сначала запусти шаг 8"
assert "val_loader_new" in globals(), "Сначала запусти шаг 8"

DEVICE_NEW = torch.device("cuda")

# ------------------------------------------------------------
# 1. Архитектура TextCNN
# ------------------------------------------------------------

class TextCNNNew(nn.Module):
    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX_NEW
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                embedding_dim,
                num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

    def forward(self, x):
        x = self.embedding(x)
        x = x.transpose(1, 2)

        features = []

        for conv in self.convs:
            h = F.relu(conv(x))
            h = F.max_pool1d(h, h.size(2)).squeeze(2)
            features.append(h)

        x = torch.cat(features, dim=1)
        x = self.dropout(x)

        return self.fc(x)


# ------------------------------------------------------------
# 2. Создаём модель
# ------------------------------------------------------------

model_textcnn_new = TextCNNNew(
    vocab_size=VOCAB_SIZE_NEW
).to(DEVICE_NEW)

criterion_textcnn_new = nn.CrossEntropyLoss()

optimizer_textcnn_new = torch.optim.AdamW(
    model_textcnn_new.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

print("Устройство:", DEVICE_NEW)
print("Параметров модели:",
      sum(p.numel() for p in model_textcnn_new.parameters()))


# ------------------------------------------------------------
# 3. Функции обучения и валидации
# ------------------------------------------------------------

def train_one_epoch_new(model, loader, optimizer, criterion):
    model.train()

    total_loss = 0.0
    total_examples = 0

    for inputs, labels in loader:
        inputs = inputs.to(DEVICE_NEW, non_blocking=True)
        labels = labels.to(DEVICE_NEW, non_blocking=True)

        optimizer.zero_grad()

        logits = model(inputs)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), max_norm=1.0
        )

        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)

    return total_loss / total_examples


@torch.no_grad()
def evaluate_new(model, loader, criterion):
    model.eval()

    total_loss = 0.0
    total_examples = 0

    all_preds = []
    all_labels = []

    for inputs, labels in loader:
        inputs = inputs.to(DEVICE_NEW, non_blocking=True)
        labels = labels.to(DEVICE_NEW, non_blocking=True)

        logits = model(inputs)
        loss = criterion(logits, labels)

        preds = logits.argmax(dim=1)

        total_loss += loss.item() * labels.size(0)
        total_examples += labels.size(0)

        all_preds.extend(preds.cpu().tolist())
        all_labels.extend(labels.cpu().tolist())

    val_loss = total_loss / total_examples

    macro_f1 = f1_score(
        all_labels, all_preds,
        average="macro",
        labels=[0, 1, 2],
        zero_division=0
    )

    accuracy = accuracy_score(all_labels, all_preds)

    return val_loss, macro_f1, accuracy, all_labels, all_preds


# ------------------------------------------------------------
# 4. Обучение с Early Stopping
# ------------------------------------------------------------

EPOCHS_NEW = 20
PATIENCE_NEW = 5

best_f1_new = -1.0
best_epoch_new = 0
bad_epochs_new = 0

history_textcnn_new = []
best_state_textcnn_new = None

start_time = time.time()

for epoch in range(1, EPOCHS_NEW + 1):

    train_loss = train_one_epoch_new(
        model_textcnn_new,
        train_loader_new,
        optimizer_textcnn_new,
        criterion_textcnn_new
    )

    val_loss, val_f1, val_acc, _, _ = evaluate_new(
        model_textcnn_new,
        val_loader_new,
        criterion_textcnn_new
    )

    history_textcnn_new.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": val_f1,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {val_f1:.4f} | "
        f"Accuracy: {val_acc:.4f}",
        flush=True
    )

    if val_f1 > best_f1_new:
        best_f1_new = val_f1
        best_epoch_new = epoch
        bad_epochs_new = 0

        best_state_textcnn_new = copy.deepcopy(
            model_textcnn_new.state_dict()
        )

        torch.save(
            best_state_textcnn_new,
            "/content/textcnn_merged_best.pt"
        )

    else:
        bad_epochs_new += 1

        if bad_epochs_new >= PATIENCE_NEW:
            print("Early stopping")
            break


# ------------------------------------------------------------
# 5. Итоговая оценка лучшей модели
# ------------------------------------------------------------

model_textcnn_new.load_state_dict(best_state_textcnn_new)

_, final_f1, final_acc, y_true, y_pred = evaluate_new(
    model_textcnn_new,
    val_loader_new,
    criterion_textcnn_new
)

print("\n" + "=" * 55)
print("TEXTCNN — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_new)
print(f"Macro-F1: {final_f1:.4f}")
print(f"Accuracy: {final_acc:.4f}")
print(f"Время обучения: {(time.time() - start_time) / 60:.1f} мин")

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=["Neutral", "Positive", "Negative"],
        digits=4,
        zero_division=0
    )
)

print("\nМодель сохранена:")
print("/content/textcnn_merged_best.pt")

In [ ]:
# ============================================================
# ШАГ 11. Установка и загрузка Navec
# ============================================================

!pip -q install navec

import os
import urllib.request
from navec import Navec

NAVEC_PATH_NEW = "/content/navec_hudlit_v1_12B_500K_300d_100q.tar"

NAVEC_URL_NEW = (
    "https://storage.yandexcloud.net/natasha-navec/packs/"
    "navec_hudlit_v1_12B_500K_300d_100q.tar"
)

if not os.path.exists(NAVEC_PATH_NEW):
    print("Скачиваем Navec...")
    urllib.request.urlretrieve(NAVEC_URL_NEW, NAVEC_PATH_NEW)

print("Загружаем эмбеддинги...")
navec_new = Navec.load(NAVEC_PATH_NEW)

print("Navec успешно загружен!")

# Проверяем покрытие словаря
covered_new = sum(
    1 for word in word2idx_new
    if word in navec_new
)

coverage_new = covered_new / len(word2idx_new) * 100

print(f"Размер словаря: {len(word2idx_new):,}")
print(f"Слов найдено в Navec: {covered_new:,}")
print(f"Покрытие словаря: {coverage_new:.2f}%")
print(f"Размерность эмбеддингов: {navec_new['привет'].shape[0]}")

### 2.4. BiLSTM + Navec + Focal Loss на объединённом датасете

In [ ]:
# ============================================================
# ШАГ 12. BiLSTM + NAVEC + FOCAL LOSS
# ============================================================

import copy
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

assert torch.cuda.is_available(), "Подключи GPU T4"
assert "navec_new" in globals(), "Сначала выполни шаг 11"

DEVICE_NEW = torch.device("cuda")

# ------------------------------------------------------------
# 1. Создаём матрицу предобученных эмбеддингов
# ------------------------------------------------------------

EMBED_DIM_NEW = 300

rng_new = np.random.default_rng(42)

embedding_matrix_new = rng_new.normal(
    loc=0.0,
    scale=0.05,
    size=(VOCAB_SIZE_NEW, EMBED_DIM_NEW)
).astype(np.float32)

embedding_matrix_new[PAD_IDX_NEW] = 0.0

found_embeddings_new = 0

for word, idx in word2idx_new.items():
    if word in navec_new:
        embedding_matrix_new[idx] = navec_new[word]
        found_embeddings_new += 1

print("Матрица эмбеддингов:", embedding_matrix_new.shape)
print("Предобученных векторов:", found_embeddings_new)


# ------------------------------------------------------------
# 2. Архитектура BiLSTM
# ------------------------------------------------------------

class BiLSTMNew(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        hidden_dim=128,
        dropout=0.4,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX_NEW
        )

        self.embedding.weight.data.copy_(
            torch.from_numpy(embedding_matrix)
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x):

        mask = (x != PAD_IDX_NEW)

        embedded = self.embedding(x)

        # Исключаем padding из вычислений LSTM
        lengths = mask.sum(dim=1).cpu()

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths,
            batch_first=True,
            enforce_sorted=False
        )

        packed_output, _ = self.lstm(packed)

        output, _ = nn.utils.rnn.pad_packed_sequence(
            packed_output,
            batch_first=True,
            total_length=x.size(1)
        )

        mask_float = mask.unsqueeze(-1).float()

        pooled = (
            (output * mask_float).sum(dim=1)
            / mask_float.sum(dim=1).clamp(min=1)
        )

        pooled = self.dropout(pooled)

        return self.fc(pooled)


# ------------------------------------------------------------
# 3. Focal Loss
# ------------------------------------------------------------

class FocalLossNew(nn.Module):

    def __init__(self, alpha, gamma=2.0):
        super().__init__()

        self.register_buffer(
            "alpha",
            torch.tensor(alpha, dtype=torch.float32)
        )

        self.gamma = gamma

    def forward(self, logits, targets):

        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        alpha_t = self.alpha[targets]

        loss = (
            alpha_t
            * ((1 - pt) ** self.gamma)
            * ce_loss
        )

        return loss.mean()


# ------------------------------------------------------------
# 4. Создаём модель
# ------------------------------------------------------------

model_bilstm_new = BiLSTMNew(
    embedding_matrix=embedding_matrix_new,
    hidden_dim=128,
    dropout=0.4
).to(DEVICE_NEW)

# Новый порядок классов:
# 0 = Neutral, 1 = Positive, 2 = Negative

criterion_bilstm_new = FocalLossNew(
    alpha=[1.5, 0.75, 1.0],
    gamma=2.0
).to(DEVICE_NEW)

optimizer_bilstm_new = torch.optim.AdamW(
    model_bilstm_new.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

print(
    "Параметров модели:",
    sum(p.numel() for p in model_bilstm_new.parameters())
)


# ------------------------------------------------------------
# 5. Обучение с Early Stopping
# ------------------------------------------------------------

EPOCHS_BILSTM_NEW = 20
PATIENCE_BILSTM_NEW = 5

best_f1_bilstm_new = -1.0
best_epoch_bilstm_new = 0
bad_epochs_bilstm_new = 0

history_bilstm_new = []
best_state_bilstm_new = None

start_time_bilstm_new = time.time()

for epoch in range(1, EPOCHS_BILSTM_NEW + 1):

    train_loss = train_one_epoch_new(
        model_bilstm_new,
        train_loader_new,
        optimizer_bilstm_new,
        criterion_bilstm_new
    )

    val_loss, val_f1, val_acc, _, _ = evaluate_new(
        model_bilstm_new,
        val_loader_new,
        criterion_bilstm_new
    )

    history_bilstm_new.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_macro_f1": val_f1,
        "val_accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {val_f1:.4f} | "
        f"Accuracy: {val_acc:.4f}",
        flush=True
    )

    if val_f1 > best_f1_bilstm_new:

        best_f1_bilstm_new = val_f1
        best_epoch_bilstm_new = epoch
        bad_epochs_bilstm_new = 0

        best_state_bilstm_new = copy.deepcopy(
            model_bilstm_new.state_dict()
        )

        torch.save(
            best_state_bilstm_new,
            "/content/bilstm_navec_focal_merged_best.pt"
        )

    else:
        bad_epochs_bilstm_new += 1

        if bad_epochs_bilstm_new >= PATIENCE_BILSTM_NEW:
            print("Early stopping")
            break


# ------------------------------------------------------------
# 6. Итоговая оценка лучшей модели
# ------------------------------------------------------------

model_bilstm_new.load_state_dict(best_state_bilstm_new)

_, final_f1, final_acc, y_true, y_pred = evaluate_new(
    model_bilstm_new,
    val_loader_new,
    criterion_bilstm_new
)

print("\n" + "=" * 55)
print("BiLSTM + NAVEC + FOCAL — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_bilstm_new)
print(f"Macro-F1: {final_f1:.4f}")
print(f"Accuracy: {final_acc:.4f}")
print(
    f"Время обучения: "
    f"{(time.time() - start_time_bilstm_new) / 60:.1f} мин"
)

print("\nClassification Report:")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=[
            "Neutral",
            "Positive",
            "Negative"
        ],
        digits=4,
        zero_division=0
    )
)

print("\nМодель сохранена:")
print("/content/bilstm_navec_focal_merged_best.pt")

## Эксперимент 2. TextCNN + веса классов (Class Weights)


In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 2 — TEXTCNN + CLASS WEIGHTS
# Объединённый датасет | Полная самостоятельная ячейка
# ============================================================

import os
import time
import copy
import random

import numpy as np
import torch
import torch.nn as nn

from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

# ============================================================
# 1. ПРОВЕРКА ПОДГОТОВЛЕННЫХ ДАННЫХ
# ============================================================

required = [
    "TextCNNNew",
    "VOCAB_SIZE_NEW",
    "DEVICE_NEW",
    "y_train_new",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Не найдены переменные: "
        + ", ".join(missing)
        + "\nСначала запусти ячейки подготовки данных."
    )

# ============================================================
# 2. ФИКСИРУЕМ RANDOM SEED
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(DEVICE_NEW)

print("Устройство:", device)

# ============================================================
# 3. РАССЧИТЫВАЕМ ВЕСА КЛАССОВ
# ============================================================

classes = np.array([0, 1, 2])

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=np.asarray(y_train_new)
)

class_weights_tensor = torch.tensor(
    class_weights,
    dtype=torch.float32,
    device=device
)

print("\n" + "=" * 55)
print("ВЕСА КЛАССОВ")
print("=" * 55)

for i, weight in enumerate(class_weights):
    print(f"{CLASS_NAMES_NEW[i]}: {weight:.4f}")

# ============================================================
# 4. СОЗДАЁМ TEXTCNN
# ============================================================

model_cw_new = TextCNNNew(
    vocab_size=VOCAB_SIZE_NEW,
    embedding_dim=128,
    num_filters=128,
    kernel_sizes=(3, 4, 5),
    dropout=0.5,
    num_classes=3
).to(device)

print("\nМодель успешно создана!")

print(
    "Количество параметров:",
    sum(p.numel() for p in model_cw_new.parameters())
)

# ============================================================
# 5. LOSS И OPTIMIZER
# ============================================================

criterion = nn.CrossEntropyLoss(
    weight=class_weights_tensor
)

optimizer = torch.optim.AdamW(
    model_cw_new.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

# ============================================================
# 6. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_cw(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        logits = model(inputs)

        loss = criterion(logits, labels)

        loss.backward()

        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

# ============================================================
# 7. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_epoch_cw(model, loader, criterion):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        logits = model(inputs)

        loss = criterion(logits, labels)

        predictions = torch.argmax(logits, dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(labels.cpu().numpy())
        all_pred.extend(predictions.cpu().numpy())

    avg_loss = total_loss / total_samples

    return (
        avg_loss,
        np.asarray(all_true),
        np.asarray(all_pred)
    )

# ============================================================
# 8. НАСТРОЙКИ ОБУЧЕНИЯ
# ============================================================

EPOCHS = 20
PATIENCE = 5

best_f1 = -1.0
best_epoch = 0
best_state = None
patience_counter = 0

history_cw_new = []

start_time = time.time()

print("\n" + "=" * 55)
print("НАЧИНАЕМ ОБУЧЕНИЕ TEXTCNN + CLASS WEIGHTS")
print("=" * 55)

# ============================================================
# 9. ОБУЧЕНИЕ
# ============================================================

for epoch in range(1, EPOCHS + 1):

    train_loss = train_epoch_cw(
        model_cw_new,
        train_loader_new,
        optimizer,
        criterion
    )

    val_loss, y_true, y_pred = evaluate_epoch_cw(
        model_cw_new,
        val_loader_new,
        criterion
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(y_true, y_pred)

    history_cw_new.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1:

        best_f1 = macro_f1
        best_epoch = epoch

        # Храним лучший checkpoint на CPU
        best_state = {
            key: value.detach().cpu().clone()
            for key, value in model_cw_new.state_dict().items()
        }

        patience_counter = 0

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE:
        print("\nEarly stopping")
        break

# ============================================================
# 10. ВОССТАНАВЛИВАЕМ ЛУЧШУЮ МОДЕЛЬ
# ============================================================

model_cw_new.load_state_dict(best_state)
model_cw_new.eval()

# ============================================================
# 11. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

val_loss, y_true_cw_new, y_pred_cw_new = evaluate_epoch_cw(
    model_cw_new,
    val_loader_new,
    criterion
)

final_f1_cw_new = f1_score(
    y_true_cw_new,
    y_pred_cw_new,
    average="macro",
    zero_division=0
)

final_accuracy_cw_new = accuracy_score(
    y_true_cw_new,
    y_pred_cw_new
)

elapsed = (time.time() - start_time) / 60

print("\n" + "=" * 55)
print("TEXTCNN + CLASS WEIGHTS — ИТОГИ")
print("=" * 55)

print(f"Лучшая эпоха: {best_epoch}")
print(f"Macro-F1: {final_f1_cw_new:.4f}")
print(f"Accuracy: {final_accuracy_cw_new:.4f}")
print(f"Время обучения: {elapsed:.1f} мин")

print("\nClassification Report:")

print(
    classification_report(
        y_true_cw_new,
        y_pred_cw_new,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

# ============================================================
# 12. СОХРАНЯЕМ МОДЕЛЬ И ИСТОРИЮ ОБУЧЕНИЯ
# ============================================================

import pandas as pd

save_path_cw_new = (
    "/content/textcnn_class_weights_merged_best.pt"
)

history_path_cw_new = (
    "/content/textcnn_class_weights_history.csv"
)

torch.save(
    model_cw_new.state_dict(),
    save_path_cw_new
)

pd.DataFrame(history_cw_new).to_csv(
    history_path_cw_new,
    index=False
)

print("\nМодель сохранена:", save_path_cw_new)
print("История обучения:", history_path_cw_new)

# ============================================================
# 13. СРАВНЕНИЕ С BASELINE
# ============================================================

baseline_f1 = 0.7335
difference = final_f1_cw_new - baseline_f1

print("\n" + "=" * 55)
print("СРАВНЕНИЕ С TEXTCNN BASELINE")
print("=" * 55)

print(f"TextCNN baseline:        {baseline_f1:.4f}")
print(f"TextCNN + Class Weights: {final_f1_cw_new:.4f}")
print(f"Разница Macro-F1:        {difference:+.4f}")

if difference > 0:
    print("\nРезультат: веса классов улучшили Macro-F1.")
elif difference < 0:
    print("\nРезультат: веса классов не улучшили Macro-F1.")
else:
    print("\nРезультат: Macro-F1 не изменился.")

## Эксперимент 3. TextCNN + аугментация Neutral и Negative

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 3. TEXTCNN + AUGMENTATION
# ============================================================

import random
import time
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report

# 1. Проверка данных

required = [
    "train_new", "word2idx_new", "MAX_LEN_NEW",
    "VOCAB_SIZE_NEW", "DEVICE_NEW", "TextCNNNew",
    "val_loader_new", "CLASS_NAMES_NEW"
]

missing = [x for x in required if x not in globals()]
if missing:
    raise RuntimeError("Не найдены переменные: " + ", ".join(missing))

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device(DEVICE_NEW)

# 2. Аугментация: удаление одного случайного слова

rng = random.Random(seed)

def augment_text_exp3(text):
    words = str(text).split()
    if len(words) < 8:
        return str(text)

    idx = rng.randrange(len(words))
    return " ".join(words[:idx] + words[idx + 1:])

original = train_new[["text", "sentiment"]].copy()
original = original.reset_index(drop=True)

print("Исходное распределение:")
print(original["sentiment"].value_counts().sort_index())

extra_rows = []

for label in [0, 2]:
    subset = original[original["sentiment"] == label]
    count = int(len(subset) * 0.20)

    sampled = subset.sample(
        n=count,
        random_state=seed,
        replace=False
    )

    for text in sampled["text"]:
        extra_rows.append({
            "text": augment_text_exp3(text),
            "sentiment": label
        })

augmented = pd.DataFrame(extra_rows)

train_aug_new = pd.concat(
    [original, augmented],
    ignore_index=True
)

train_aug_new = train_aug_new.sample(
    frac=1,
    random_state=seed
).reset_index(drop=True)

print("\nПосле аугментации:")
print(train_aug_new["sentiment"].value_counts().sort_index())
print("Всего:", len(train_aug_new))

# 3. Подготовка Dataset

pad_id = word2idx_new.get("<PAD>", 0)
unk_id = word2idx_new.get("<UNK>", 1)

class AugDatasetExp3(Dataset):
    def __init__(self, texts, labels):
        self.labels = [int(x) for x in labels]
        self.sequences = []

        for text in texts:
            tokens = re.findall(
                r"[а-яёa-z0-9]+",
                str(text).lower()
            )[:MAX_LEN_NEW]

            ids = [word2idx_new.get(t, unk_id) for t in tokens]
            self.sequences.append(ids if ids else [unk_id])

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def collate_exp3(batch):
    sequences, labels = zip(*batch)
    max_len = max(len(x) for x in sequences)

    inputs = torch.full(
        (len(sequences), max_len),
        pad_id,
        dtype=torch.long
    )

    for i, seq in enumerate(sequences):
        inputs[i, :len(seq)] = torch.tensor(seq)

    return inputs, torch.tensor(labels, dtype=torch.long)


train_dataset_aug_new = AugDatasetExp3(
    train_aug_new["text"].tolist(),
    train_aug_new["sentiment"].tolist()
)

train_loader_aug_new = DataLoader(
    train_dataset_aug_new,
    batch_size=64,
    shuffle=True,
    collate_fn=collate_exp3,
    num_workers=0
)

print("Батчей:", len(train_loader_aug_new))

# 4. Модель

model_aug_new = TextCNNNew(
    vocab_size=VOCAB_SIZE_NEW,
    embedding_dim=128,
    num_filters=128,
    kernel_sizes=(3, 4, 5),
    dropout=0.5,
    num_classes=3
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(
    model_aug_new.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

# 5. Обучение и валидация

def train_exp3():
    model_aug_new.train()
    total_loss = 0.0
    total_count = 0

    for inputs, labels in train_loader_aug_new:
        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model_aug_new(inputs)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        n = len(labels)
        total_loss += loss.item() * n
        total_count += n

    return total_loss / total_count


@torch.no_grad()
def validate_exp3():
    model_aug_new.eval()
    total_loss = 0.0
    total_count = 0
    all_true = []
    all_pred = []

    for inputs, labels in val_loader_new:
        inputs = inputs.to(device)
        labels = labels.to(device)

        logits = model_aug_new(inputs)
        loss = criterion(logits, labels)
        predictions = logits.argmax(dim=1)

        n = len(labels)
        total_loss += loss.item() * n
        total_count += n

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

    return (
        total_loss / total_count,
        np.asarray(all_true),
        np.asarray(all_pred)
    )

# 6. Запуск

best_f1_aug_new = -1.0
best_epoch_aug_new = 0
best_state_aug_new = None
patience_counter = 0

history_aug_new = []
start = time.time()

for epoch in range(1, 21):
    train_loss = train_exp3()
    val_loss, y_true, y_pred = validate_exp3()

    macro_f1 = f1_score(
        y_true, y_pred,
        average="macro",
        zero_division=0
    )
    accuracy = accuracy_score(y_true, y_pred)

    history_aug_new.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1_aug_new:
        best_f1_aug_new = macro_f1
        best_epoch_aug_new = epoch

        best_state_aug_new = {
            k: v.detach().cpu().clone()
            for k, v in model_aug_new.state_dict().items()
        }
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= 5:
        print("Early stopping")
        break

# 7. Итоги

model_aug_new.load_state_dict(best_state_aug_new)

_, y_true_aug_new, y_pred_aug_new = validate_exp3()

final_f1_aug_new = f1_score(
    y_true_aug_new,
    y_pred_aug_new,
    average="macro",
    zero_division=0
)

final_acc_aug_new = accuracy_score(
    y_true_aug_new,
    y_pred_aug_new
)

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 3 — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_aug_new)
print(f"Macro-F1: {final_f1_aug_new:.4f}")
print(f"Accuracy: {final_acc_aug_new:.4f}")
print(f"Время: {(time.time() - start) / 60:.1f} мин")

print("\nClassification Report:")
print(classification_report(
    y_true_aug_new,
    y_pred_aug_new,
    labels=[0, 1, 2],
    target_names=CLASS_NAMES_NEW,
    digits=4,
    zero_division=0
))

# 8. Сохранение

model_path_aug_new = "/content/textcnn_augmentation_merged_best.pt"
history_path_aug_new = "/content/textcnn_augmentation_history.csv"

torch.save(model_aug_new.state_dict(), model_path_aug_new)
pd.DataFrame(history_aug_new).to_csv(
    history_path_aug_new,
    index=False
)

print("\nМодель сохранена:", model_path_aug_new)
print("История сохранена:", history_path_aug_new)

print("\nСРАВНЕНИЕ:")
print("TextCNN baseline:        0.7335")
print("TextCNN + Class Weights: 0.7314")
print(f"TextCNN + Augmentation:  {final_f1_aug_new:.4f}")

В эксперименте с аугментацией обучающая выборка была дополнена примерами миноритарных классов. Количество объектов класса Neutral было увеличено с 46 184 до 55 420, класса Negative — с 53 107 до 63 728. Лучшее значение Macro-F1 составило 0.7351 на 4-й эпохе, что немного превышает baseline TextCNN (0.7335) и вариант с использованием class weights (0.7314). Наиболее сложным для классификации остаётся класс Neutral с F1-score 0.6373. После 4-й эпохи наблюдается переобучение, поэтому была выбрана модель с лучшим validation Macro-F1.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Названия классов
class_names = ["Neutral", "Positive", "Negative"]

# =========================
# 1. CONFUSION MATRIX
# =========================

cm = confusion_matrix(y_true, y_pred)

print("Confusion Matrix:")
print(cm)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, values_format="d")
plt.title("TextCNN + Augmentation — Confusion Matrix")
plt.show()


# =========================
# 2. НОРМАЛИЗОВАННАЯ MATRIX
# =========================

cm_norm = confusion_matrix(
    y_true,
    y_pred,
    normalize="true"
)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_norm,
    display_labels=class_names
)

fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, values_format=".2f")
plt.title("Normalized Confusion Matrix")
plt.show()


# =========================
# 3. ТАБЛИЦА ОШИБОК
# =========================

errors_df = pd.DataFrame({
    "true": y_true,
    "pred": y_pred
})

errors_df["true_name"] = errors_df["true"].map({
    0: "Neutral",
    1: "Positive",
    2: "Negative"
})

errors_df["pred_name"] = errors_df["pred"].map({
    0: "Neutral",
    1: "Positive",
    2: "Negative"
})

errors_only = errors_df[
    errors_df["true"] != errors_df["pred"]
].copy()

print("\nКоличество ошибок по направлениям:")
print(
    errors_only
    .groupby(["true_name", "pred_name"])
    .size()
    .sort_values(ascending=False)
)


# =========================
# 4. КУДА УЛЕТАЕТ NEUTRAL
# =========================

neutral_errors = errors_only[
    errors_only["true"] == 0
]

print("\nОшибки Neutral:")
print(neutral_errors["pred_name"].value_counts())

# Эксперимент 4. TextCNN + Augmentation + Focal Loss

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 4. TEXTCNN + AUGMENTATION + FOCAL LOSS
# ============================================================

import random
import time
import re
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score, accuracy_score, classification_report


# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required = [
    "train_new",
    "word2idx_new",
    "MAX_LEN_NEW",
    "VOCAB_SIZE_NEW",
    "DEVICE_NEW",
    "TextCNNNew",
    "val_loader_new",
    "CLASS_NAMES_NEW"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing)
    )


seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device(DEVICE_NEW)


# ============================================================
# 2. АУГМЕНТАЦИЯ
# Удаляем одно случайное слово
# ============================================================

rng = random.Random(seed)


def augment_text_exp4(text):

    words = str(text).split()

    if len(words) < 8:
        return str(text)

    idx = rng.randrange(len(words))

    return " ".join(
        words[:idx] + words[idx + 1:]
    )


original_exp4 = train_new[
    ["text", "sentiment"]
].copy()

original_exp4 = original_exp4.reset_index(
    drop=True
)


print("Исходное распределение:")

print(
    original_exp4[
        "sentiment"
    ].value_counts().sort_index()
)


extra_rows_exp4 = []


# Аугментируем классы 0 и 2 на 20%
for label in [0, 2]:

    subset = original_exp4[
        original_exp4["sentiment"] == label
    ]

    count = int(
        len(subset) * 0.20
    )

    sampled = subset.sample(
        n=count,
        random_state=seed,
        replace=False
    )

    for text in sampled["text"]:

        extra_rows_exp4.append({
            "text": augment_text_exp4(text),
            "sentiment": label
        })


augmented_exp4 = pd.DataFrame(
    extra_rows_exp4
)


train_aug_exp4 = pd.concat(
    [
        original_exp4,
        augmented_exp4
    ],
    ignore_index=True
)


train_aug_exp4 = train_aug_exp4.sample(
    frac=1,
    random_state=seed
).reset_index(
    drop=True
)


print("\nПосле аугментации:")

print(
    train_aug_exp4[
        "sentiment"
    ].value_counts().sort_index()
)

print(
    "Всего:",
    len(train_aug_exp4)
)


# ============================================================
# 3. DATASET
# ============================================================

pad_id = word2idx_new.get(
    "<PAD>",
    0
)

unk_id = word2idx_new.get(
    "<UNK>",
    1
)


class AugDatasetExp4(Dataset):

    def __init__(
        self,
        texts,
        labels
    ):

        self.labels = [
            int(x)
            for x in labels
        ]

        self.sequences = []

        for text in texts:

            tokens = re.findall(
                r"[а-яёa-z0-9]+",
                str(text).lower()
            )[:MAX_LEN_NEW]

            ids = [
                word2idx_new.get(
                    token,
                    unk_id
                )
                for token in tokens
            ]

            if len(ids) == 0:
                ids = [unk_id]

            self.sequences.append(
                ids
            )

    def __len__(self):

        return len(
            self.labels
        )

    def __getitem__(
        self,
        idx
    ):

        return (
            self.sequences[idx],
            self.labels[idx]
        )


# ============================================================
# 4. COLLATE
# ============================================================

def collate_exp4(batch):

    sequences, labels = zip(
        *batch
    )

    max_len = max(
        len(x)
        for x in sequences
    )

    inputs = torch.full(
        (
            len(sequences),
            max_len
        ),
        pad_id,
        dtype=torch.long
    )

    for i, seq in enumerate(
        sequences
    ):

        inputs[
            i,
            :len(seq)
        ] = torch.tensor(
            seq,
            dtype=torch.long
        )

    labels = torch.tensor(
        labels,
        dtype=torch.long
    )

    return (
        inputs,
        labels
    )


# ============================================================
# 5. DATALOADER
# ============================================================

train_dataset_exp4 = AugDatasetExp4(

    train_aug_exp4[
        "text"
    ].tolist(),

    train_aug_exp4[
        "sentiment"
    ].tolist()
)


train_loader_exp4 = DataLoader(

    train_dataset_exp4,

    batch_size=64,

    shuffle=True,

    collate_fn=collate_exp4,

    num_workers=0
)


print(
    "Батчей:",
    len(train_loader_exp4)
)


# ============================================================
# 6. FOCAL LOSS
# ============================================================

class FocalLoss(nn.Module):

    def __init__(
        self,
        alpha=None,
        gamma=2.0,
        reduction="mean"
    ):

        super().__init__()

        self.gamma = gamma

        self.reduction = reduction


        if alpha is not None:

            self.register_buffer(

                "alpha",

                torch.tensor(
                    alpha,
                    dtype=torch.float32
                )
            )

        else:

            self.alpha = None


    def forward(
        self,
        logits,
        targets
    ):

        ce_loss = F.cross_entropy(

            logits,

            targets,

            reduction="none"
        )


        pt = torch.exp(
            -ce_loss
        )


        focal_loss = (

            (1 - pt)
            ** self.gamma

        ) * ce_loss


        if self.alpha is not None:

            alpha_t = self.alpha[
                targets
            ]

            focal_loss = (

                alpha_t
                * focal_loss
            )


        if self.reduction == "mean":

            return focal_loss.mean()


        if self.reduction == "sum":

            return focal_loss.sum()


        return focal_loss


# ============================================================
# 7. НОВАЯ МОДЕЛЬ
# ============================================================

model_focal_new = TextCNNNew(

    vocab_size=VOCAB_SIZE_NEW,

    embedding_dim=128,

    num_filters=128,

    kernel_sizes=(
        3,
        4,
        5
    ),

    dropout=0.5,

    num_classes=3

).to(
    device
)


# ============================================================
# 8. LOSS
# ============================================================

criterion_focal = FocalLoss(

    alpha=[
        1.2,   # Neutral
        1.0,   # Positive
        1.1    # Negative
    ],

    gamma=2.0

).to(
    device
)


# ============================================================
# 9. OPTIMIZER
# ============================================================

optimizer_focal = torch.optim.AdamW(

    model_focal_new.parameters(),

    lr=5e-4,

    weight_decay=1e-4
)


# ============================================================
# 10. ОБУЧЕНИЕ
# ============================================================

def train_exp4():

    model_focal_new.train()

    total_loss = 0.0

    total_count = 0


    for inputs, labels in train_loader_exp4:

        inputs = inputs.to(
            device
        )

        labels = labels.to(
            device
        )


        optimizer_focal.zero_grad()


        logits = model_focal_new(
            inputs
        )


        loss = criterion_focal(

            logits,

            labels
        )


        loss.backward()


        optimizer_focal.step()


        n = len(
            labels
        )


        total_loss += (

            loss.item()
            * n
        )


        total_count += n


    return (

        total_loss
        / total_count
    )


# ============================================================
# 11. ВАЛИДАЦИЯ
# ============================================================

@torch.no_grad()
def validate_exp4():

    model_focal_new.eval()

    total_loss = 0.0

    total_count = 0

    all_true = []

    all_pred = []


    for inputs, labels in val_loader_new:

        inputs = inputs.to(
            device
        )

        labels = labels.to(
            device
        )


        logits = model_focal_new(
            inputs
        )


        loss = criterion_focal(

            logits,

            labels
        )


        predictions = logits.argmax(
            dim=1
        )


        n = len(
            labels
        )


        total_loss += (

            loss.item()
            * n
        )


        total_count += n


        all_true.extend(

            labels.cpu().tolist()
        )


        all_pred.extend(

            predictions.cpu().tolist()
        )


    return (

        total_loss / total_count,

        np.asarray(
            all_true
        ),

        np.asarray(
            all_pred
        )
    )


# ============================================================
# 12. ЗАПУСК ОБУЧЕНИЯ
# ============================================================

best_f1_focal_new = -1.0

best_epoch_focal_new = 0

best_state_focal_new = None

patience_counter = 0

history_focal_new = []

start = time.time()


for epoch in range(
    1,
    21
):

    train_loss = train_exp4()

    (
        val_loss,
        y_true,
        y_pred
    ) = validate_exp4()


    macro_f1 = f1_score(

        y_true,

        y_pred,

        average="macro",

        zero_division=0
    )


    accuracy = accuracy_score(

        y_true,

        y_pred
    )


    history_focal_new.append({

        "epoch": epoch,

        "train_loss": train_loss,

        "val_loss": val_loss,

        "macro_f1": macro_f1,

        "accuracy": accuracy
    })


    print(

        f"Epoch {epoch:02d} | "

        f"Train Loss: {train_loss:.4f} | "

        f"Val Loss: {val_loss:.4f} | "

        f"Macro-F1: {macro_f1:.4f} | "

        f"Accuracy: {accuracy:.4f}",

        flush=True
    )


    if macro_f1 > best_f1_focal_new:

        best_f1_focal_new = macro_f1

        best_epoch_focal_new = epoch


        best_state_focal_new = {

            k: v.detach().cpu().clone()

            for k, v
            in model_focal_new.state_dict().items()
        }


        patience_counter = 0


    else:

        patience_counter += 1


    if patience_counter >= 5:

        print(
            "Early stopping"
        )

        break


# ============================================================
# 13. ВОЗВРАЩАЕМ ЛУЧШУЮ МОДЕЛЬ
# ============================================================

model_focal_new.load_state_dict(

    best_state_focal_new
)


# ============================================================
# 14. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

(
    _,
    y_true_focal_new,
    y_pred_focal_new

) = validate_exp4()


final_f1_focal_new = f1_score(

    y_true_focal_new,

    y_pred_focal_new,

    average="macro",

    zero_division=0
)


final_acc_focal_new = accuracy_score(

    y_true_focal_new,

    y_pred_focal_new
)


# ============================================================
# 15. ИТОГИ
# ============================================================

print(
    "\n" + "=" * 55
)

print(
    "ЭКСПЕРИМЕНТ 4 — ИТОГИ"
)

print(
    "=" * 55
)


print(
    "Лучшая эпоха:",
    best_epoch_focal_new
)


print(
    f"Macro-F1: "
    f"{final_f1_focal_new:.4f}"
)


print(
    f"Accuracy: "
    f"{final_acc_focal_new:.4f}"
)


print(
    f"Время: "
    f"{(time.time() - start) / 60:.1f} мин"
)


print(
    "\nClassification Report:"
)


print(
    classification_report(

        y_true_focal_new,

        y_pred_focal_new,

        labels=[
            0,
            1,
            2
        ],

        target_names=CLASS_NAMES_NEW,

        digits=4,

        zero_division=0
    )
)


# ============================================================
# 16. СОХРАНЕНИЕ
# ============================================================

model_path_focal_new = (
    "/content/"
    "textcnn_augmentation_focal_best.pt"
)


history_path_focal_new = (
    "/content/"
    "textcnn_augmentation_focal_history.csv"
)


torch.save(

    model_focal_new.state_dict(),

    model_path_focal_new
)


pd.DataFrame(

    history_focal_new

).to_csv(

    history_path_focal_new,

    index=False
)


print(
    "\nМодель сохранена:",
    model_path_focal_new
)


print(
    "История сохранена:",
    history_path_focal_new
)


# ============================================================
# 17. СРАВНЕНИЕ
# ============================================================

print(
    "\nСРАВНЕНИЕ:"
)


print(
    "TextCNN baseline:                 0.7335"
)


print(
    "TextCNN + Class Weights:          0.7314"
)


print(
    "TextCNN + Augmentation:           0.7351"
)


print(
    f"TextCNN + Augmentation + Focal:   "
    f"{final_f1_focal_new:.4f}"
)

## Эксперимент 5. TextCNN + Navec (без Focal Loss)

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 5. TEXTCNN + NAVEC
# Новый объединённый датасет
# Без аугментации, Class Weights и Focal Loss
# ============================================================

import os
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ПОДГОТОВЛЕННЫХ ДАННЫХ
# ============================================================

required = [
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "DEVICE_NEW",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW",
    "navec_new"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing)
    )

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device(DEVICE_NEW)

print("Устройство:", device)

# ============================================================
# 2. ПОДГОТОВКА ЭМБЕДДИНГОВ NAVEC
# ============================================================

EMBEDDING_DIM = 300
PAD_IDX = 0

assert len(word2idx_new) == VOCAB_SIZE_NEW, (
    "Размер словаря не совпадает с VOCAB_SIZE_NEW"
)

rng = np.random.default_rng(SEED)

embedding_matrix = rng.normal(
    loc=0.0,
    scale=0.05,
    size=(VOCAB_SIZE_NEW, EMBEDDING_DIM)
).astype(np.float32)

embedding_matrix[PAD_IDX] = 0.0

found_words = 0

print("\nПодготавливаем эмбеддинги Navec...")

for word, idx in word2idx_new.items():

    if idx == PAD_IDX:
        continue

    if word in navec_new:
        embedding_matrix[idx] = navec_new[word]
        found_words += 1

coverage = found_words / VOCAB_SIZE_NEW * 100

print("\n" + "=" * 55)
print("NAVEC — ПОКРЫТИЕ СЛОВАРЯ")
print("=" * 55)

print("Размер словаря:", VOCAB_SIZE_NEW)
print("Найдено в Navec:", found_words)
print(f"Покрытие: {coverage:.2f}%")
print("Размерность эмбеддингов:", EMBEDDING_DIM)

embedding_tensor = torch.from_numpy(embedding_matrix)

# ============================================================
# 3. МОДЕЛЬ TEXTCNN + NAVEC
# ============================================================

class TextCNNNavecExp5(nn.Module):

    def __init__(
        self,
        embedding_weights,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_weights.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX
        )

        with torch.no_grad():
            self.embedding.weight.copy_(embedding_weights)

        # Эмбеддинги обучаемые
        self.embedding.weight.requires_grad = True

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

        self.min_seq_len = max(kernel_sizes)

    def forward(self, x):

        # x: [batch, sequence_length]

        x = self.embedding(x)

        # [batch, sequence_length, embedding_dim]
        x = x.transpose(1, 2)

        # [batch, embedding_dim, sequence_length]

        # Защита от коротких последовательностей
        if x.size(2) < self.min_seq_len:
            x = F.pad(
                x,
                (0, self.min_seq_len - x.size(2))
            )

        pooled_outputs = []

        for conv in self.convs:

            h = F.relu(conv(x))

            pooled = F.max_pool1d(
                h,
                kernel_size=h.size(2)
            ).squeeze(2)

            pooled_outputs.append(pooled)

        x = torch.cat(pooled_outputs, dim=1)

        x = self.dropout(x)

        return self.fc(x)


model_navec_new = TextCNNNavecExp5(
    embedding_weights=embedding_tensor,
    num_filters=128,
    kernel_sizes=(3, 4, 5),
    dropout=0.5,
    num_classes=3
).to(device)

del embedding_matrix
del embedding_tensor

print("\nМодель создана.")

print(
    "Количество параметров:",
    sum(p.numel() for p in model_navec_new.parameters())
)

# ============================================================
# 4. LOSS И OPTIMIZER
# ============================================================

# Обычная CrossEntropyLoss
# Без Class Weights и Focal Loss

criterion_navec_new = nn.CrossEntropyLoss()

optimizer_navec_new = torch.optim.AdamW(
    model_navec_new.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

# ============================================================
# 5. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_navec_exp5(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        optimizer_navec_new.zero_grad()

        logits = model(inputs)

        loss = criterion_navec_new(
            logits,
            labels
        )

        loss.backward()

        optimizer_navec_new.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 6. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_navec_exp5(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(device)
        labels = labels.to(device)

        logits = model(inputs)

        loss = criterion_navec_new(
            logits,
            labels
        )

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(
            labels.cpu().numpy()
        )

        all_pred.extend(
            predictions.cpu().numpy()
        )

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )


# ============================================================
# 7. НАСТРОЙКИ ОБУЧЕНИЯ
# ============================================================

EPOCHS_NAVEC_NEW = 20
PATIENCE_NAVEC_NEW = 5

best_f1_navec_new = -1.0
best_epoch_navec_new = 0
best_state_navec_new = None

patience_counter = 0

history_navec_new = []

start_time = time.time()

print("\n" + "=" * 55)
print("НАЧИНАЕМ ОБУЧЕНИЕ TEXTCNN + NAVEC")
print("=" * 55)

# ============================================================
# 8. ОБУЧЕНИЕ
# ============================================================

for epoch in range(1, EPOCHS_NAVEC_NEW + 1):

    train_loss = train_epoch_navec_exp5(
        model_navec_new,
        train_loader_new
    )

    val_loss, y_true, y_pred = evaluate_navec_exp5(
        model_navec_new,
        val_loader_new
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_navec_new.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1_navec_new:

        best_f1_navec_new = macro_f1
        best_epoch_navec_new = epoch

        best_state_navec_new = {
            key: value.detach().cpu().clone()
            for key, value in model_navec_new.state_dict().items()
        }

        patience_counter = 0

    else:
        patience_counter += 1

    if patience_counter >= PATIENCE_NAVEC_NEW:
        print("\nEarly stopping")
        break

# ============================================================
# 9. ВОССТАНАВЛИВАЕМ ЛУЧШУЮ МОДЕЛЬ
# ============================================================

model_navec_new.load_state_dict(
    best_state_navec_new
)

model_navec_new.eval()

# ============================================================
# 10. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

_, y_true_navec_new, y_pred_navec_new = evaluate_navec_exp5(
    model_navec_new,
    val_loader_new
)

final_f1_navec_new = f1_score(
    y_true_navec_new,
    y_pred_navec_new,
    average="macro",
    zero_division=0
)

final_accuracy_navec_new = accuracy_score(
    y_true_navec_new,
    y_pred_navec_new
)

elapsed = (time.time() - start_time) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 5 — TEXTCNN + NAVEC")
print("=" * 55)

print(f"Лучшая эпоха: {best_epoch_navec_new}")
print(f"Macro-F1: {final_f1_navec_new:.4f}")
print(f"Accuracy: {final_accuracy_navec_new:.4f}")
print(f"Время обучения: {elapsed:.1f} мин")

print("\nClassification Report:")

print(
    classification_report(
        y_true_navec_new,
        y_pred_navec_new,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true_navec_new,
        y_pred_navec_new,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 11. СОХРАНЕНИЕ
# ============================================================

model_path_navec_new = (
    "/content/textcnn_navec_merged_best.pt"
)

history_path_navec_new = (
    "/content/textcnn_navec_history.csv"
)

torch.save(
    model_navec_new.state_dict(),
    model_path_navec_new
)

pd.DataFrame(history_navec_new).to_csv(
    history_path_navec_new,
    index=False
)

print("\nМодель сохранена:", model_path_navec_new)
print("История сохранена:", history_path_navec_new)

# ============================================================
# 12. СРАВНЕНИЕ ЭКСПЕРИМЕНТОВ
# ============================================================

print("\n" + "=" * 55)
print("СРАВНЕНИЕ НА НОВОМ ДАТАСЕТЕ")
print("=" * 55)

print("TextCNN baseline:               0.7335")
print("TextCNN + Class Weights:        0.7314")
print("TextCNN + Augmentation:         0.7351")
print("TextCNN + Augmentation + Focal: 0.7301")
print(f"TextCNN + Navec:                {final_f1_navec_new:.4f}")
print("BiLSTM + Navec + Focal:         0.7577")

difference = final_f1_navec_new - 0.7335

print(
    f"\nРазница с TextCNN baseline: {difference:+.4f}"
)

## Эксперимент 6. TextCNN + Navec + Focal Loss

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 6. TEXTCNN + NAVEC + FOCAL LOSS
# Новый объединённый датасет
# ============================================================

import os
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required = [
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "DEVICE_NEW",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW",
    "navec_new"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing)
    )

SEED_EXP6 = 42

random.seed(SEED_EXP6)
np.random.seed(SEED_EXP6)
torch.manual_seed(SEED_EXP6)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_EXP6)

device_exp6 = torch.device(DEVICE_NEW)

print("Устройство:", device_exp6)

# ============================================================
# 2. МАТРИЦА NAVEC
# ============================================================

EMBED_DIM_EXP6 = 300
PAD_IDX_EXP6 = 0

if len(word2idx_new) != VOCAB_SIZE_NEW:
    raise ValueError(
        "Размер word2idx_new не совпадает с VOCAB_SIZE_NEW"
    )

rng_exp6 = np.random.default_rng(SEED_EXP6)

embedding_matrix_exp6 = rng_exp6.normal(
    loc=0.0,
    scale=0.05,
    size=(VOCAB_SIZE_NEW, EMBED_DIM_EXP6)
).astype(np.float32)

embedding_matrix_exp6[PAD_IDX_EXP6] = 0.0

found_exp6 = 0

print("\nПодготавливаем Navec...")

for word, idx in word2idx_new.items():

    if idx == PAD_IDX_EXP6:
        continue

    if word in navec_new:
        embedding_matrix_exp6[idx] = navec_new[word]
        found_exp6 += 1

coverage_exp6 = found_exp6 / VOCAB_SIZE_NEW * 100

print("\n" + "=" * 55)
print("NAVEC")
print("=" * 55)

print("Размер словаря:", VOCAB_SIZE_NEW)
print("Найдено в Navec:", found_exp6)
print(f"Покрытие: {coverage_exp6:.2f}%")
print("Размерность:", EMBED_DIM_EXP6)

embedding_tensor_exp6 = torch.from_numpy(
    embedding_matrix_exp6
)

# ============================================================
# 3. TEXTCNN + NAVEC
# ============================================================

class TextCNNNavecExp6(nn.Module):

    def __init__(
        self,
        embedding_weights,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_weights.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX_EXP6
        )

        with torch.no_grad():
            self.embedding.weight.copy_(embedding_weights)

        self.embedding.weight.requires_grad = True

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

        self.min_seq_len = max(kernel_sizes)

    def forward(self, x):

        x = self.embedding(x)
        x = x.transpose(1, 2)

        if x.size(2) < self.min_seq_len:
            x = F.pad(
                x,
                (0, self.min_seq_len - x.size(2))
            )

        pooled_outputs = []

        for conv in self.convs:

            h = F.relu(conv(x))

            pooled = F.max_pool1d(
                h,
                kernel_size=h.size(2)
            ).squeeze(2)

            pooled_outputs.append(pooled)

        x = torch.cat(pooled_outputs, dim=1)

        x = self.dropout(x)

        return self.fc(x)


model_exp6 = TextCNNNavecExp6(
    embedding_weights=embedding_tensor_exp6,
    num_filters=128,
    kernel_sizes=(3, 4, 5),
    dropout=0.5,
    num_classes=3
).to(device_exp6)

del embedding_matrix_exp6
del embedding_tensor_exp6

print("\nМодель создана!")

print(
    "Количество параметров:",
    sum(p.numel() for p in model_exp6.parameters())
)

# ============================================================
# 4. FOCAL LOSS
# ============================================================

class FocalLossExp6(nn.Module):

    def __init__(self, gamma=2.0, alpha=None):
        super().__init__()

        self.gamma = gamma

        if alpha is not None:
            self.register_buffer(
                "alpha",
                torch.tensor(alpha, dtype=torch.float32)
            )
        else:
            self.alpha = None

    def forward(self, logits, targets):

        log_probs = F.log_softmax(logits, dim=1)
        probs = log_probs.exp()

        target_log_probs = log_probs.gather(
            1,
            targets.unsqueeze(1)
        ).squeeze(1)

        target_probs = probs.gather(
            1,
            targets.unsqueeze(1)
        ).squeeze(1)

        focal_factor = (
            1.0 - target_probs
        ).pow(self.gamma)

        loss = -focal_factor * target_log_probs

        if self.alpha is not None:
            alpha_t = self.alpha[targets]
            loss = alpha_t * loss

        return loss.mean()


ALPHA_EXP6 = [1.5, 0.75, 1.0]
GAMMA_EXP6 = 2.0

criterion_exp6 = FocalLossExp6(
    gamma=GAMMA_EXP6,
    alpha=ALPHA_EXP6
).to(device_exp6)

optimizer_exp6 = torch.optim.AdamW(
    model_exp6.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

print("\nFocal Loss:")
print("Gamma:", GAMMA_EXP6)
print("Alpha:", ALPHA_EXP6)

# ============================================================
# 5. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_exp6(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(device_exp6)
        labels = labels.to(device_exp6)

        optimizer_exp6.zero_grad()

        logits = model(inputs)

        loss = criterion_exp6(logits, labels)

        loss.backward()
        optimizer_exp6.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 6. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_exp6(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(device_exp6)
        labels = labels.to(device_exp6)

        logits = model(inputs)

        loss = criterion_exp6(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )


# ============================================================
# 7. НАСТРОЙКИ ОБУЧЕНИЯ
# ============================================================

EPOCHS_EXP6 = 20
PATIENCE_EXP6 = 5

best_f1_exp6 = -1.0
best_epoch_exp6 = 0
best_state_exp6 = None

patience_counter_exp6 = 0

history_exp6 = []

start_time_exp6 = time.time()

print("\n" + "=" * 55)
print("НАЧИНАЕМ ОБУЧЕНИЕ TEXTCNN + NAVEC + FOCAL LOSS")
print("=" * 55)

# ============================================================
# 8. ОБУЧЕНИЕ
# ============================================================

for epoch in range(1, EPOCHS_EXP6 + 1):

    train_loss = train_epoch_exp6(
        model_exp6,
        train_loader_new
    )

    val_loss, y_true, y_pred = evaluate_exp6(
        model_exp6,
        val_loader_new
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_exp6.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1_exp6:

        best_f1_exp6 = macro_f1
        best_epoch_exp6 = epoch

        best_state_exp6 = {
            key: value.detach().cpu().clone()
            for key, value in model_exp6.state_dict().items()
        }

        patience_counter_exp6 = 0

    else:
        patience_counter_exp6 += 1

    if patience_counter_exp6 >= PATIENCE_EXP6:
        print("\nEarly stopping")
        break

# ============================================================
# 9. ВОССТАНАВЛИВАЕМ ЛУЧШУЮ МОДЕЛЬ
# ============================================================

if best_state_exp6 is None:
    raise RuntimeError("Не удалось сохранить лучшую модель.")

model_exp6.load_state_dict(best_state_exp6)
model_exp6.eval()

# ============================================================
# 10. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

_, y_true_exp6, y_pred_exp6 = evaluate_exp6(
    model_exp6,
    val_loader_new
)

final_f1_exp6 = f1_score(
    y_true_exp6,
    y_pred_exp6,
    average="macro",
    zero_division=0
)

final_accuracy_exp6 = accuracy_score(
    y_true_exp6,
    y_pred_exp6
)

elapsed_exp6 = (
    time.time() - start_time_exp6
) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 6 — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_exp6)
print(f"Macro-F1: {final_f1_exp6:.4f}")
print(f"Accuracy: {final_accuracy_exp6:.4f}")
print(f"Время обучения: {elapsed_exp6:.1f} мин")

print("\nClassification Report:")

print(
    classification_report(
        y_true_exp6,
        y_pred_exp6,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true_exp6,
        y_pred_exp6,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 11. СОХРАНЕНИЕ
# ============================================================

model_path_exp6 = (
    "/content/textcnn_navec_focal_merged_best.pt"
)

history_path_exp6 = (
    "/content/textcnn_navec_focal_history.csv"
)

torch.save(
    model_exp6.state_dict(),
    model_path_exp6
)

pd.DataFrame(history_exp6).to_csv(
    history_path_exp6,
    index=False
)

print("\nМодель сохранена:", model_path_exp6)
print("История сохранена:", history_path_exp6)

# ============================================================
# 12. СРАВНЕНИЕ
# ============================================================

baseline_navec_f1 = 0.7516

difference_exp6 = (
    final_f1_exp6 - baseline_navec_f1
)

print("\n" + "=" * 55)
print("СРАВНЕНИЕ ЭКСПЕРИМЕНТОВ 5 И 6")
print("=" * 55)

print("TextCNN + Navec:         0.7516")
print(
    f"TextCNN + Navec + Focal: {final_f1_exp6:.4f}"
)

print(
    f"Разница Macro-F1:        {difference_exp6:+.4f}"
)

if difference_exp6 > 0:
    print("\nFocal Loss улучшил Macro-F1.")
elif difference_exp6 < 0:
    print("\nFocal Loss не улучшил Macro-F1.")
else:
    print("\nMacro-F1 не изменился.")

## Эксперимент 7. BiLSTM + Navec (без Focal Loss)

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 7. BILSTM + NAVEC
# БЕЗ FOCAL LOSS
# Новый объединённый датасет
# ============================================================

import os
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required_exp7 = [
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "DEVICE_NEW",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW",
    "navec_new"
]

missing_exp7 = [
    name for name in required_exp7
    if name not in globals()
]

if missing_exp7:
    raise RuntimeError(
        "Не найдены переменные: "
        + ", ".join(missing_exp7)
    )

SEED_EXP7 = 42

random.seed(SEED_EXP7)
np.random.seed(SEED_EXP7)
torch.manual_seed(SEED_EXP7)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_EXP7)

device_exp7 = torch.device(DEVICE_NEW)

print("Устройство:", device_exp7)

# ============================================================
# 2. ПОДГОТОВКА NAVEC
# ============================================================

EMBED_DIM_EXP7 = 300
PAD_IDX_EXP7 = 0

if len(word2idx_new) != VOCAB_SIZE_NEW:
    raise ValueError(
        "Размер словаря не совпадает с VOCAB_SIZE_NEW"
    )

rng_exp7 = np.random.default_rng(SEED_EXP7)

embedding_matrix_exp7 = rng_exp7.normal(
    loc=0.0,
    scale=0.05,
    size=(VOCAB_SIZE_NEW, EMBED_DIM_EXP7)
).astype(np.float32)

embedding_matrix_exp7[PAD_IDX_EXP7] = 0.0

found_exp7 = 0

print("\nПодготавливаем Navec...")

for word, idx in word2idx_new.items():

    if idx == PAD_IDX_EXP7:
        continue

    if word in navec_new:
        embedding_matrix_exp7[idx] = navec_new[word]
        found_exp7 += 1

coverage_exp7 = found_exp7 / VOCAB_SIZE_NEW * 100

print("\n" + "=" * 55)
print("NAVEC")
print("=" * 55)

print("Размер словаря:", VOCAB_SIZE_NEW)
print("Найдено в Navec:", found_exp7)
print(f"Покрытие: {coverage_exp7:.2f}%")
print("Размерность:", EMBED_DIM_EXP7)

embedding_tensor_exp7 = torch.from_numpy(
    embedding_matrix_exp7
)

# ============================================================
# 3. АРХИТЕКТУРА BILSTM + NAVEC
# ============================================================

class BiLSTMNavecExp7(nn.Module):

    def __init__(
        self,
        embedding_weights,
        hidden_size=128,
        dropout=0.4,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_weights.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX_EXP7
        )

        with torch.no_grad():
            self.embedding.weight.copy_(embedding_weights)

        # Navec дообучается вместе с моделью
        self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_size * 2,
            num_classes
        )

    def forward(self, input_ids):

        # [batch, seq_len]
        mask = (input_ids != PAD_IDX_EXP7)

        # [batch, seq_len, 300]
        embedded = self.embedding(input_ids)

        # [batch, seq_len, hidden_size * 2]
        lstm_output, _ = self.lstm(embedded)

        # Masked mean pooling
        mask_float = mask.unsqueeze(-1).to(
            dtype=lstm_output.dtype
        )

        masked_output = lstm_output * mask_float

        summed = masked_output.sum(dim=1)

        lengths = mask_float.sum(dim=1).clamp(min=1.0)

        pooled = summed / lengths

        pooled = self.dropout(pooled)

        return self.fc(pooled)


model_exp7 = BiLSTMNavecExp7(
    embedding_weights=embedding_tensor_exp7,
    hidden_size=128,
    dropout=0.4,
    num_classes=3
).to(device_exp7)

del embedding_matrix_exp7
del embedding_tensor_exp7

print("\nМодель BiLSTM + Navec создана!")

print(
    "Количество параметров:",
    sum(p.numel() for p in model_exp7.parameters())
)

# ============================================================
# 4. LOSS И OPTIMIZER
# ============================================================

# ВАЖНО:
# В этом эксперименте используем обычную CrossEntropyLoss.
# Без Focal Loss и без Class Weights.

criterion_exp7 = nn.CrossEntropyLoss()

optimizer_exp7 = torch.optim.AdamW(
    model_exp7.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

print("\nФункция потерь: CrossEntropyLoss")
print("Оптимизатор: AdamW")
print("Learning rate: 0.0005")

# ============================================================
# 5. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_exp7(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(device_exp7)
        labels = labels.to(device_exp7)

        optimizer_exp7.zero_grad()

        logits = model(inputs)

        loss = criterion_exp7(logits, labels)

        loss.backward()

        optimizer_exp7.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 6. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_exp7(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(device_exp7)
        labels = labels.to(device_exp7)

        logits = model(inputs)

        loss = criterion_exp7(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )


# ============================================================
# 7. НАСТРОЙКИ ОБУЧЕНИЯ
# ============================================================

EPOCHS_EXP7 = 20
PATIENCE_EXP7 = 5

best_f1_exp7 = -1.0
best_epoch_exp7 = 0
best_state_exp7 = None

patience_counter_exp7 = 0

history_exp7 = []

start_time_exp7 = time.time()

print("\n" + "=" * 55)
print("НАЧИНАЕМ ОБУЧЕНИЕ BILSTM + NAVEC")
print("LOSS: CROSS ENTROPY")
print("=" * 55)

# ============================================================
# 8. ОБУЧЕНИЕ
# ============================================================

for epoch in range(1, EPOCHS_EXP7 + 1):

    train_loss = train_epoch_exp7(
        model_exp7,
        train_loader_new
    )

    val_loss, y_true, y_pred = evaluate_exp7(
        model_exp7,
        val_loader_new
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_exp7.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1_exp7:

        best_f1_exp7 = macro_f1
        best_epoch_exp7 = epoch

        best_state_exp7 = {
            key: value.detach().cpu().clone()
            for key, value in model_exp7.state_dict().items()
        }

        patience_counter_exp7 = 0

    else:
        patience_counter_exp7 += 1

    if patience_counter_exp7 >= PATIENCE_EXP7:
        print("\nEarly stopping")
        break

# ============================================================
# 9. ВОССТАНОВЛЕНИЕ ЛУЧШЕЙ МОДЕЛИ
# ============================================================

if best_state_exp7 is None:
    raise RuntimeError(
        "Не удалось сохранить лучшую модель."
    )

model_exp7.load_state_dict(best_state_exp7)

model_exp7.eval()

# ============================================================
# 10. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

_, y_true_exp7, y_pred_exp7 = evaluate_exp7(
    model_exp7,
    val_loader_new
)

final_f1_exp7 = f1_score(
    y_true_exp7,
    y_pred_exp7,
    average="macro",
    zero_division=0
)

final_accuracy_exp7 = accuracy_score(
    y_true_exp7,
    y_pred_exp7
)

elapsed_exp7 = (
    time.time() - start_time_exp7
) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 7 — ИТОГИ")
print("=" * 55)

print(f"Лучшая эпоха: {best_epoch_exp7}")
print(f"Macro-F1: {final_f1_exp7:.4f}")
print(f"Accuracy: {final_accuracy_exp7:.4f}")
print(f"Время обучения: {elapsed_exp7:.1f} мин")

print("\nClassification Report:")

print(
    classification_report(
        y_true_exp7,
        y_pred_exp7,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true_exp7,
        y_pred_exp7,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 11. СОХРАНЕНИЕ
# ============================================================

model_path_exp7 = (
    "/content/bilstm_navec_crossentropy_merged_best.pt"
)

history_path_exp7 = (
    "/content/bilstm_navec_crossentropy_history.csv"
)

torch.save(
    model_exp7.state_dict(),
    model_path_exp7
)

pd.DataFrame(history_exp7).to_csv(
    history_path_exp7,
    index=False
)

print("\nМодель сохранена:", model_path_exp7)
print("История сохранена:", history_path_exp7)

# ============================================================
# 12. СРАВНЕНИЕ С FOCAL LOSS
# ============================================================

focal_f1_exp7 = 0.7577

difference_exp7 = final_f1_exp7 - focal_f1_exp7

print("\n" + "=" * 55)
print("СРАВНЕНИЕ BILSTM")
print("=" * 55)

print(
    f"BiLSTM + Navec (CrossEntropy): {final_f1_exp7:.4f}"
)
print(
    f"BiLSTM + Navec + Focal:        {focal_f1_exp7:.4f}"
)
print(
    f"Разница Macro-F1:              {difference_exp7:+.4f}"
)

if difference_exp7 > 0:
    print("\nCrossEntropy показала лучший Macro-F1.")
elif difference_exp7 < 0:
    print("\nFocal Loss показала лучший Macro-F1.")
else:
    print("\nРезультаты по Macro-F1 совпали.")

## Эксперимент 9. TextCNN + Lemmatization + Augmentation

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 9. TEXTCNN + LEMMATIZATION + AUGMENTATION
# НОВЫЙ ОБЪЕДИНЁННЫЙ ДАТАСЕТ
# ============================================================

import sys
import subprocess
import importlib.util

if importlib.util.find_spec("pymorphy3") is None:
    subprocess.check_call([
        sys.executable, "-m", "pip", "-q",
        "install", "pymorphy3"
    ])

import os
import re
import time
import random
from collections import Counter
from functools import lru_cache

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import pymorphy3

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ИСХОДНЫХ ДАННЫХ
# ============================================================

required_exp9 = [
    "train_new",
    "X_val_new",
    "y_val_new",
    "CLASS_NAMES_NEW",
    "DEVICE_NEW"
]

missing_exp9 = [
    name for name in required_exp9
    if name not in globals()
]

if missing_exp9:
    raise RuntimeError(
        "Не найдены переменные: "
        + ", ".join(missing_exp9)
    )

if not {"text", "sentiment"}.issubset(train_new.columns):
    raise ValueError(
        "В train_new должны быть колонки text и sentiment"
    )

SEED_EXP9 = 42

random.seed(SEED_EXP9)
np.random.seed(SEED_EXP9)
torch.manual_seed(SEED_EXP9)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_EXP9)

DEVICE_EXP9 = torch.device(DEVICE_NEW)

print("Устройство:", DEVICE_EXP9)

# ============================================================
# 2. НАСТРОЙКИ
# ============================================================

MAX_LEN_EXP9 = 256
EMBED_DIM_EXP9 = 128
NUM_FILTERS_EXP9 = 128
KERNEL_SIZES_EXP9 = (3, 4, 5)
DROPOUT_EXP9 = 0.5

BATCH_SIZE_EXP9 = 64
EPOCHS_EXP9 = 20
PATIENCE_EXP9 = 5

LR_EXP9 = 5e-4
WEIGHT_DECAY_EXP9 = 1e-4

AUGMENT_FRACTION_EXP9 = 0.20
AUGMENT_CLASSES_EXP9 = [0, 2]

PAD_IDX_EXP9 = 0
UNK_IDX_EXP9 = 1

CACHE_PATH_EXP9 = (
    "/content/experiment9_lemmatized_cache.pkl"
)

MODEL_PATH_EXP9 = (
    "/content/textcnn_lemma_augmentation_best.pt"
)

HISTORY_PATH_EXP9 = (
    "/content/textcnn_lemma_augmentation_history.csv"
)

# ============================================================
# 3. ПОДГОТОВКА ТЕКСТОВ
# ============================================================

train_texts_exp9 = (
    train_new["text"]
    .fillna("")
    .astype(str)
    .tolist()
)

train_labels_exp9 = (
    train_new["sentiment"]
    .astype(int)
    .to_numpy()
)

val_texts_exp9 = [
    "" if pd.isna(text) else str(text)
    for text in X_val_new
]

val_labels_exp9 = np.asarray(
    y_val_new,
    dtype=np.int64
)

print("\nИсходные данные:")
print("Train:", len(train_texts_exp9))
print("Validation:", len(val_texts_exp9))

# ============================================================
# 4. ЛЕММАТИЗАЦИЯ
# ============================================================

morph_exp9 = pymorphy3.MorphAnalyzer()

TOKEN_PATTERN_EXP9 = re.compile(
    r"[а-яёa-z0-9]+",
    flags=re.IGNORECASE
)

RUSSIAN_WORD_EXP9 = re.compile(
    r"^[а-яё]+$"
)

@lru_cache(maxsize=300000)
def lemmatize_word_exp9(word):

    if RUSSIAN_WORD_EXP9.fullmatch(word):
        return morph_exp9.parse(word)[0].normal_form

    return word


def lemmatize_text_exp9(text):

    tokens = TOKEN_PATTERN_EXP9.findall(
        str(text).lower()
    )

    return [
        lemmatize_word_exp9(word)
        for word in tokens
    ]


print("\n" + "=" * 55)
print("ЛЕММАТИЗАЦИЯ")
print("=" * 55)

if os.path.exists(CACHE_PATH_EXP9):

    print("Найден кеш. Загружаем...")

    cache_exp9 = pd.read_pickle(
        CACHE_PATH_EXP9
    )

    if (
        cache_exp9.get("train_count") == len(train_texts_exp9)
        and cache_exp9.get("val_count") == len(val_texts_exp9)
    ):

        train_tokens_exp9 = cache_exp9["train_tokens"]
        val_tokens_exp9 = cache_exp9["val_tokens"]

        print("Лемматизированные тексты загружены.")

    else:
        raise RuntimeError(
            "Кеш не соответствует размеру датасета. "
            "Удалите CACHE_PATH_EXP9 и запустите ячейку заново."
        )

else:

    print("Лемматизируем обучающую выборку...")

    train_tokens_exp9 = []

    for i, text in enumerate(train_texts_exp9, 1):

        train_tokens_exp9.append(
            lemmatize_text_exp9(text)
        )

        if i % 20000 == 0:
            print(
                f"Train: {i}/{len(train_texts_exp9)}",
                flush=True
            )

    print("\nЛемматизируем валидационную выборку...")

    val_tokens_exp9 = []

    for i, text in enumerate(val_texts_exp9, 1):

        val_tokens_exp9.append(
            lemmatize_text_exp9(text)
        )

        if i % 5000 == 0:
            print(
                f"Validation: {i}/{len(val_texts_exp9)}",
                flush=True
            )

    pd.to_pickle(
        {
            "train_count": len(train_texts_exp9),
            "val_count": len(val_texts_exp9),
            "train_tokens": train_tokens_exp9,
            "val_tokens": val_tokens_exp9
        },
        CACHE_PATH_EXP9
    )

    print("\nКеш сохранён:", CACHE_PATH_EXP9)

print("\nПример лемматизации:")

if train_texts_exp9:
    print("Оригинал:", train_texts_exp9[0][:250])
    print("Леммы:", " ".join(train_tokens_exp9[0])[:250])

# ============================================================
# 5. АУГМЕНТАЦИЯ
# ============================================================

print("\n" + "=" * 55)
print("АУГМЕНТАЦИЯ NEUTRAL И NEGATIVE")
print("=" * 55)

rng_aug_exp9 = random.Random(SEED_EXP9)

augmented_tokens_exp9 = list(train_tokens_exp9)
augmented_labels_exp9 = train_labels_exp9.tolist()


def augment_delete_word_exp9(tokens):

    result = list(tokens)

    if len(result) <= 1:
        return result

    delete_position = rng_aug_exp9.randrange(
        len(result)
    )

    del result[delete_position]

    return result


for class_id in AUGMENT_CLASSES_EXP9:

    class_indices = [
        i
        for i, label in enumerate(train_labels_exp9)
        if label == class_id and len(train_tokens_exp9[i]) > 1
    ]

    original_count = int(
        np.sum(train_labels_exp9 == class_id)
    )

    target_extra = round(
        original_count * AUGMENT_FRACTION_EXP9
    )

    if not class_indices:
        print(
            f"Класс {class_id}: нет подходящих текстов"
        )
        continue

    selected_indices = rng_aug_exp9.choices(
        class_indices,
        k=target_extra
    )

    for idx in selected_indices:

        new_tokens = augment_delete_word_exp9(
            train_tokens_exp9[idx]
        )

        augmented_tokens_exp9.append(
            new_tokens
        )

        augmented_labels_exp9.append(
            class_id
        )

    print(
        f"Класс {class_id}: "
        f"было {original_count}, "
        f"добавлено {target_extra}"
    )

print("\nВсего после аугментации:")
print(len(augmented_tokens_exp9))

print("\nРаспределение классов:")

print(
    pd.Series(augmented_labels_exp9)
    .value_counts()
    .sort_index()
)

# ============================================================
# 6. СОЗДАНИЕ СЛОВАРЯ
# ============================================================

print("\n" + "=" * 55)
print("СОЗДАНИЕ СЛОВАРЯ")
print("=" * 55)

word_counts_exp9 = Counter()

# Словарь строится только по исходному train.
# Validation не участвует в построении словаря.

for tokens in train_tokens_exp9:
    word_counts_exp9.update(tokens)

word2idx_exp9 = {
    "<PAD>": PAD_IDX_EXP9,
    "<UNK>": UNK_IDX_EXP9
}

for word, count in word_counts_exp9.items():

    if word not in word2idx_exp9:
        word2idx_exp9[word] = len(word2idx_exp9)

VOCAB_SIZE_EXP9 = len(word2idx_exp9)

print("Размер словаря:", VOCAB_SIZE_EXP9)

# ============================================================
# 7. DATASET И DATALOADER
# ============================================================

class LemmaDatasetExp9(Dataset):

    def __init__(self, token_lists, labels):

        self.token_lists = token_lists
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        tokens = self.token_lists[idx][:MAX_LEN_EXP9]

        ids = [
            word2idx_exp9.get(
                word,
                UNK_IDX_EXP9
            )
            for word in tokens
        ]

        if len(ids) < MAX_LEN_EXP9:

            ids += [
                PAD_IDX_EXP9
            ] * (MAX_LEN_EXP9 - len(ids))

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(
                int(self.labels[idx]),
                dtype=torch.long
            )
        )


train_dataset_exp9 = LemmaDatasetExp9(
    augmented_tokens_exp9,
    augmented_labels_exp9
)

val_dataset_exp9 = LemmaDatasetExp9(
    val_tokens_exp9,
    val_labels_exp9
)

generator_exp9 = torch.Generator()
generator_exp9.manual_seed(SEED_EXP9)

train_loader_exp9 = DataLoader(
    train_dataset_exp9,
    batch_size=BATCH_SIZE_EXP9,
    shuffle=True,
    generator=generator_exp9,
    num_workers=0,
    pin_memory=(DEVICE_EXP9.type == "cuda")
)

val_loader_exp9 = DataLoader(
    val_dataset_exp9,
    batch_size=BATCH_SIZE_EXP9,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE_EXP9.type == "cuda")
)

print("\nDataLoader созданы.")
print("Train batches:", len(train_loader_exp9))
print("Validation batches:", len(val_loader_exp9))

# ============================================================
# 8. АРХИТЕКТУРА TEXTCNN
# ============================================================

class TextCNNLemmaExp9(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX_EXP9
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                embedding_dim,
                num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

        self.min_seq_len = max(kernel_sizes)

    def forward(self, x):

        x = self.embedding(x)

        x = x.transpose(1, 2)

        if x.size(2) < self.min_seq_len:
            x = F.pad(
                x,
                (0, self.min_seq_len - x.size(2))
            )

        pooled_outputs = []

        for conv in self.convs:

            h = F.relu(conv(x))

            pooled = F.max_pool1d(
                h,
                kernel_size=h.size(2)
            ).squeeze(2)

            pooled_outputs.append(pooled)

        x = torch.cat(pooled_outputs, dim=1)

        x = self.dropout(x)

        return self.fc(x)


model_exp9 = TextCNNLemmaExp9(
    vocab_size=VOCAB_SIZE_EXP9,
    embedding_dim=EMBED_DIM_EXP9,
    num_filters=NUM_FILTERS_EXP9,
    kernel_sizes=KERNEL_SIZES_EXP9,
    dropout=DROPOUT_EXP9,
    num_classes=3
).to(DEVICE_EXP9)

print("\nМодель создана!")

print(
    "Количество параметров:",
    sum(p.numel() for p in model_exp9.parameters())
)

# ============================================================
# 9. LOSS И OPTIMIZER
# ============================================================

criterion_exp9 = nn.CrossEntropyLoss()

optimizer_exp9 = torch.optim.AdamW(
    model_exp9.parameters(),
    lr=LR_EXP9,
    weight_decay=WEIGHT_DECAY_EXP9
)

# ============================================================
# 10. ФУНКЦИИ ОБУЧЕНИЯ И ВАЛИДАЦИИ
# ============================================================

def train_epoch_exp9(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP9)
        labels = labels.to(DEVICE_EXP9)

        optimizer_exp9.zero_grad()

        logits = model(inputs)

        loss = criterion_exp9(logits, labels)

        loss.backward()
        optimizer_exp9.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


@torch.no_grad()
def evaluate_exp9(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP9)
        labels = labels.to(DEVICE_EXP9)

        logits = model(inputs)

        loss = criterion_exp9(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )

# ============================================================
# 11. ОБУЧЕНИЕ
# ============================================================

best_f1_exp9 = -1.0
best_epoch_exp9 = 0
best_state_exp9 = None

patience_counter_exp9 = 0
history_exp9 = []

start_time_exp9 = time.time()

print("\n" + "=" * 55)
print("ОБУЧЕНИЕ TEXTCNN + LEMMATIZATION + AUGMENTATION")
print("=" * 55)

for epoch in range(1, EPOCHS_EXP9 + 1):

    train_loss = train_epoch_exp9(
        model_exp9,
        train_loader_exp9
    )

    val_loss, y_true, y_pred = evaluate_exp9(
        model_exp9,
        val_loader_exp9
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_exp9.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1_exp9:

        best_f1_exp9 = macro_f1
        best_epoch_exp9 = epoch

        best_state_exp9 = {
            key: value.detach().cpu().clone()
            for key, value in model_exp9.state_dict().items()
        }

        patience_counter_exp9 = 0

    else:
        patience_counter_exp9 += 1

    if patience_counter_exp9 >= PATIENCE_EXP9:
        print("\nEarly stopping")
        break

# ============================================================
# 12. ВОССТАНОВЛЕНИЕ ЛУЧШЕЙ МОДЕЛИ
# ============================================================

if best_state_exp9 is None:
    raise RuntimeError(
        "Не удалось сохранить лучшую модель."
    )

model_exp9.load_state_dict(best_state_exp9)
model_exp9.eval()

# ============================================================
# 13. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

_, y_true_exp9, y_pred_exp9 = evaluate_exp9(
    model_exp9,
    val_loader_exp9
)

final_f1_exp9 = f1_score(
    y_true_exp9,
    y_pred_exp9,
    average="macro",
    zero_division=0
)

final_accuracy_exp9 = accuracy_score(
    y_true_exp9,
    y_pred_exp9
)

elapsed_exp9 = (
    time.time() - start_time_exp9
) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 9 — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_exp9)
print(f"Macro-F1: {final_f1_exp9:.4f}")
print(f"Accuracy: {final_accuracy_exp9:.4f}")
print(f"Время обучения: {elapsed_exp9:.1f} мин")

print("\nClassification Report:")

print(
    classification_report(
        y_true_exp9,
        y_pred_exp9,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true_exp9,
        y_pred_exp9,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 14. СОХРАНЕНИЕ
# ============================================================

torch.save(
    {
        "model_state_dict": model_exp9.state_dict(),
        "word2idx": word2idx_exp9,
        "max_len": MAX_LEN_EXP9,
        "class_names": CLASS_NAMES_NEW,
        "best_epoch": best_epoch_exp9,
        "macro_f1": final_f1_exp9,
        "config": {
            "embedding_dim": EMBED_DIM_EXP9,
            "num_filters": NUM_FILTERS_EXP9,
            "kernel_sizes": KERNEL_SIZES_EXP9,
            "dropout": DROPOUT_EXP9
        }
    },
    MODEL_PATH_EXP9
)

pd.DataFrame(history_exp9).to_csv(
    HISTORY_PATH_EXP9,
    index=False
)

print("\nМодель сохранена:", MODEL_PATH_EXP9)
print("История сохранена:", HISTORY_PATH_EXP9)

# ============================================================
# 15. СРАВНЕНИЕ
# ============================================================

baseline_exp9 = 0.7335
augmentation_exp9 = 0.7351
best_model_exp9 = 0.7650

print("\n" + "=" * 55)
print("СРАВНЕНИЕ ЭКСПЕРИМЕНТОВ")
print("=" * 55)

print(f"TextCNN baseline:             {baseline_exp9:.4f}")
print(f"TextCNN + Augmentation:       {augmentation_exp9:.4f}")
print(f"TextCNN + Lemma + Augment:    {final_f1_exp9:.4f}")
print(f"BiLSTM + Navec:               {best_model_exp9:.4f}")

print(
    "\nРазница с TextCNN baseline:",
    f"{final_f1_exp9 - baseline_exp9:+.4f}"
)

print(
    "Разница с TextCNN + Augmentation:",
    f"{final_f1_exp9 - augmentation_exp9:+.4f}"
)

print(
    "Разница с лучшей BiLSTM:",
    f"{final_f1_exp9 - best_model_exp9:+.4f}"
)

Эксперимент 10. TextCNN + Mild Class Weights

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 10. TEXTCNN + MILD CLASS WEIGHTS
# НОВЫЙ ОБЪЕДИНЁННЫЙ ДАТАСЕТ
# ============================================================

import os
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required_exp10 = [
    "train_new",
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "DEVICE_NEW",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW"
]

missing_exp10 = [
    name for name in required_exp10
    if name not in globals()
]

if missing_exp10:
    raise RuntimeError(
        "Не найдены переменные: "
        + ", ".join(missing_exp10)
    )

SEED_EXP10 = 42

random.seed(SEED_EXP10)
np.random.seed(SEED_EXP10)
torch.manual_seed(SEED_EXP10)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_EXP10)

DEVICE_EXP10 = torch.device(DEVICE_NEW)

print("Устройство:", DEVICE_EXP10)

# ============================================================
# 2. НАСТРОЙКИ
# ============================================================

EMBED_DIM_EXP10 = 128
NUM_FILTERS_EXP10 = 128
KERNEL_SIZES_EXP10 = (3, 4, 5)
DROPOUT_EXP10 = 0.5

EPOCHS_EXP10 = 20
PATIENCE_EXP10 = 5

LR_EXP10 = 5e-4
WEIGHT_DECAY_EXP10 = 1e-4

NUM_CLASSES_EXP10 = 3
PAD_IDX_EXP10 = 0

# Сила коррекции дисбаланса:
# 0.0 = обычная CrossEntropy
# 1.0 = полные сбалансированные веса
MILD_FACTOR_EXP10 = 0.5

MODEL_PATH_EXP10 = (
    "/content/textcnn_mild_weights_best.pt"
)

HISTORY_PATH_EXP10 = (
    "/content/textcnn_mild_weights_history.csv"
)

# ============================================================
# 3. РАСЧЁТ ВЕСОВ КЛАССОВ
# ============================================================

print("\n" + "=" * 55)
print("РАСЧЁТ MILD CLASS WEIGHTS")
print("=" * 55)

train_labels_exp10 = (
    train_new["sentiment"]
    .astype(int)
    .to_numpy()
)

class_counts_exp10 = np.bincount(
    train_labels_exp10,
    minlength=NUM_CLASSES_EXP10
)

if np.any(class_counts_exp10 == 0):
    raise ValueError(
        "В обучающей выборке отсутствует один из классов."
    )

total_samples_exp10 = class_counts_exp10.sum()

balanced_weights_exp10 = (
    total_samples_exp10
    / (NUM_CLASSES_EXP10 * class_counts_exp10)
)

mild_weights_exp10 = (
    1.0
    + MILD_FACTOR_EXP10
    * (balanced_weights_exp10 - 1.0)
)

print("Количество примеров по классам:")

for class_id, count in enumerate(class_counts_exp10):
    print(
        f"{CLASS_NAMES_NEW[class_id]}: {count}"
    )

print("\nПолные сбалансированные веса:")
print(np.round(balanced_weights_exp10, 4))

print("\nМягкие веса:")
print(np.round(mild_weights_exp10, 4))

print("\nКоэффициент коррекции:", MILD_FACTOR_EXP10)

class_weights_tensor_exp10 = torch.tensor(
    mild_weights_exp10,
    dtype=torch.float32,
    device=DEVICE_EXP10
)

# ============================================================
# 4. АРХИТЕКТУРА TEXTCNN
# ============================================================

class TextCNNMildExp10(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX_EXP10
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

        self.min_seq_len = max(kernel_sizes)

    def forward(self, x):

        x = self.embedding(x)

        x = x.transpose(1, 2)

        if x.size(2) < self.min_seq_len:
            x = F.pad(
                x,
                (0, self.min_seq_len - x.size(2))
            )

        pooled_outputs = []

        for conv in self.convs:

            h = F.relu(conv(x))

            pooled = F.max_pool1d(
                h,
                kernel_size=h.size(2)
            ).squeeze(2)

            pooled_outputs.append(pooled)

        x = torch.cat(pooled_outputs, dim=1)

        x = self.dropout(x)

        return self.fc(x)


model_exp10 = TextCNNMildExp10(
    vocab_size=VOCAB_SIZE_NEW,
    embedding_dim=EMBED_DIM_EXP10,
    num_filters=NUM_FILTERS_EXP10,
    kernel_sizes=KERNEL_SIZES_EXP10,
    dropout=DROPOUT_EXP10,
    num_classes=NUM_CLASSES_EXP10
).to(DEVICE_EXP10)

print("\nМодель создана!")

print(
    "Количество параметров:",
    sum(p.numel() for p in model_exp10.parameters())
)

# ============================================================
# 5. LOSS И OPTIMIZER
# ============================================================

criterion_exp10 = nn.CrossEntropyLoss(
    weight=class_weights_tensor_exp10
)

optimizer_exp10 = torch.optim.AdamW(
    model_exp10.parameters(),
    lr=LR_EXP10,
    weight_decay=WEIGHT_DECAY_EXP10
)

print("\nLoss: Weighted CrossEntropyLoss")
print("Optimizer: AdamW")
print("Learning rate:", LR_EXP10)

# ============================================================
# 6. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_exp10(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP10)
        labels = labels.to(DEVICE_EXP10)

        optimizer_exp10.zero_grad()

        logits = model(inputs)

        loss = criterion_exp10(logits, labels)

        loss.backward()
        optimizer_exp10.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 7. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_exp10(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP10)
        labels = labels.to(DEVICE_EXP10)

        logits = model(inputs)

        loss = criterion_exp10(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )


# ============================================================
# 8. ОБУЧЕНИЕ
# ============================================================

best_f1_exp10 = -1.0
best_epoch_exp10 = 0
best_state_exp10 = None

patience_counter_exp10 = 0

history_exp10 = []

start_time_exp10 = time.time()

print("\n" + "=" * 55)
print("НАЧИНАЕМ ОБУЧЕНИЕ TEXTCNN + MILD CLASS WEIGHTS")
print("=" * 55)

for epoch in range(1, EPOCHS_EXP10 + 1):

    train_loss = train_epoch_exp10(
        model_exp10,
        train_loader_new
    )

    val_loss, y_true, y_pred = evaluate_exp10(
        model_exp10,
        val_loader_new
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_exp10.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1_exp10:

        best_f1_exp10 = macro_f1
        best_epoch_exp10 = epoch

        best_state_exp10 = {
            key: value.detach().cpu().clone()
            for key, value in model_exp10.state_dict().items()
        }

        patience_counter_exp10 = 0

    else:
        patience_counter_exp10 += 1

    if patience_counter_exp10 >= PATIENCE_EXP10:
        print("\nEarly stopping")
        break

# ============================================================
# 9. ВОССТАНОВЛЕНИЕ ЛУЧШЕЙ МОДЕЛИ
# ============================================================

if best_state_exp10 is None:
    raise RuntimeError(
        "Не удалось сохранить лучшую модель."
    )

model_exp10.load_state_dict(best_state_exp10)
model_exp10.eval()

# ============================================================
# 10. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

_, y_true_exp10, y_pred_exp10 = evaluate_exp10(
    model_exp10,
    val_loader_new
)

final_f1_exp10 = f1_score(
    y_true_exp10,
    y_pred_exp10,
    average="macro",
    zero_division=0
)

final_accuracy_exp10 = accuracy_score(
    y_true_exp10,
    y_pred_exp10
)

elapsed_exp10 = (
    time.time() - start_time_exp10
) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 10 — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_exp10)
print(f"Macro-F1: {final_f1_exp10:.4f}")
print(f"Accuracy: {final_accuracy_exp10:.4f}")
print(f"Время обучения: {elapsed_exp10:.1f} мин")

print("\nClassification Report:")

print(
    classification_report(
        y_true_exp10,
        y_pred_exp10,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true_exp10,
        y_pred_exp10,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 11. СОХРАНЕНИЕ
# ============================================================

torch.save(
    {
        "model_state_dict": model_exp10.state_dict(),
        "class_weights": mild_weights_exp10.tolist(),
        "best_epoch": best_epoch_exp10,
        "macro_f1": final_f1_exp10,
        "config": {
            "embedding_dim": EMBED_DIM_EXP10,
            "num_filters": NUM_FILTERS_EXP10,
            "kernel_sizes": KERNEL_SIZES_EXP10,
            "dropout": DROPOUT_EXP10,
            "mild_factor": MILD_FACTOR_EXP10
        }
    },
    MODEL_PATH_EXP10
)

pd.DataFrame(history_exp10).to_csv(
    HISTORY_PATH_EXP10,
    index=False
)

print("\nМодель сохранена:", MODEL_PATH_EXP10)
print("История сохранена:", HISTORY_PATH_EXP10)

# ============================================================
# 12. СРАВНЕНИЕ
# ============================================================

baseline_f1_exp10 = 0.7335
full_weights_f1_exp10 = 0.7314
best_bilstm_f1_exp10 = 0.7650

print("\n" + "=" * 55)
print("СРАВНЕНИЕ ЭКСПЕРИМЕНТОВ")
print("=" * 55)

print(
    f"TextCNN baseline:            {baseline_f1_exp10:.4f}"
)
print(
    f"TextCNN + Full Weights:      {full_weights_f1_exp10:.4f}"
)
print(
    f"TextCNN + Mild Weights:      {final_f1_exp10:.4f}"
)
print(
    f"BiLSTM + Navec:              {best_bilstm_f1_exp10:.4f}"
)

print(
    "\nРазница с baseline:",
    f"{final_f1_exp10 - baseline_f1_exp10:+.4f}"
)

print(
    "Разница с полными весами:",
    f"{final_f1_exp10 - full_weights_f1_exp10:+.4f}"
)

print(
    "Разница с лучшей BiLSTM:",
    f"{final_f1_exp10 - best_bilstm_f1_exp10:+.4f}"
)

## Эксперимент 11. TextCNN + Neutral-only Augmentation


In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 11. TEXTCNN + NEUTRAL-ONLY AUGMENTATION
# НОВЫЙ ОБЪЕДИНЁННЫЙ ДАТАСЕТ
# ============================================================

import os
import re
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ПОДГОТОВЛЕННЫХ ДАННЫХ
# ============================================================

required_exp11 = [
    "train_new",
    "X_val_new",
    "y_val_new",
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "MAX_LEN_NEW",
    "DEVICE_NEW",
    "CLASS_NAMES_NEW"
]

missing_exp11 = [
    name for name in required_exp11
    if name not in globals()
]

if missing_exp11:
    raise RuntimeError(
        "Не найдены переменные: "
        + ", ".join(missing_exp11)
    )

if not {"text", "sentiment"}.issubset(train_new.columns):
    raise ValueError(
        "В train_new должны быть колонки text и sentiment"
    )

# ============================================================
# 2. SEED И НАСТРОЙКИ
# ============================================================

SEED_EXP11 = 42

random.seed(SEED_EXP11)
np.random.seed(SEED_EXP11)
torch.manual_seed(SEED_EXP11)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_EXP11)

DEVICE_EXP11 = torch.device(DEVICE_NEW)

MAX_LEN_EXP11 = MAX_LEN_NEW
VOCAB_SIZE_EXP11 = VOCAB_SIZE_NEW

PAD_IDX_EXP11 = word2idx_new.get("<PAD>", 0)
UNK_IDX_EXP11 = word2idx_new.get("<UNK>", 1)

EMBED_DIM_EXP11 = 128
NUM_FILTERS_EXP11 = 128
KERNEL_SIZES_EXP11 = (3, 4, 5)
DROPOUT_EXP11 = 0.5

BATCH_SIZE_EXP11 = 64
EPOCHS_EXP11 = 20
PATIENCE_EXP11 = 5

LR_EXP11 = 5e-4
WEIGHT_DECAY_EXP11 = 1e-4

NEUTRAL_CLASS_EXP11 = 0
AUGMENT_FRACTION_EXP11 = 0.20

MODEL_PATH_EXP11 = (
    "/content/textcnn_neutral_only_aug_best.pt"
)

HISTORY_PATH_EXP11 = (
    "/content/textcnn_neutral_only_aug_history.csv"
)

print("Устройство:", DEVICE_EXP11)

# ============================================================
# 3. ИСХОДНЫЕ ДАННЫЕ
# ============================================================

train_texts_exp11 = (
    train_new["text"]
    .fillna("")
    .astype(str)
    .tolist()
)

train_labels_exp11 = (
    train_new["sentiment"]
    .astype(int)
    .tolist()
)

val_texts_exp11 = [
    "" if pd.isna(text) else str(text)
    for text in X_val_new
]

val_labels_exp11 = np.asarray(
    y_val_new,
    dtype=np.int64
)

print("\n" + "=" * 55)
print("ИСХОДНЫЕ ДАННЫЕ")
print("=" * 55)

print("Train:", len(train_texts_exp11))
print("Validation:", len(val_texts_exp11))

print("\nРаспределение классов до аугментации:")

print(
    pd.Series(train_labels_exp11)
    .value_counts()
    .sort_index()
)

# ============================================================
# 4. ТОКЕНИЗАЦИЯ
# ============================================================

TOKEN_PATTERN_EXP11 = re.compile(
    r"[а-яёa-z0-9]+",
    flags=re.IGNORECASE
)

def tokenize_exp11(text):

    return TOKEN_PATTERN_EXP11.findall(
        str(text).lower()
    )

# ============================================================
# 5. АУГМЕНТАЦИЯ ТОЛЬКО NEUTRAL
# ============================================================

print("\n" + "=" * 55)
print("АУГМЕНТАЦИЯ ТОЛЬКО NEUTRAL")
print("=" * 55)

rng_exp11 = random.Random(SEED_EXP11)

neutral_indices_exp11 = [
    i
    for i, label in enumerate(train_labels_exp11)
    if label == NEUTRAL_CLASS_EXP11
    and len(tokenize_exp11(train_texts_exp11[i])) > 1
]

neutral_count_exp11 = sum(
    label == NEUTRAL_CLASS_EXP11
    for label in train_labels_exp11
)

target_extra_exp11 = round(
    neutral_count_exp11 * AUGMENT_FRACTION_EXP11
)

if not neutral_indices_exp11:
    raise RuntimeError(
        "Не найдены нейтральные тексты для аугментации."
    )

selected_indices_exp11 = rng_exp11.choices(
    neutral_indices_exp11,
    k=target_extra_exp11
)

def delete_one_word_exp11(text):

    tokens = tokenize_exp11(text)

    if len(tokens) <= 1:
        return " ".join(tokens)

    delete_idx = rng_exp11.randrange(len(tokens))

    del tokens[delete_idx]

    return " ".join(tokens)


augmented_texts_exp11 = list(train_texts_exp11)
augmented_labels_exp11 = list(train_labels_exp11)

for idx in selected_indices_exp11:

    augmented_texts_exp11.append(
        delete_one_word_exp11(
            train_texts_exp11[idx]
        )
    )

    augmented_labels_exp11.append(
        NEUTRAL_CLASS_EXP11
    )

print("Neutral до аугментации:", neutral_count_exp11)
print("Добавлено примеров:", target_extra_exp11)

print("\nВсего обучающих примеров:")
print(len(augmented_texts_exp11))

print("\nРаспределение после аугментации:")

print(
    pd.Series(augmented_labels_exp11)
    .value_counts()
    .sort_index()
)

# ============================================================
# 6. DATASET
# ============================================================

class SentimentDatasetExp11(Dataset):

    def __init__(self, texts, labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        tokens = tokenize_exp11(
            self.texts[idx]
        )

        ids = [
            word2idx_new.get(
                token,
                UNK_IDX_EXP11
            )
            for token in tokens[:MAX_LEN_EXP11]
        ]

        if len(ids) < MAX_LEN_EXP11:

            ids += [
                PAD_IDX_EXP11
            ] * (MAX_LEN_EXP11 - len(ids))

        return (
            torch.tensor(ids, dtype=torch.long),
            torch.tensor(
                int(self.labels[idx]),
                dtype=torch.long
            )
        )


train_dataset_exp11 = SentimentDatasetExp11(
    augmented_texts_exp11,
    augmented_labels_exp11
)

val_dataset_exp11 = SentimentDatasetExp11(
    val_texts_exp11,
    val_labels_exp11
)

generator_exp11 = torch.Generator()
generator_exp11.manual_seed(SEED_EXP11)

train_loader_exp11 = DataLoader(
    train_dataset_exp11,
    batch_size=BATCH_SIZE_EXP11,
    shuffle=True,
    generator=generator_exp11,
    num_workers=0,
    pin_memory=(DEVICE_EXP11.type == "cuda")
)

val_loader_exp11 = DataLoader(
    val_dataset_exp11,
    batch_size=BATCH_SIZE_EXP11,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE_EXP11.type == "cuda")
)

print("\nDataLoader созданы.")

print("Train batches:", len(train_loader_exp11))
print("Validation batches:", len(val_loader_exp11))

# ============================================================
# 7. АРХИТЕКТУРА TEXTCNN
# ============================================================

class TextCNNExp11(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX_EXP11
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

        self.min_seq_len = max(kernel_sizes)

    def forward(self, x):

        x = self.embedding(x)

        x = x.transpose(1, 2)

        if x.size(2) < self.min_seq_len:
            x = F.pad(
                x,
                (0, self.min_seq_len - x.size(2))
            )

        pooled_outputs = []

        for conv in self.convs:

            h = F.relu(conv(x))

            pooled = F.max_pool1d(
                h,
                kernel_size=h.size(2)
            ).squeeze(2)

            pooled_outputs.append(pooled)

        x = torch.cat(
            pooled_outputs,
            dim=1
        )

        x = self.dropout(x)

        return self.fc(x)


model_exp11 = TextCNNExp11(
    vocab_size=VOCAB_SIZE_EXP11,
    embedding_dim=EMBED_DIM_EXP11,
    num_filters=NUM_FILTERS_EXP11,
    kernel_sizes=KERNEL_SIZES_EXP11,
    dropout=DROPOUT_EXP11,
    num_classes=3
).to(DEVICE_EXP11)

print("\nМодель создана!")

print(
    "Количество параметров:",
    sum(
        p.numel()
        for p in model_exp11.parameters()
    )
)

# ============================================================
# 8. LOSS И OPTIMIZER
# ============================================================

criterion_exp11 = nn.CrossEntropyLoss()

optimizer_exp11 = torch.optim.AdamW(
    model_exp11.parameters(),
    lr=LR_EXP11,
    weight_decay=WEIGHT_DECAY_EXP11
)

# ============================================================
# 9. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_exp11(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP11)
        labels = labels.to(DEVICE_EXP11)

        optimizer_exp11.zero_grad()

        logits = model(inputs)

        loss = criterion_exp11(
            logits,
            labels
        )

        loss.backward()
        optimizer_exp11.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 10. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_exp11(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP11)
        labels = labels.to(DEVICE_EXP11)

        logits = model(inputs)

        loss = criterion_exp11(
            logits,
            labels
        )

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(
            labels.cpu().tolist()
        )

        all_pred.extend(
            predictions.cpu().tolist()
        )

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )

# ============================================================
# 11. ОБУЧЕНИЕ
# ============================================================

best_f1_exp11 = -1.0
best_epoch_exp11 = 0
best_state_exp11 = None

patience_counter_exp11 = 0
history_exp11 = []

start_time_exp11 = time.time()

print("\n" + "=" * 55)
print("ОБУЧЕНИЕ TEXTCNN + NEUTRAL-ONLY AUGMENTATION")
print("=" * 55)

for epoch in range(1, EPOCHS_EXP11 + 1):

    train_loss = train_epoch_exp11(
        model_exp11,
        train_loader_exp11
    )

    val_loss, y_true, y_pred = evaluate_exp11(
        model_exp11,
        val_loader_exp11
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_exp11.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1_exp11:

        best_f1_exp11 = macro_f1
        best_epoch_exp11 = epoch

        best_state_exp11 = {
            key: value.detach().cpu().clone()
            for key, value in model_exp11.state_dict().items()
        }

        patience_counter_exp11 = 0

    else:
        patience_counter_exp11 += 1

    if patience_counter_exp11 >= PATIENCE_EXP11:
        print("\nEarly stopping")
        break

# ============================================================
# 12. ВОССТАНОВЛЕНИЕ ЛУЧШЕЙ МОДЕЛИ
# ============================================================

if best_state_exp11 is None:
    raise RuntimeError(
        "Не удалось сохранить лучшую модель."
    )

model_exp11.load_state_dict(
    best_state_exp11
)

model_exp11.eval()

# ============================================================
# 13. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

_, y_true_exp11, y_pred_exp11 = evaluate_exp11(
    model_exp11,
    val_loader_exp11
)

final_f1_exp11 = f1_score(
    y_true_exp11,
    y_pred_exp11,
    average="macro",
    zero_division=0
)

final_accuracy_exp11 = accuracy_score(
    y_true_exp11,
    y_pred_exp11
)

elapsed_exp11 = (
    time.time() - start_time_exp11
) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 11 — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_exp11)
print(f"Macro-F1: {final_f1_exp11:.4f}")
print(f"Accuracy: {final_accuracy_exp11:.4f}")
print(f"Время обучения: {elapsed_exp11:.1f} мин")

print("\nClassification Report:")

print(
    classification_report(
        y_true_exp11,
        y_pred_exp11,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true_exp11,
        y_pred_exp11,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 14. СОХРАНЕНИЕ
# ============================================================

torch.save(
    {
        "model_state_dict": model_exp11.state_dict(),
        "best_epoch": best_epoch_exp11,
        "macro_f1": final_f1_exp11,
        "config": {
            "embedding_dim": EMBED_DIM_EXP11,
            "num_filters": NUM_FILTERS_EXP11,
            "kernel_sizes": KERNEL_SIZES_EXP11,
            "dropout": DROPOUT_EXP11,
            "augmentation_class": NEUTRAL_CLASS_EXP11,
            "augmentation_fraction": AUGMENT_FRACTION_EXP11
        }
    },
    MODEL_PATH_EXP11
)

pd.DataFrame(history_exp11).to_csv(
    HISTORY_PATH_EXP11,
    index=False
)

print("\nМодель сохранена:", MODEL_PATH_EXP11)
print("История сохранена:", HISTORY_PATH_EXP11)

# ============================================================
# 15. СРАВНЕНИЕ
# ============================================================

baseline_f1_exp11 = 0.7335
aug_both_f1_exp11 = 0.7351
mild_weights_f1_exp11 = 0.7349
best_bilstm_f1_exp11 = 0.7650

print("\n" + "=" * 55)
print("СРАВНЕНИЕ ЭКСПЕРИМЕНТОВ")
print("=" * 55)

print(
    f"TextCNN baseline:                {baseline_f1_exp11:.4f}"
)
print(
    f"TextCNN + Neutral/Negative Aug:  {aug_both_f1_exp11:.4f}"
)
print(
    f"TextCNN + Mild Class Weights:    {mild_weights_f1_exp11:.4f}"
)
print(
    f"TextCNN + Neutral-only Aug:      {final_f1_exp11:.4f}"
)
print(
    f"BiLSTM + Navec:                  {best_bilstm_f1_exp11:.4f}"
)

print(
    "\nРазница с baseline:",
    f"{final_f1_exp11 - baseline_f1_exp11:+.4f}"
)

print(
    "Разница с аугментацией двух классов:",
    f"{final_f1_exp11 - aug_both_f1_exp11:+.4f}"
)

print(
    "Разница с лучшей BiLSTM:",
    f"{final_f1_exp11 - best_bilstm_f1_exp11:+.4f}"
)

## Эксперимент 12. TextCNN + Focal Loss: подбор gamma

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 12. TEXTCNN + FOCAL LOSS
# GAMMA = 1.0 И GAMMA = 1.5
# ИСПРАВЛЕННАЯ ВЕРСИЯ
# ============================================================

import re
import gc
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required_exp12 = [
    "train_new",
    "X_val_new",
    "y_val_new",
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "MAX_LEN_NEW",
    "DEVICE_NEW",
    "CLASS_NAMES_NEW"
]

missing_exp12 = [
    name for name in required_exp12
    if name not in globals()
]

if missing_exp12:
    raise RuntimeError(
        "Не найдены переменные: "
        + ", ".join(missing_exp12)
    )

DEVICE_EXP12 = torch.device(DEVICE_NEW)

SEED_EXP12 = 42

EMBED_DIM_EXP12 = 128
NUM_FILTERS_EXP12 = 128
KERNEL_SIZES_EXP12 = (3, 4, 5)
DROPOUT_EXP12 = 0.5

BATCH_SIZE_EXP12 = 64
MAX_EPOCHS_EXP12 = 20
PATIENCE_EXP12 = 5

LR_EXP12 = 5e-4
WEIGHT_DECAY_EXP12 = 1e-4

ALPHA_EXP12 = [1.5, 0.75, 1.0]
GAMMAS_EXP12 = [1.0, 1.5]

MAX_LEN_EXP12 = MAX_LEN_NEW
PAD_IDX_EXP12 = 0
UNK_IDX_EXP12 = 1

print("Устройство:", DEVICE_EXP12)

# ============================================================
# 2. SEED
# ============================================================

def set_seed_exp12(seed=42):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed_exp12(SEED_EXP12)

# ============================================================
# 3. ПОДГОТОВКА ДАННЫХ
# ============================================================

train_texts_exp12 = (
    train_new["text"]
    .fillna("")
    .astype(str)
    .tolist()
)

train_labels_exp12 = (
    train_new["sentiment"]
    .astype(int)
    .tolist()
)

val_texts_exp12 = [
    "" if pd.isna(x) else str(x)
    for x in X_val_new
]

val_labels_exp12 = [
    int(x) for x in y_val_new
]

print("\n" + "=" * 55)
print("ДАННЫЕ")
print("=" * 55)

print("Train:", len(train_texts_exp12))
print("Validation:", len(val_texts_exp12))

print("\nРаспределение классов:")

print(
    pd.Series(train_labels_exp12)
    .value_counts()
    .sort_index()
)

# ============================================================
# 4. ТОКЕНИЗАЦИЯ
# ============================================================

TOKEN_PATTERN_EXP12 = re.compile(
    r"[а-яёa-z0-9]+"
)

def tokenize_exp12(text):

    return TOKEN_PATTERN_EXP12.findall(
        str(text).lower()
    )

# ============================================================
# 5. DATASET С PADDING И TRUNCATION
# ============================================================

class SentimentDatasetExp12(Dataset):

    def __init__(self, texts, labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):

        return len(self.labels)

    def __getitem__(self, idx):

        tokens = tokenize_exp12(
            self.texts[idx]
        )

        tokens = tokens[:MAX_LEN_EXP12]

        ids = [
            word2idx_new.get(
                token,
                UNK_IDX_EXP12
            )
            for token in tokens
        ]

        # Дополняем до одинаковой длины
        padding_length = (
            MAX_LEN_EXP12 - len(ids)
        )

        if padding_length > 0:

            ids += (
                [PAD_IDX_EXP12] * padding_length
            )

        return (
            torch.tensor(
                ids,
                dtype=torch.long
            ),
            torch.tensor(
                int(self.labels[idx]),
                dtype=torch.long
            )
        )


train_dataset_exp12 = SentimentDatasetExp12(
    train_texts_exp12,
    train_labels_exp12
)

val_dataset_exp12 = SentimentDatasetExp12(
    val_texts_exp12,
    val_labels_exp12
)

# Проверяем размеры до обучения
sample_x_exp12, sample_y_exp12 = (
    train_dataset_exp12[0]
)

assert sample_x_exp12.shape == (
    MAX_LEN_EXP12,
)

assert sample_y_exp12.ndim == 0

print("\nDataset созданы.")
print("Длина последовательности:", MAX_LEN_EXP12)
print("Проверка padding: OK")

# ============================================================
# 6. TEXTCNN
# ============================================================

class TextCNNExp12(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=128,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):

        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX_EXP12
        )

        self.convs = nn.ModuleList([
            nn.Conv1d(
                embedding_dim,
                num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

    def forward(self, x):

        x = self.embedding(x)
        x = x.transpose(1, 2)

        pooled = []

        for conv in self.convs:

            h = F.relu(conv(x))

            h = F.max_pool1d(
                h,
                kernel_size=h.size(2)
            ).squeeze(2)

            pooled.append(h)

        x = torch.cat(pooled, dim=1)
        x = self.dropout(x)

        return self.fc(x)

# ============================================================
# 7. FOCAL LOSS
# ============================================================

class FocalLossExp12(nn.Module):

    def __init__(self, alpha, gamma=2.0):

        super().__init__()

        self.gamma = gamma

        self.register_buffer(
            "alpha",
            torch.tensor(
                alpha,
                dtype=torch.float32
            )
        )

    def forward(self, logits, targets):

        ce_loss = F.cross_entropy(
            logits,
            targets,
            reduction="none"
        )

        pt = torch.exp(-ce_loss)

        alpha_t = self.alpha[targets]

        focal_loss = (
            alpha_t
            * ((1.0 - pt) ** self.gamma)
            * ce_loss
        )

        return focal_loss.mean()

# ============================================================
# 8. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_exp12(
    model,
    loader,
    optimizer,
    criterion
):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP12)
        labels = labels.to(DEVICE_EXP12)

        optimizer.zero_grad()

        logits = model(inputs)

        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

# ============================================================
# 9. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_exp12(
    model,
    loader,
    criterion
):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP12)
        labels = labels.to(DEVICE_EXP12)

        logits = model(inputs)

        loss = criterion(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(
            labels.cpu().tolist()
        )

        all_pred.extend(
            predictions.cpu().tolist()
        )

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )

# ============================================================
# 10. ОБУЧЕНИЕ ДВУХ МОДЕЛЕЙ
# ============================================================

results_exp12 = []
histories_exp12 = {}

for gamma in GAMMAS_EXP12:

    print("\n" + "=" * 60)
    print(
        f"TEXTCNN + FOCAL LOSS | GAMMA = {gamma}"
    )
    print("=" * 60)

    # Одинаковый seed для обеих моделей
    set_seed_exp12(SEED_EXP12)

    generator_exp12 = torch.Generator()
    generator_exp12.manual_seed(SEED_EXP12)

    train_loader_exp12 = DataLoader(
        train_dataset_exp12,
        batch_size=BATCH_SIZE_EXP12,
        shuffle=True,
        generator=generator_exp12,
        num_workers=0,
        pin_memory=(DEVICE_EXP12.type == "cuda")
    )

    val_loader_exp12 = DataLoader(
        val_dataset_exp12,
        batch_size=BATCH_SIZE_EXP12,
        shuffle=False,
        num_workers=0,
        pin_memory=(DEVICE_EXP12.type == "cuda")
    )

    model_exp12 = TextCNNExp12(
        vocab_size=VOCAB_SIZE_NEW,
        embedding_dim=EMBED_DIM_EXP12,
        num_filters=NUM_FILTERS_EXP12,
        kernel_sizes=KERNEL_SIZES_EXP12,
        dropout=DROPOUT_EXP12,
        num_classes=3
    ).to(DEVICE_EXP12)

    criterion_exp12 = FocalLossExp12(
        alpha=ALPHA_EXP12,
        gamma=gamma
    ).to(DEVICE_EXP12)

    optimizer_exp12 = torch.optim.AdamW(
        model_exp12.parameters(),
        lr=LR_EXP12,
        weight_decay=WEIGHT_DECAY_EXP12
    )

    print(
        "Количество параметров:",
        sum(
            p.numel()
            for p in model_exp12.parameters()
        )
    )

    best_f1 = -1.0
    best_epoch = 0
    best_state = None

    patience_counter = 0
    history = []

    start_time = time.time()

    # ========================================================
    # 11. ЦИКЛ ЭПОХ
    # ========================================================

    for epoch in range(
        1,
        MAX_EPOCHS_EXP12 + 1
    ):

        train_loss = train_epoch_exp12(
            model_exp12,
            train_loader_exp12,
            optimizer_exp12,
            criterion_exp12
        )

        val_loss, y_true, y_pred = evaluate_exp12(
            model_exp12,
            val_loader_exp12,
            criterion_exp12
        )

        macro_f1 = f1_score(
            y_true,
            y_pred,
            average="macro",
            zero_division=0
        )

        accuracy = accuracy_score(
            y_true,
            y_pred
        )

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "macro_f1": macro_f1,
            "accuracy": accuracy
        })

        print(
            f"Epoch {epoch:02d} | "
            f"Train Loss: {train_loss:.4f} | "
            f"Val Loss: {val_loss:.4f} | "
            f"Macro-F1: {macro_f1:.4f} | "
            f"Accuracy: {accuracy:.4f}",
            flush=True
        )

        if macro_f1 > best_f1:

            best_f1 = macro_f1
            best_epoch = epoch

            best_state = {
                key: value.detach().cpu().clone()
                for key, value
                in model_exp12.state_dict().items()
            }

            patience_counter = 0

        else:

            patience_counter += 1

        if patience_counter >= PATIENCE_EXP12:

            print("\nEarly stopping")
            break

    # ========================================================
    # 12. ВОССТАНОВЛЕНИЕ ЛУЧШЕЙ МОДЕЛИ
    # ========================================================

    if best_state is None:
        raise RuntimeError(
            "Лучшая модель не найдена."
        )

    model_exp12.load_state_dict(best_state)

    _, y_true, y_pred = evaluate_exp12(
        model_exp12,
        val_loader_exp12,
        criterion_exp12
    )

    final_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    final_accuracy = accuracy_score(
        y_true,
        y_pred
    )

    elapsed = (
        time.time() - start_time
    ) / 60

    print("\n" + "=" * 55)
    print(f"ИТОГИ GAMMA = {gamma}")
    print("=" * 55)

    print("Лучшая эпоха:", best_epoch)
    print(f"Macro-F1: {final_f1:.4f}")
    print(f"Accuracy: {final_accuracy:.4f}")
    print(f"Время: {elapsed:.1f} мин")

    print("\nClassification Report:")

    print(
        classification_report(
            y_true,
            y_pred,
            labels=[0, 1, 2],
            target_names=CLASS_NAMES_NEW,
            digits=4,
            zero_division=0
        )
    )

    print("\nConfusion Matrix:")

    print(
        confusion_matrix(
            y_true,
            y_pred,
            labels=[0, 1, 2]
        )
    )

    # ========================================================
    # 13. СОХРАНЕНИЕ МОДЕЛИ
    # ========================================================

    gamma_name = str(gamma).replace(".", "_")

    model_path = (
        f"/content/textcnn_focal_gamma_{gamma_name}_best.pt"
    )

    history_path = (
        f"/content/textcnn_focal_gamma_{gamma_name}_history.csv"
    )

    torch.save(
        {
            "model_state_dict": model_exp12.state_dict(),
            "best_epoch": best_epoch,
            "macro_f1": final_f1,
            "accuracy": final_accuracy,
            "gamma": gamma,
            "alpha": ALPHA_EXP12,
            "config": {
                "vocab_size": VOCAB_SIZE_NEW,
                "max_len": MAX_LEN_EXP12,
                "embedding_dim": EMBED_DIM_EXP12,
                "num_filters": NUM_FILTERS_EXP12,
                "kernel_sizes": KERNEL_SIZES_EXP12,
                "dropout": DROPOUT_EXP12
            }
        },
        model_path
    )

    pd.DataFrame(history).to_csv(
        history_path,
        index=False
    )

    results_exp12.append({
        "gamma": gamma,
        "macro_f1": final_f1,
        "accuracy": final_accuracy,
        "best_epoch": best_epoch,
        "training_minutes": round(elapsed, 2)
    })

    histories_exp12[gamma] = history

    print("\nМодель сохранена:", model_path)
    print("История сохранена:", history_path)

    # ========================================================
    # 14. ОЧИСТКА ПАМЯТИ
    # ========================================================

    del model_exp12
    del optimizer_exp12
    del criterion_exp12
    del best_state

    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# ============================================================
# 15. ИТОГОВОЕ СРАВНЕНИЕ
# ============================================================

results_df_exp12 = pd.DataFrame(
    results_exp12
).sort_values(
    "macro_f1",
    ascending=False
)

print("\n" + "=" * 60)
print("ЭКСПЕРИМЕНТ 12 — ИТОГИ")
print("=" * 60)

print(
    results_df_exp12.to_string(index=False)
)

best_result_exp12 = results_df_exp12.iloc[0]

print(
    "\nЛучшая gamma:",
    best_result_exp12["gamma"]
)

print(
    "Лучший Macro-F1:",
    f"{best_result_exp12['macro_f1']:.4f}"
)

print("\nКонтрольные результаты:")

print("TextCNN baseline:             0.7335")
print("TextCNN + Aug + Focal gamma2: 0.7301")
print("BiLSTM + Navec:               0.7650")

print(
    "\nРазница с baseline:",
    f"{best_result_exp12['macro_f1'] - 0.7335:+.4f}"
)

print(
    "Разница с лучшей BiLSTM:",
    f"{best_result_exp12['macro_f1'] - 0.7650:+.4f}"
)

RESULTS_PATH_EXP12 = (
    "/content/textcnn_focal_gamma_comparison.csv"
)

results_df_exp12.to_csv(
    RESULTS_PATH_EXP12,
    index=False
)

print("\nСравнение сохранено:", RESULTS_PATH_EXP12)

## Эксперимент 13. TextCNN + FastText

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 13. TEXTCNN + RUSSIAN FASTTEXT
# НОВЫЙ ОБЪЕДИНЁННЫЙ ДАТАСЕТ
# ============================================================

import os
import re
import gc
import time
import random
import subprocess
import sys
import urllib.request

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required_exp13 = [
    "train_new",
    "X_val_new",
    "y_val_new",
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "MAX_LEN_NEW",
    "DEVICE_NEW",
    "CLASS_NAMES_NEW"
]

missing_exp13 = [
    name for name in required_exp13
    if name not in globals()
]

if missing_exp13:
    raise RuntimeError(
        "Не найдены переменные: "
        + ", ".join(missing_exp13)
    )

if not {"text", "sentiment"}.issubset(train_new.columns):
    raise ValueError(
        "В train_new должны быть колонки text и sentiment."
    )

DEVICE_EXP13 = torch.device(DEVICE_NEW)

print("Устройство:", DEVICE_EXP13)

# ============================================================
# 2. НАСТРОЙКИ
# ============================================================

SEED_EXP13 = 42

EMBED_DIM_EXP13 = 300
NUM_FILTERS_EXP13 = 128
KERNEL_SIZES_EXP13 = (3, 4, 5)
DROPOUT_EXP13 = 0.5

BATCH_SIZE_EXP13 = 64
EPOCHS_EXP13 = 20
PATIENCE_EXP13 = 5

LR_EXP13 = 5e-4
WEIGHT_DECAY_EXP13 = 1e-4

MAX_LEN_EXP13 = int(MAX_LEN_NEW)
VOCAB_SIZE_EXP13 = int(VOCAB_SIZE_NEW)

PAD_IDX_EXP13 = 0
UNK_IDX_EXP13 = 1

FASTTEXT_DIR_EXP13 = "/content/fasttext_ru"
FASTTEXT_BIN_EXP13 = (
    FASTTEXT_DIR_EXP13 + "/cc.ru.300.bin"
)
FASTTEXT_BZ2_EXP13 = (
    FASTTEXT_DIR_EXP13 + "/cc.ru.300.bin.gz"
)

FASTTEXT_URL_EXP13 = (
    "https://dl.fbaipublicfiles.com/"
    "fasttext/vectors-crawl/cc.ru.300.bin.gz"
)

EMBED_CACHE_EXP13 = (
    FASTTEXT_DIR_EXP13 + "/embedding_matrix.npy"
)

MODEL_PATH_EXP13 = (
    "/content/textcnn_fasttext_ru_best.pt"
)

HISTORY_PATH_EXP13 = (
    "/content/textcnn_fasttext_ru_history.csv"
)

os.makedirs(
    FASTTEXT_DIR_EXP13,
    exist_ok=True
)

# ============================================================
# 3. SEED
# ============================================================

def set_seed_exp13(seed=42):

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed_exp13(SEED_EXP13)

# ============================================================
# 4. УСТАНОВКА FASTTEXT
# ============================================================

try:
    import fasttext

except ImportError:

    print("Устанавливаем fasttext...")

    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "fasttext"
    ])

    import fasttext

print("FastText установлен.")

# ============================================================
# 5. ЗАГРУЗКА РУССКОЯЗЫЧНОЙ МОДЕЛИ
# ============================================================

print("\n" + "=" * 55)
print("ПОДГОТОВКА FASTTEXT")
print("=" * 55)

if not os.path.exists(EMBED_CACHE_EXP13):

    if not os.path.exists(FASTTEXT_BIN_EXP13):

        if not os.path.exists(FASTTEXT_BZ2_EXP13):

            print(
                "Скачиваем cc.ru.300.bin.gz..."
            )

            urllib.request.urlretrieve(
                FASTTEXT_URL_EXP13,
                FASTTEXT_BZ2_EXP13
            )

            print("Архив скачан.")

        print("Распаковываем FastText...")

        import gzip
        import shutil

        with gzip.open(
            FASTTEXT_BZ2_EXP13,
            "rb"
        ) as source:

            with open(
                FASTTEXT_BIN_EXP13,
                "wb"
            ) as destination:

                shutil.copyfileobj(
                    source,
                    destination
                )

        print("Модель распакована.")

    print("Загружаем FastText в память...")

    fasttext_model_exp13 = fasttext.load_model(
        FASTTEXT_BIN_EXP13
    )

    print("FastText загружен.")

else:

    print(
        "Найдена готовая матрица эмбеддингов."
    )

# ============================================================
# 6. СОЗДАНИЕ МАТРИЦЫ ЭМБЕДДИНГОВ
# ============================================================

print("\n" + "=" * 55)
print("МАТРИЦА ЭМБЕДДИНГОВ")
print("=" * 55)

if os.path.exists(EMBED_CACHE_EXP13):

    embedding_matrix_exp13 = np.load(
        EMBED_CACHE_EXP13
    )

    expected_shape = (
        VOCAB_SIZE_EXP13,
        EMBED_DIM_EXP13
    )

    if embedding_matrix_exp13.shape != expected_shape:

        raise ValueError(
            "Размер сохранённой матрицы не совпадает "
            "с текущим словарём. Удалите старый кеш."
        )

    print("Матрица загружена из кеша.")

else:

    embedding_matrix_exp13 = np.zeros(
        (
            VOCAB_SIZE_EXP13,
            EMBED_DIM_EXP13
        ),
        dtype=np.float32
    )

    # FastText создаёт векторы и для OOV-слов
    # благодаря символьным n-граммам.
    #
    # Точные совпадения считаем отдельно.

    exact_matches_exp13 = 0
    total_words_exp13 = 0

    fasttext_words_exp13 = set(
        fasttext_model_exp13.get_words()
    )

    for word, idx in word2idx_new.items():

        if idx in (
            PAD_IDX_EXP13,
            UNK_IDX_EXP13
        ):
            continue

        if idx >= VOCAB_SIZE_EXP13:
            continue

        embedding_matrix_exp13[idx] = (
            fasttext_model_exp13.get_word_vector(word)
        )

        total_words_exp13 += 1

        if word in fasttext_words_exp13:
            exact_matches_exp13 += 1

        if total_words_exp13 % 20000 == 0:

            print(
                f"Обработано слов: "
                f"{total_words_exp13}"
            )

    # UNK оставляем случайным вектором
    rng_exp13 = np.random.default_rng(
        SEED_EXP13
    )

    embedding_matrix_exp13[
        UNK_IDX_EXP13
    ] = rng_exp13.normal(
        0,
        0.05,
        EMBED_DIM_EXP13
    ).astype(np.float32)

    # PAD должен оставаться нулевым
    embedding_matrix_exp13[
        PAD_IDX_EXP13
    ] = 0.0

    np.save(
        EMBED_CACHE_EXP13,
        embedding_matrix_exp13
    )

    print("\nТочных совпадений:")
    print(exact_matches_exp13)

    print("Всего обработано:")
    print(total_words_exp13)

    print(
        "Доля точных совпадений:",
        f"{exact_matches_exp13 / max(total_words_exp13, 1):.2%}"
    )

    print(
        "\nДля OOV-слов FastText использует "
        "символьные n-граммы."
    )

    print(
        "Матрица сохранена:",
        EMBED_CACHE_EXP13
    )

print(
    "Размер матрицы:",
    embedding_matrix_exp13.shape
)

# Освобождаем RAM от исходной модели FastText
if "fasttext_model_exp13" in globals():

    del fasttext_model_exp13
    gc.collect()

# ============================================================
# 7. ПОДГОТОВКА ДАННЫХ
# ============================================================

train_texts_exp13 = (
    train_new["text"]
    .fillna("")
    .astype(str)
    .tolist()
)

train_labels_exp13 = (
    train_new["sentiment"]
    .astype(int)
    .tolist()
)

val_texts_exp13 = [
    "" if pd.isna(x) else str(x)
    for x in X_val_new
]

val_labels_exp13 = [
    int(x) for x in y_val_new
]

TOKEN_PATTERN_EXP13 = re.compile(
    r"[а-яёa-z0-9]+"
)

def tokenize_exp13(text):

    return TOKEN_PATTERN_EXP13.findall(
        str(text).lower()
    )

# ============================================================
# 8. DATASET
# ============================================================

class SentimentDatasetExp13(Dataset):

    def __init__(self, texts, labels):

        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):

        tokens = tokenize_exp13(
            self.texts[idx]
        )

        ids = [
            word2idx_new.get(
                token,
                UNK_IDX_EXP13
            )
            for token in tokens[:MAX_LEN_EXP13]
        ]

        if len(ids) < MAX_LEN_EXP13:

            ids += [
                PAD_IDX_EXP13
            ] * (MAX_LEN_EXP13 - len(ids))

        return (
            torch.tensor(
                ids,
                dtype=torch.long
            ),
            torch.tensor(
                int(self.labels[idx]),
                dtype=torch.long
            )
        )


train_dataset_exp13 = SentimentDatasetExp13(
    train_texts_exp13,
    train_labels_exp13
)

val_dataset_exp13 = SentimentDatasetExp13(
    val_texts_exp13,
    val_labels_exp13
)

generator_exp13 = torch.Generator()
generator_exp13.manual_seed(SEED_EXP13)

train_loader_exp13 = DataLoader(
    train_dataset_exp13,
    batch_size=BATCH_SIZE_EXP13,
    shuffle=True,
    generator=generator_exp13,
    num_workers=0,
    pin_memory=(DEVICE_EXP13.type == "cuda")
)

val_loader_exp13 = DataLoader(
    val_dataset_exp13,
    batch_size=BATCH_SIZE_EXP13,
    shuffle=False,
    num_workers=0,
    pin_memory=(DEVICE_EXP13.type == "cuda")
)

print("\nDataLoader созданы.")
print("Train batches:", len(train_loader_exp13))
print("Validation batches:", len(val_loader_exp13))

# ============================================================
# 9. TEXTCNN С FASTTEXT
# ============================================================

class TextCNNFastTextExp13(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):

        super().__init__()

        matrix = torch.from_numpy(
            embedding_matrix
        ).float()

        self.embedding = nn.Embedding.from_pretrained(
            matrix,
            freeze=False,
            padding_idx=PAD_IDX_EXP13
        )

        embedding_dim = matrix.shape[1]

        self.convs = nn.ModuleList([
            nn.Conv1d(
                embedding_dim,
                num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

    def forward(self, x):

        x = self.embedding(x)
        x = x.transpose(1, 2)

        pooled = []

        for conv in self.convs:

            h = F.relu(conv(x))

            h = F.max_pool1d(
                h,
                kernel_size=h.size(2)
            ).squeeze(2)

            pooled.append(h)

        x = torch.cat(pooled, dim=1)
        x = self.dropout(x)

        return self.fc(x)


model_exp13 = TextCNNFastTextExp13(
    embedding_matrix=embedding_matrix_exp13,
    num_filters=NUM_FILTERS_EXP13,
    kernel_sizes=KERNEL_SIZES_EXP13,
    dropout=DROPOUT_EXP13,
    num_classes=3
).to(DEVICE_EXP13)

print("\nМодель создана!")

print(
    "Количество параметров:",
    sum(
        p.numel()
        for p in model_exp13.parameters()
    )
)

# ============================================================
# 10. LOSS И OPTIMIZER
# ============================================================

criterion_exp13 = nn.CrossEntropyLoss()

optimizer_exp13 = torch.optim.AdamW(
    model_exp13.parameters(),
    lr=LR_EXP13,
    weight_decay=WEIGHT_DECAY_EXP13
)

# ============================================================
# 11. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_exp13(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP13)
        labels = labels.to(DEVICE_EXP13)

        optimizer_exp13.zero_grad()

        logits = model(inputs)

        loss = criterion_exp13(
            logits,
            labels
        )

        loss.backward()
        optimizer_exp13.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples

# ============================================================
# 12. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_exp13(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_EXP13)
        labels = labels.to(DEVICE_EXP13)

        logits = model(inputs)

        loss = criterion_exp13(
            logits,
            labels
        )

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(
            labels.cpu().tolist()
        )

        all_pred.extend(
            predictions.cpu().tolist()
        )

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )

# ============================================================
# 13. ОБУЧЕНИЕ
# ============================================================

best_f1_exp13 = -1.0
best_epoch_exp13 = 0
best_state_exp13 = None

patience_counter_exp13 = 0
history_exp13 = []

start_time_exp13 = time.time()

print("\n" + "=" * 55)
print("ОБУЧЕНИЕ TEXTCNN + FASTTEXT")
print("=" * 55)

for epoch in range(1, EPOCHS_EXP13 + 1):

    train_loss = train_epoch_exp13(
        model_exp13,
        train_loader_exp13
    )

    val_loss, y_true, y_pred = evaluate_exp13(
        model_exp13,
        val_loader_exp13
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_exp13.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f}",
        flush=True
    )

    if macro_f1 > best_f1_exp13:

        best_f1_exp13 = macro_f1
        best_epoch_exp13 = epoch

        best_state_exp13 = {
            key: value.detach().cpu().clone()
            for key, value
            in model_exp13.state_dict().items()
        }

        patience_counter_exp13 = 0

    else:

        patience_counter_exp13 += 1

    if patience_counter_exp13 >= PATIENCE_EXP13:

        print("\nEarly stopping")
        break

# ============================================================
# 14. ВОССТАНОВЛЕНИЕ ЛУЧШЕЙ МОДЕЛИ
# ============================================================

model_exp13.load_state_dict(
    best_state_exp13
)

_, y_true_exp13, y_pred_exp13 = evaluate_exp13(
    model_exp13,
    val_loader_exp13
)

final_f1_exp13 = f1_score(
    y_true_exp13,
    y_pred_exp13,
    average="macro",
    zero_division=0
)

final_accuracy_exp13 = accuracy_score(
    y_true_exp13,
    y_pred_exp13
)

elapsed_exp13 = (
    time.time() - start_time_exp13
) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 13 — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_exp13)
print(f"Macro-F1: {final_f1_exp13:.4f}")
print(f"Accuracy: {final_accuracy_exp13:.4f}")
print(f"Время обучения: {elapsed_exp13:.1f} мин")

print("\nClassification Report:")

print(
    classification_report(
        y_true_exp13,
        y_pred_exp13,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:")

print(
    confusion_matrix(
        y_true_exp13,
        y_pred_exp13,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 15. СОХРАНЕНИЕ
# ============================================================

torch.save(
    {
        "model_state_dict": model_exp13.state_dict(),
        "best_epoch": best_epoch_exp13,
        "macro_f1": final_f1_exp13,
        "accuracy": final_accuracy_exp13,
        "config": {
            "embedding_dim": EMBED_DIM_EXP13,
            "num_filters": NUM_FILTERS_EXP13,
            "kernel_sizes": KERNEL_SIZES_EXP13,
            "dropout": DROPOUT_EXP13,
            "embedding_source": "FastText cc.ru.300.bin",
            "freeze": False
        }
    },
    MODEL_PATH_EXP13
)

pd.DataFrame(history_exp13).to_csv(
    HISTORY_PATH_EXP13,
    index=False
)

print("\nМодель сохранена:", MODEL_PATH_EXP13)
print("История сохранена:", HISTORY_PATH_EXP13)

# ============================================================
# 16. СРАВНЕНИЕ
# ============================================================

print("\n" + "=" * 55)
print("СРАВНЕНИЕ ЭКСПЕРИМЕНТОВ")
print("=" * 55)

print("TextCNN baseline:          0.7335")
print("TextCNN + Navec:           0.7516")
print(
    f"TextCNN + FastText:        {final_f1_exp13:.4f}"
)
print("BiLSTM + Navec:            0.7650")

print(
    "\nРазница с baseline:",
    f"{final_f1_exp13 - 0.7335:+.4f}"
)

print(
    "Разница с TextCNN + Navec:",
    f"{final_f1_exp13 - 0.7516:+.4f}"
)

print(
    "Разница с лучшей BiLSTM:",
    f"{final_f1_exp13 - 0.7650:+.4f}"
)

In [ ]:
import os

print("Проверка файлов:")
print("Датасет:", os.path.exists("/content/merged_sentiment_dataset.csv.gz"))
print("FastText:", os.path.exists("/content/fasttext_ru/cc.ru.300.bin"))
print("Navec:", os.path.exists("/content/navec_hudlit_v1_12B_500K_300d_100q.tar"))

print("\nПроверка переменных:")

for name in [
    "train_new",
    "X_val_new",
    "y_val_new",
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "MAX_LEN_NEW",
    "DEVICE_NEW"
]:
    print(name, "—", "есть" if name in globals() else "нет")

In [ ]:
import os

path = "/content/merged_sentiment_dataset.csv.gz"

if os.path.exists(path):
    print("✅ Датасет загружен успешно!")
else:
    print("❌ Файл пока не найден. Проверь название.")

In [ ]:
import os
import re
import pandas as pd
import torch
from collections import Counter

# 1. Проверяем файл
DATA_PATH_NEW = "/content/merged_sentiment_dataset.csv.gz"

assert os.path.exists(DATA_PATH_NEW), "Файл не найден!"

# 2. Загружаем датасет
df_new = pd.read_csv(DATA_PATH_NEW, compression="gzip")

print("Датасет загружен:", df_new.shape)

# 3. Разделяем на train и validation
train_new = df_new[df_new["split"] == "train"].copy()
val_new = df_new[df_new["split"] == "validation"].copy()

# 4. Токенизация
def tokenize_new(text):
    return re.findall(r"[а-яёa-z0-9]+", str(text).lower())

# 5. Удаляем пустые тексты после токенизации
train_new["text"] = train_new["text"].fillna("").astype(str)
val_new["text"] = val_new["text"].fillna("").astype(str)

train_new = train_new[
    train_new["text"].apply(lambda x: len(tokenize_new(x)) > 0)
].copy()

val_new = val_new[
    val_new["text"].apply(lambda x: len(tokenize_new(x)) > 0)
].copy()

# 6. Формируем переменные
X_train_new = train_new["text"].tolist()
y_train_new = train_new["sentiment"].astype(int).tolist()

X_val_new = val_new["text"].tolist()
y_val_new = val_new["sentiment"].astype(int).tolist()

CLASS_NAMES_NEW = ["Neutral", "Positive", "Negative"]

MAX_LEN_NEW = 256
PAD_IDX_NEW = 0
UNK_IDX_NEW = 1

DEVICE_NEW = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 7. Создаём словарь только по train
counter_new = Counter()

for text in X_train_new:
    counter_new.update(tokenize_new(text))

word2idx_new = {
    "<PAD>": PAD_IDX_NEW,
    "<UNK>": UNK_IDX_NEW
}

for word in counter_new:
    if word not in word2idx_new:
        word2idx_new[word] = len(word2idx_new)

VOCAB_SIZE_NEW = len(word2idx_new)

# 8. Проверяем результаты
print("\n" + "=" * 50)
print("ДАННЫЕ ВОССТАНОВЛЕНЫ")
print("=" * 50)

print("Train:", len(train_new))
print("Validation:", len(val_new))
print("Размер словаря:", VOCAB_SIZE_NEW)
print("Максимальная длина:", MAX_LEN_NEW)
print("Устройство:", DEVICE_NEW)

print("\nРаспределение train:")
print(train_new["sentiment"].value_counts().sort_index())

print("\nРаспределение validation:")
print(val_new["sentiment"].value_counts().sort_index())ё

## 🏆 Промежуточный итог: лучшая нейросеть

По результатам проведённых экспериментов на новом объединённом датасете **лучшей моделью стала BiLSTM + Navec + CrossEntropyLoss**.

| Модель | Macro-F1 | Accuracy |
|---|---:|---:|
| **BiLSTM + Navec + CrossEntropy** 🏆 | **0.7650** | **0.7789** |
| BiLSTM + Navec + Focal Loss | 0.7577 | 0.7570 |
| TextCNN + Navec | 0.7516 | 0.7611 |
| TextCNN baseline | 0.7335 | 0.7384 |

### Вывод

**BiLSTM + Navec** показала наилучшее качество классификации тональности, превзойдя базовую TextCNN на **0.0315 Macro-F1**.

Использование предобученных русскоязычных эмбеддингов Navec в сочетании с двунаправленной LSTM оказалось наиболее эффективным среди проверенных подходов.

*Результаты предварительные: оценка выполнена на валидационной выборке. Эксперимент с FastText ещё не завершён.*


In [ ]:
import os

print("Датасет загружен:", os.path.exists("/content/merged_sentiment_dataset.csv.gz"))

for name in ["X_train_new", "y_train_new", "tokenize_new", "MAX_LEN_NEW", "word2idx_new"]:
    print(name, "—", "есть" if name in globals() else "нет")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

path = "/content/merged_sentiment_dataset.csv.gz"

print("Файл найден:", os.path.exists(path))

if os.path.exists(path):
    print("Размер файла:", round(os.path.getsize(path) / 1024**2, 2), "МБ")

In [ ]:
# ============================================================
# ШАГ 3. Восстановление данных после перезапуска Colab
# ============================================================

import re
import pandas as pd
import numpy as np
import torch

# 1. Загружаем датасет
df_new = pd.read_csv(
    "/content/merged_sentiment_dataset.csv.gz",
    compression="gzip"
)

# 2. Восстанавливаем токенизацию
def tokenize_new(text):
    return re.findall(r"[а-яёa-z0-9]+", str(text).lower())

# 3. Разделяем данные
train_new = df_new[df_new["split"] == "train"].copy()
val_new = df_new[df_new["split"] == "validation"].copy()

# 4. Очищаем тексты
for data in [train_new, val_new]:
    data["text"] = data["text"].fillna("").astype(str)

train_new = train_new[
    train_new["text"].apply(lambda x: len(tokenize_new(x)) > 0)
].copy()

val_new = val_new[
    val_new["text"].apply(lambda x: len(tokenize_new(x)) > 0)
].copy()

# 5. Создаём переменные
X_train_new = train_new["text"].tolist()
y_train_new = train_new["sentiment"].astype(int).to_numpy()

X_val_new = val_new["text"].tolist()
y_val_new = val_new["sentiment"].astype(int).to_numpy()

CLASS_NAMES_NEW = ["Neutral", "Positive", "Negative"]
MAX_LEN_NEW = 256

DEVICE_NEW = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 6. Проверка
print("Train:", len(X_train_new))
print("Validation:", len(X_val_new))
print("MAX_LEN:", MAX_LEN_NEW)
print("Устройство:", DEVICE_NEW)

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 13. ПОДГОТОВКА ДАННЫХ ДЛЯ FASTTEXT
# ============================================================

import random
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Настройки
SEED_NEW = 42
MIN_FREQ_NEW = 2
MAX_LEN_NEW = 256
BATCH_SIZE_NEW = 64

PAD_IDX_NEW = 0
UNK_IDX_NEW = 1

random.seed(SEED_NEW)
np.random.seed(SEED_NEW)
torch.manual_seed(SEED_NEW)

DEVICE_NEW = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Создаём словарь только по обучающей выборке
word_counts_new = Counter()

for text in X_train_new:
    word_counts_new.update(tokenize_new(text))

word2idx_new = {
    "<PAD>": PAD_IDX_NEW,
    "<UNK>": UNK_IDX_NEW
}

for word, count in word_counts_new.items():
    if count >= MIN_FREQ_NEW:
        word2idx_new[word] = len(word2idx_new)

VOCAB_SIZE_NEW = len(word2idx_new)

# 3. Dataset
class SentimentDatasetNew(Dataset):

    def __init__(self, texts, labels):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.encoded = []

        for text in texts:
            tokens = tokenize_new(text)[:MAX_LEN_NEW]

            ids = [
                word2idx_new.get(token, UNK_IDX_NEW)
                for token in tokens
            ]

            self.encoded.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], int(self.labels[idx])


# 4. Дополнение текстов внутри батча
def collate_new(batch):
    texts, labels = zip(*batch)

    max_batch_len = min(
        max(len(text) for text in texts),
        MAX_LEN_NEW
    )

    padded = torch.full(
        (len(texts), max_batch_len),
        PAD_IDX_NEW,
        dtype=torch.long
    )

    for i, ids in enumerate(texts):
        padded[i, :len(ids)] = torch.tensor(
            ids, dtype=torch.long
        )

    return (
        padded,
        torch.tensor(labels, dtype=torch.long)
    )


# 5. Создаём загрузчики данных
train_dataset_new = SentimentDatasetNew(
    X_train_new, y_train_new
)

val_dataset_new = SentimentDatasetNew(
    X_val_new, y_val_new
)

train_loader_new = DataLoader(
    train_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=True,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)

val_loader_new = DataLoader(
    val_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=False,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)

# 6. Проверка
print("Устройство:", DEVICE_NEW)
print("Размер словаря:", VOCAB_SIZE_NEW)
print("Train batches:", len(train_loader_new))
print("Validation batches:", len(val_loader_new))
print("MAX_LEN:", MAX_LEN_NEW)

X_batch, y_batch = next(iter(train_loader_new))

print("Форма текстов:", tuple(X_batch.shape))
print("Форма меток:", tuple(y_batch.shape))

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 13. ШАГ 5 — СКАЧИВАНИЕ FASTTEXT
# ============================================================

import os
import subprocess

FASTTEXT_PATH_NEW = "/content/cc.ru.300.vec.gz"

FASTTEXT_URL_NEW = (
    "https://dl.fbaipublicfiles.com/fasttext/"
    "vectors-crawl/cc.ru.300.vec.gz"
)

if os.path.exists(FASTTEXT_PATH_NEW):
    print("Архив уже существует.")
else:
    print("Скачиваем русскоязычный FastText...")

    subprocess.run(
        [
            "wget",
            "-c",
            "--progress=dot:giga",
            "-O",
            FASTTEXT_PATH_NEW,
            FASTTEXT_URL_NEW
        ],
        check=True
    )

print(
    "Размер архива:",
    round(os.path.getsize(FASTTEXT_PATH_NEW) / 1024**3, 2),
    "ГБ"
)

print("Скачивание завершено.")

## Эксперимент 13. TextCNN + FastText

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 13. ШАГ 6 — FASTTEXT EMBEDDING MATRIX
# ============================================================

import gzip
import os
import time
import numpy as np

FASTTEXT_DIM_NEW = 300

FASTTEXT_MATRIX_PATH_NEW = "/content/fasttext_matrix_new.npy"

start_time = time.time()

# Матрица: случайная инициализация для слов без вектора
rng = np.random.default_rng(42)

fasttext_matrix_new = rng.normal(
    loc=0.0,
    scale=0.05,
    size=(VOCAB_SIZE_NEW, FASTTEXT_DIM_NEW)
).astype(np.float32)

# PAD должен быть нулевым
fasttext_matrix_new[PAD_IDX_NEW] = 0.0

found_new = 0
processed_new = 0

print("Начинаем потоковое чтение FastText...")

with gzip.open(FASTTEXT_PATH_NEW, "rt", encoding="utf-8") as f:

    # Первая строка содержит размер словаря и размерность
    header = f.readline().strip()
    print("Заголовок FastText:", header)

    for line in f:
        processed_new += 1

        # Отделяем слово от вектора
        word, separator, vector_text = line.partition(" ")

        if separator and word in word2idx_new:
            vector = np.fromstring(
                vector_text,
                sep=" ",
                dtype=np.float32
            )

            if len(vector) == FASTTEXT_DIM_NEW:
                idx = word2idx_new[word]
                fasttext_matrix_new[idx] = vector
                found_new += 1

        if processed_new % 500000 == 0:
            print(
                f"Обработано: {processed_new:,} | "
                f"Найдено: {found_new:,} | "
                f"Время: {(time.time() - start_time) / 60:.1f} мин"
            )

# Сохраняем готовую матрицу
np.save(FASTTEXT_MATRIX_PATH_NEW, fasttext_matrix_new)

coverage_new = found_new / (VOCAB_SIZE_NEW - 2) * 100

print("\n" + "=" * 50)
print("FASTTEXT ГОТОВ")
print("=" * 50)

print("Размер матрицы:", fasttext_matrix_new.shape)
print("Найдено слов:", found_new)
print(f"Покрытие словаря: {coverage_new:.2f}%")
print(f"Время: {(time.time() - start_time) / 60:.1f} мин")
print(
    "Размер матрицы на диске:",
    round(os.path.getsize(FASTTEXT_MATRIX_PATH_NEW) / 1024**2, 1),
    "МБ"
)

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 13. TEXTCNN + FASTTEXT
# ============================================================

import copy
import random
import time
import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Проверка переменных
# ------------------------------------------------------------

required = [
    "fasttext_matrix_new",
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "PAD_IDX_NEW",
    "DEVICE_NEW",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Отсутствуют переменные: " + ", ".join(missing)
    )

# ------------------------------------------------------------
# 2. Воспроизводимость
# ------------------------------------------------------------

SEED_NEW = 42

random.seed(SEED_NEW)
np.random.seed(SEED_NEW)
torch.manual_seed(SEED_NEW)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_NEW)

# ------------------------------------------------------------
# 3. Архитектура TextCNN
# ------------------------------------------------------------

class TextCNNFastTextNew(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        num_filters=128,
        kernel_sizes=(3, 4, 5),
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX_NEW
        )

        self.embedding.weight.data.copy_(
            torch.from_numpy(embedding_matrix)
        )

        # Обучаем эмбеддинги вместе с моделью
        self.embedding.weight.requires_grad = True

        self.convs = nn.ModuleList([
            nn.Conv1d(
                in_channels=embedding_dim,
                out_channels=num_filters,
                kernel_size=k
            )
            for k in kernel_sizes
        ])

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            num_filters * len(kernel_sizes),
            num_classes
        )

    def forward(self, x):

        # На случай короткого батча
        min_length = max(
            conv.kernel_size[0] for conv in self.convs
        )

        if x.size(1) < min_length:
            x = nn.functional.pad(
                x,
                (0, min_length - x.size(1)),
                value=PAD_IDX_NEW
            )

        x = self.embedding(x)
        x = x.transpose(1, 2)

        features = []

        for conv in self.convs:
            h = torch.relu(conv(x))
            h = torch.max(h, dim=2).values
            features.append(h)

        x = torch.cat(features, dim=1)
        x = self.dropout(x)

        return self.fc(x)


# ------------------------------------------------------------
# 4. Создаём модель
# ------------------------------------------------------------

model_fasttext_new = TextCNNFastTextNew(
    embedding_matrix=fasttext_matrix_new,
    num_filters=128,
    kernel_sizes=(3, 4, 5),
    dropout=0.5,
    num_classes=3
).to(DEVICE_NEW)

criterion_fasttext_new = nn.CrossEntropyLoss()

optimizer_fasttext_new = torch.optim.AdamW(
    model_fasttext_new.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

print("Устройство:", DEVICE_NEW)

print(
    "Количество параметров:",
    f"{sum(p.numel() for p in model_fasttext_new.parameters()):,}"
)

# ------------------------------------------------------------
# 5. Обучение одной эпохи
# ------------------------------------------------------------

def train_epoch_fasttext_new(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(
            DEVICE_NEW,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE_NEW,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(inputs)
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ------------------------------------------------------------
# 6. Валидация
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_fasttext_new(model, loader, criterion):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    y_true = []
    y_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(
            DEVICE_NEW,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE_NEW,
            non_blocking=True
        )

        logits = model(inputs)
        loss = criterion(logits, labels)

        predictions = torch.argmax(logits, dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

    val_loss = total_loss / total_samples

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(y_true, y_pred)

    return val_loss, macro_f1, accuracy, y_true, y_pred


# ------------------------------------------------------------
# 7. Основной цикл обучения
# ------------------------------------------------------------

EPOCHS_FASTTEXT_NEW = 20
PATIENCE_FASTTEXT_NEW = 5

best_f1_fasttext_new = -1.0
best_epoch_fasttext_new = 0
best_state_fasttext_new = None

patience_counter_fasttext_new = 0
history_fasttext_new = []

start_time = time.time()

for epoch in range(1, EPOCHS_FASTTEXT_NEW + 1):

    epoch_start = time.time()

    train_loss = train_epoch_fasttext_new(
        model_fasttext_new,
        train_loader_new,
        optimizer_fasttext_new,
        criterion_fasttext_new
    )

    val_loss, val_f1, val_acc, _, _ = evaluate_fasttext_new(
        model_fasttext_new,
        val_loader_new,
        criterion_fasttext_new
    )

    history_fasttext_new.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": val_f1,
        "accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {val_f1:.4f} | "
        f"Accuracy: {val_acc:.4f} | "
        f"Time: {time.time() - epoch_start:.1f}s",
        flush=True
    )

    if val_f1 > best_f1_fasttext_new:

        best_f1_fasttext_new = val_f1
        best_epoch_fasttext_new = epoch

        # Храним лучший checkpoint на CPU
        best_state_fasttext_new = {
            key: value.detach().cpu().clone()
            for key, value in model_fasttext_new.state_dict().items()
        }

        patience_counter_fasttext_new = 0

        print("  -> Новая лучшая модель", flush=True)

    else:

        patience_counter_fasttext_new += 1

        if patience_counter_fasttext_new >= PATIENCE_FASTTEXT_NEW:
            print("\nEarly stopping")
            break


# ------------------------------------------------------------
# 8. Восстанавливаем лучшую модель
# ------------------------------------------------------------

model_fasttext_new.load_state_dict(
    best_state_fasttext_new
)

# ------------------------------------------------------------
# 9. Финальная оценка
# ------------------------------------------------------------

val_loss, val_f1, val_acc, y_true, y_pred = evaluate_fasttext_new(
    model_fasttext_new,
    val_loader_new,
    criterion_fasttext_new
)

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 13 — TEXTCNN + FASTTEXT")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_fasttext_new)
print(f"Macro-F1: {val_f1:.4f}")
print(f"Accuracy: {val_acc:.4f}")

print(
    f"Время обучения: "
    f"{(time.time() - start_time) / 60:.1f} мин"
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )
)

# ------------------------------------------------------------
# 10. Сохраняем результаты
# ------------------------------------------------------------

import pandas as pd

pd.DataFrame(history_fasttext_new).to_csv(
    "/content/experiment_13_fasttext_history.csv",
    index=False
)

torch.save(
    {
        "model_state_dict": best_state_fasttext_new,
        "best_epoch": best_epoch_fasttext_new,
        "macro_f1": val_f1,
        "accuracy": val_acc
    },
    "/content/experiment_13_fasttext_best.pt"
)

print("\nРезультаты и лучшая модель сохранены в /content/")

### Эксперимент 13. Результаты TextCNN + FastText

Использованы предобученные русскоязычные эмбеддинги FastText (300 измерений). Покрытие словаря составило **81.47%**.

**Результаты:**
- Macro-F1: **0.7523**
- Accuracy: **0.7617**
- Лучшая эпоха: **1**
- Время обучения: **16.4 минуты**

**Сравнение:**

| Модель | Macro-F1 |
|---|---:|
| BiLSTM + Navec + CrossEntropy | **0.7650** |
| TextCNN + FastText | 0.7523 |
| TextCNN + Navec | 0.7516 |
| TextCNN baseline | 0.7335 |

**Вывод:** FastText улучшил результат базовой TextCNN на 0.0188 Macro-F1, но практически не превзошёл Navec (+0.0007). Несмотря на более высокое покрытие словаря, существенного преимущества FastText не выявлено.

Наблюдается выраженное переобучение после первой эпохи.

**Лучшая модель исследования на текущем этапе — BiLSTM + Navec + CrossEntropy (Macro-F1 = 0.7650).**

## Эксперимент 14. BiLSTM + FastText + CrossEntropyLoss

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 14. BiLSTM + FASTTEXT + CROSSENTROPY
# ============================================================

import os
import gc
import copy
import time
import random

import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required = [
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "PAD_IDX_NEW",
    "DEVICE_NEW",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing)
    )

FASTTEXT_MATRIX_PATH_NEW = "/content/fasttext_matrix_new.npy"

if not os.path.exists(FASTTEXT_MATRIX_PATH_NEW):
    raise FileNotFoundError(
        "Не найдена матрица FastText. "
        "Сначала выполни шаг 6 эксперимента 13."
    )

# Загружаем матрицу без создания лишней копии в RAM
fasttext_matrix_14 = np.load(
    FASTTEXT_MATRIX_PATH_NEW,
    mmap_mode="r"
)

assert fasttext_matrix_14.shape == (
    VOCAB_SIZE_NEW, 300
), "Размер матрицы FastText не совпадает со словарём!"

print("Устройство:", DEVICE_NEW)
print("Размер словаря:", VOCAB_SIZE_NEW)
print("Матрица FastText:", fasttext_matrix_14.shape)

# ============================================================
# 2. ВОСПРОИЗВОДИМОСТЬ
# ============================================================

SEED_14 = 42

random.seed(SEED_14)
np.random.seed(SEED_14)
torch.manual_seed(SEED_14)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_14)

# ============================================================
# 3. ОСВОБОЖДАЕМ ПАМЯТЬ ОТ ЭКСПЕРИМЕНТА 13
# ============================================================

# Удаляем только модель и оптимизатор эксперимента 13.
# Данные и результаты предыдущих экспериментов не трогаем.

for variable_name in [
    "model_fasttext_new",
    "optimizer_fasttext_new",
    "criterion_fasttext_new",
    "fasttext_matrix_new",
    "best_state_fasttext_new"
]:
    if variable_name in globals():
        del globals()[variable_name]

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("Память после эксперимента 13 очищена.")

# ============================================================
# 4. АРХИТЕКТУРА BiLSTM
# ============================================================

class BiLSTMFastText14(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        hidden_dim=128,
        num_layers=1,
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX_NEW
        )

        # Загружаем предобученные FastText-векторы
        with torch.no_grad():
            # Копируем частями, чтобы избежать большой
            # дополнительной копии матрицы в RAM
            chunk_size = 10000

            for start in range(0, vocab_size, chunk_size):
                end = min(start + chunk_size, vocab_size)

                chunk = np.array(
                    embedding_matrix[start:end],
                    copy=True
                )

                self.embedding.weight[start:end].copy_(
                    torch.from_numpy(chunk)
                )

            self.embedding.weight[PAD_IDX_NEW].zero_()

        # Эмбеддинги дообучаются
        self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x):

        # x: [batch, seq_len]
        lengths = (x != PAD_IDX_NEW).sum(dim=1)
        lengths = lengths.clamp(min=1)

        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        # Последние скрытые состояния
        # прямого и обратного направлений
        forward_hidden = hidden[-2]
        backward_hidden = hidden[-1]

        features = torch.cat(
            [forward_hidden, backward_hidden],
            dim=1
        )

        features = self.dropout(features)

        return self.fc(features)


# ============================================================
# 5. СОЗДАНИЕ МОДЕЛИ
# ============================================================

model_14 = BiLSTMFastText14(
    embedding_matrix=fasttext_matrix_14,
    hidden_dim=128,
    num_layers=1,
    dropout=0.5,
    num_classes=3
).to(DEVICE_NEW)

criterion_14 = nn.CrossEntropyLoss()

optimizer_14 = torch.optim.AdamW(
    model_14.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

print(
    "Количество параметров:",
    f"{sum(p.numel() for p in model_14.parameters()):,}"
)

# ============================================================
# 6. ОБУЧЕНИЕ ОДНОЙ ЭПОХИ
# ============================================================

def train_epoch_14(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(
            DEVICE_NEW,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE_NEW,
            non_blocking=True
        )

        optimizer.zero_grad(set_to_none=True)

        logits = model(inputs)

        loss = criterion(logits, labels)

        loss.backward()

        # Защита от взрыва градиентов
        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 7. ВАЛИДАЦИЯ
# ============================================================

@torch.no_grad()
def evaluate_14(model, loader, criterion):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    y_true = []
    y_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(
            DEVICE_NEW,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE_NEW,
            non_blocking=True
        )

        logits = model(inputs)

        loss = criterion(logits, labels)

        predictions = torch.argmax(
            logits,
            dim=1
        )

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        y_true.extend(
            labels.cpu().numpy()
        )

        y_pred.extend(
            predictions.cpu().numpy()
        )

    val_loss = total_loss / total_samples

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    return (
        val_loss,
        macro_f1,
        accuracy,
        y_true,
        y_pred
    )


# ============================================================
# 8. ОСНОВНОЙ ЦИКЛ ОБУЧЕНИЯ
# ============================================================

EPOCHS_14 = 20
PATIENCE_14 = 5

BEST_MODEL_PATH_14 = (
    "/content/experiment_14_bilstm_fasttext_best.pt"
)

best_f1_14 = -1.0
best_epoch_14 = 0

patience_counter_14 = 0
history_14 = []

start_time_14 = time.time()

for epoch in range(1, EPOCHS_14 + 1):

    epoch_start = time.time()

    train_loss = train_epoch_14(
        model_14,
        train_loader_new,
        optimizer_14,
        criterion_14
    )

    val_loss, val_f1, val_acc, _, _ = evaluate_14(
        model_14,
        val_loader_new,
        criterion_14
    )

    history_14.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": val_f1,
        "accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {val_f1:.4f} | "
        f"Accuracy: {val_acc:.4f} | "
        f"Time: {time.time() - epoch_start:.1f}s",
        flush=True
    )

    if val_f1 > best_f1_14:

        best_f1_14 = val_f1
        best_epoch_14 = epoch

        # Сохраняем на диск только лучший результат
        torch.save(
            {
                "model_state_dict": {
                    key: value.detach().cpu()
                    for key, value in model_14.state_dict().items()
                },
                "best_epoch": best_epoch_14,
                "macro_f1": best_f1_14
            },
            BEST_MODEL_PATH_14
        )

        patience_counter_14 = 0

        print(
            "  -> Новая лучшая модель сохранена",
            flush=True
        )

    else:

        patience_counter_14 += 1

        if patience_counter_14 >= PATIENCE_14:
            print("\nEarly stopping")
            break

# ============================================================
# 9. ВОССТАНАВЛИВАЕМ ЛУЧШУЮ МОДЕЛЬ
# ============================================================

checkpoint_14 = torch.load(
    BEST_MODEL_PATH_14,
    map_location="cpu",
    weights_only=True
)

model_14.load_state_dict(
    checkpoint_14["model_state_dict"]
)

del checkpoint_14
gc.collect()

# ============================================================
# 10. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

val_loss, val_f1, val_acc, y_true, y_pred = evaluate_14(
    model_14,
    val_loader_new,
    criterion_14
)

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 14 — BiLSTM + FASTTEXT")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_14)
print(f"Macro-F1: {val_f1:.4f}")
print(f"Accuracy: {val_acc:.4f}")

print(
    "Время обучения:",
    f"{(time.time() - start_time_14) / 60:.1f} мин"
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 11. СОХРАНЯЕМ РЕЗУЛЬТАТЫ
# ============================================================

pd.DataFrame(history_14).to_csv(
    "/content/experiment_14_bilstm_fasttext_history.csv",
    index=False
)

print("\nИстория обучения сохранена.")
print("Лучшая модель:", BEST_MODEL_PATH_14)

In [ ]:
import os

files = [
    "/content/merged_sentiment_dataset.csv.gz",
    "/content/cc.ru.300.vec.gz",
    "/content/fasttext_matrix_new.npy",
    "/content/experiment_13_fasttext_best.pt"
]

for path in files:
    print(
        os.path.basename(path),
        "— ЕСТЬ" if os.path.exists(path) else "— НЕТ"
    )

In [ ]:
import os

path = "/content/merged_sentiment_dataset.csv.gz"

if os.path.exists(path):
    print("Файл найден")
    print("Размер:", round(os.path.getsize(path) / 1024**2, 2), "МБ")
else:
    print("Файл не найден")

In [ ]:
import pandas as pd
import os

path = "/content/merged_sentiment_dataset.csv.gz"

try:
    df_check = pd.read_csv(
        path,
        compression="gzip",
        low_memory=False
    )

    print("Файл успешно прочитан!")
    print("Размер таблицы:", df_check.shape)
    print("Колонки:", df_check.columns.tolist())
    print("\nРаспределение по split:")
    print(df_check["split"].value_counts())

except Exception as e:
    print("Ошибка:", type(e).__name__)
    print(str(e))

In [ ]:
import os
import gzip

path = "/content/merged_sentiment_dataset.csv.gz"

print("Размер:", round(os.path.getsize(path) / 1024**2, 2), "МБ")

with open(path, "rb") as f:
    print("Первые байты:", f.read(4).hex())

try:
    with gzip.open(path, "rb") as f:
        header = f.readline()
        print("Заголовок:", header[:200].decode("utf-8", errors="replace"))
except Exception as e:
    print("Ошибка архива:", type(e).__name__, str(e))

In [ ]:
import gzip

path = "/content/merged_sentiment_dataset.csv.gz"

try:
    with gzip.open(path, "rb") as f:
        while f.read(1024 * 1024):
            pass

    print("✅ Архив целый")

except Exception as e:
    print("❌ Ошибка:", type(e).__name__)
    print(str(e))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

os.makedirs(PROJECT_DIR, exist_ok=True)

print("✅ Папка проекта создана!")
print("Путь:", PROJECT_DIR)
print("Содержимое:", os.listdir(PROJECT_DIR))

In [ ]:
import os
import gzip

path = "/content/drive/MyDrive/sentiment_research/merged_sentiment_dataset.csv.gz"

if not os.path.exists(path):
    print("❌ Файл не найден на Google Диске")
else:
    print("Файл найден")
    print("Размер:", round(os.path.getsize(path) / 1024**2, 2), "МБ")

    try:
        with gzip.open(path, "rb") as f:
            while f.read(1024 * 1024):
                pass

        print("✅ Архив целый, можно продолжать!")

    except Exception as e:
        print("❌ Архив повреждён:", type(e).__name__, str(e))

In [ ]:
# ============================================================
# ШАГ 7. ВОССТАНОВЛЕНИЕ ДАННЫХ ИЗ GOOGLE DRIVE
# ============================================================

import re
import pandas as pd
import numpy as np
import torch

DATA_PATH = (
    "/content/drive/MyDrive/sentiment_research/"
    "merged_sentiment_dataset.csv.gz"
)

# Загружаем датасет
df_new = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    low_memory=False
)

# Восстанавливаем токенизацию
def tokenize_new(text):
    return re.findall(r"[а-яёa-z0-9]+", str(text).lower())

# Разделяем выборки
train_new = df_new[df_new["split"] == "train"].copy()
val_new = df_new[df_new["split"] == "validation"].copy()

# Очищаем тексты
for data in [train_new, val_new]:
    data["text"] = data["text"].fillna("").astype(str)

train_new = train_new[
    train_new["text"].apply(
        lambda x: len(tokenize_new(x)) > 0
    )
].copy()

val_new = val_new[
    val_new["text"].apply(
        lambda x: len(tokenize_new(x)) > 0
    )
].copy()

# Восстанавливаем переменные
X_train_new = train_new["text"].tolist()
y_train_new = train_new["sentiment"].astype(int).to_numpy()

X_val_new = val_new["text"].tolist()
y_val_new = val_new["sentiment"].astype(int).to_numpy()

CLASS_NAMES_NEW = ["Neutral", "Positive", "Negative"]
MAX_LEN_NEW = 256

DEVICE_NEW = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Проверяем
print("Размер датасета:", df_new.shape)
print("Train:", len(X_train_new))
print("Validation:", len(X_val_new))
print("MAX_LEN:", MAX_LEN_NEW)
print("Устройство:", DEVICE_NEW)

In [ ]:
# ============================================================
# ШАГ 8. СЛОВАРЬ, DATASET И DATALOADER
# ============================================================

import random
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Настройки
SEED_NEW = 42
MIN_FREQ_NEW = 2
MAX_LEN_NEW = 256
BATCH_SIZE_NEW = 64

PAD_IDX_NEW = 0
UNK_IDX_NEW = 1

random.seed(SEED_NEW)
np.random.seed(SEED_NEW)
torch.manual_seed(SEED_NEW)

DEVICE_NEW = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 2. Строим словарь только на Train
word_counts_new = Counter()

for text in X_train_new:
    word_counts_new.update(tokenize_new(text))

word2idx_new = {
    "<PAD>": PAD_IDX_NEW,
    "<UNK>": UNK_IDX_NEW
}

for word, count in word_counts_new.items():
    if count >= MIN_FREQ_NEW:
        word2idx_new[word] = len(word2idx_new)

VOCAB_SIZE_NEW = len(word2idx_new)

# 3. Dataset
class SentimentDatasetNew(Dataset):

    def __init__(self, texts, labels):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.encoded = []

        for text in texts:
            tokens = tokenize_new(text)[:MAX_LEN_NEW]

            ids = [
                word2idx_new.get(token, UNK_IDX_NEW)
                for token in tokens
            ]

            self.encoded.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], int(self.labels[idx])


# 4. Динамическое дополнение текстов
def collate_new(batch):
    texts, labels = zip(*batch)

    max_batch_len = min(
        max(len(text) for text in texts),
        MAX_LEN_NEW
    )

    padded = torch.full(
        (len(texts), max_batch_len),
        PAD_IDX_NEW,
        dtype=torch.long
    )

    for i, ids in enumerate(texts):
        padded[i, :len(ids)] = torch.tensor(
            ids, dtype=torch.long
        )

    return (
        padded,
        torch.tensor(labels, dtype=torch.long)
    )


# 5. DataLoader
train_dataset_new = SentimentDatasetNew(
    X_train_new, y_train_new
)

val_dataset_new = SentimentDatasetNew(
    X_val_new, y_val_new
)

train_loader_new = DataLoader(
    train_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=True,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)

val_loader_new = DataLoader(
    val_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=False,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)

# 6. Проверка
print("Устройство:", DEVICE_NEW)
print("Размер словаря:", VOCAB_SIZE_NEW)
print("Train batches:", len(train_loader_new))
print("Validation batches:", len(val_loader_new))
print("MAX_LEN:", MAX_LEN_NEW)

X_batch, y_batch = next(iter(train_loader_new))

print("Форма текстов:", tuple(X_batch.shape))
print("Форма меток:", tuple(y_batch.shape))

In [ ]:
# ============================================================
# ШАГ 9. FASTTEXT — ВОССТАНОВЛЕНИЕ И СОХРАНЕНИЕ НА DRIVE
# ============================================================

import os
import gzip
import time
import subprocess
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

FASTTEXT_MATRIX_PATH_NEW = os.path.join(
    PROJECT_DIR, "fasttext_matrix_new.npy"
)

FASTTEXT_PATH_NEW = "/content/cc.ru.300.vec.gz"

FASTTEXT_URL_NEW = (
    "https://dl.fbaipublicfiles.com/fasttext/"
    "vectors-crawl/cc.ru.300.vec.gz"
)

FASTTEXT_DIM_NEW = 300

os.makedirs(PROJECT_DIR, exist_ok=True)

expected_shape = (VOCAB_SIZE_NEW, FASTTEXT_DIM_NEW)

# 1. Если матрица уже сохранена — используем её
if os.path.exists(FASTTEXT_MATRIX_PATH_NEW):

    matrix = np.load(
        FASTTEXT_MATRIX_PATH_NEW,
        mmap_mode="r"
    )

    if matrix.shape != expected_shape:
        raise ValueError(
            f"Неверный размер матрицы: {matrix.shape}"
        )

    print("✅ Матрица FastText найдена на Google Диске!")
    print("Размер:", matrix.shape)

    del matrix

else:

    # 2. Скачиваем архив, если его нет
    if not os.path.exists(FASTTEXT_PATH_NEW):

        print("Скачиваем FastText...")

        subprocess.run(
            [
                "wget",
                "-q",
                "--show-progress",
                "-O",
                FASTTEXT_PATH_NEW,
                FASTTEXT_URL_NEW
            ],
            check=True
        )

    print("Архив скачан.")
    print(
        "Размер:",
        round(os.path.getsize(FASTTEXT_PATH_NEW) / 1024**3, 2),
        "ГБ"
    )

    # 3. Создаём матрицу
    rng = np.random.default_rng(42)

    fasttext_matrix_new = rng.normal(
        0.0,
        0.05,
        size=expected_shape
    ).astype(np.float32)

    fasttext_matrix_new[PAD_IDX_NEW] = 0.0

    found = 0
    processed = 0
    start_time = time.time()

    print("\nИзвлекаем векторы...")

    # 4. Читаем архив постепенно
    with gzip.open(
        FASTTEXT_PATH_NEW,
        "rt",
        encoding="utf-8"
    ) as f:

        header = f.readline().strip()
        print("Заголовок:", header)

        for line in f:

            processed += 1

            word, separator, vector_text = line.partition(" ")

            if separator and word in word2idx_new:

                vector = np.fromstring(
                    vector_text,
                    sep=" ",
                    dtype=np.float32
                )

                if len(vector) == FASTTEXT_DIM_NEW:

                    idx = word2idx_new[word]
                    fasttext_matrix_new[idx] = vector
                    found += 1

            if processed % 500000 == 0:
                print(
                    f"Обработано: {processed:,} | "
                    f"Найдено: {found:,}",
                    flush=True
                )

    # 5. Сохраняем на Google Диск
    # Сначала во временный файл, затем переименовываем
    temp_path = FASTTEXT_MATRIX_PATH_NEW + ".tmp.npy"

    np.save(temp_path, fasttext_matrix_new)

    # Проверяем сохранённую матрицу
    check_matrix = np.load(
        temp_path,
        mmap_mode="r"
    )

    assert check_matrix.shape == expected_shape

    del check_matrix

    os.replace(temp_path, FASTTEXT_MATRIX_PATH_NEW)

    coverage = found / (VOCAB_SIZE_NEW - 2) * 100

    print("\n✅ FASTTEXT ГОТОВ!")
    print("Найдено слов:", found)
    print(f"Покрытие словаря: {coverage:.2f}%")
    print(
        f"Время: {(time.time() - start_time) / 60:.1f} мин"
    )

    del fasttext_matrix_new

print("\n" + "=" * 50)
print("МАТРИЦА СОХРАНЕНА")
print("=" * 50)

print("Путь:", FASTTEXT_MATRIX_PATH_NEW)
print(
    "Размер:",
    round(
        os.path.getsize(FASTTEXT_MATRIX_PATH_NEW) / 1024**2,
        1
    ),
    "МБ"
)

## Эксперимент 14. BiLSTM + FastText + CrossEntropyLoss

In [ ]:
# ============================================================
# ЭКСПЕРИМЕНТ 14. BiLSTM + FASTTEXT + CROSSENTROPY
# ============================================================

import os
import gc
import time
import random

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# ------------------------------------------------------------
# 1. Проверка данных
# ------------------------------------------------------------

required = [
    "word2idx_new", "VOCAB_SIZE_NEW", "PAD_IDX_NEW",
    "DEVICE_NEW", "train_loader_new", "val_loader_new",
    "CLASS_NAMES_NEW"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Отсутствуют переменные: " + ", ".join(missing)
    )

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

MATRIX_PATH_14 = os.path.join(
    PROJECT_DIR, "fasttext_matrix_new.npy"
)

BEST_PATH_14 = os.path.join(
    PROJECT_DIR, "experiment_14_bilstm_fasttext_best.pt"
)

HISTORY_PATH_14 = os.path.join(
    PROJECT_DIR, "experiment_14_bilstm_fasttext_history.csv"
)

if not os.path.isfile(MATRIX_PATH_14):
    raise FileNotFoundError(
        "Матрица FastText не найдена на Google Диске."
    )

fasttext_matrix_14 = np.load(
    MATRIX_PATH_14,
    mmap_mode="r"
)

assert fasttext_matrix_14.shape == (VOCAB_SIZE_NEW, 300)

print("Устройство:", DEVICE_NEW)
print("Матрица FastText:", fasttext_matrix_14.shape)

# ------------------------------------------------------------
# 2. Фиксируем seed
# ------------------------------------------------------------

SEED_14 = 42

random.seed(SEED_14)
np.random.seed(SEED_14)
torch.manual_seed(SEED_14)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_14)

# ------------------------------------------------------------
# 3. Архитектура BiLSTM
# ------------------------------------------------------------

class BiLSTMFastText14(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        hidden_dim=128,
        num_layers=1,
        dropout=0.5,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX_NEW
        )

        # Переносим векторы частями, без лишней
        # полной копии матрицы в оперативной памяти
        with torch.no_grad():
            for start in range(0, vocab_size, 10000):
                end = min(start + 10000, vocab_size)

                chunk = np.array(
                    embedding_matrix[start:end],
                    copy=True
                )

                self.embedding.weight[start:end].copy_(
                    torch.from_numpy(chunk)
                )

            self.embedding.weight[PAD_IDX_NEW].zero_()

        self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_dim * 2,
            num_classes
        )

    def forward(self, x):

        lengths = (x != PAD_IDX_NEW).sum(dim=1)
        lengths = lengths.clamp(min=1)

        embedded = self.embedding(x)

        packed = nn.utils.rnn.pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, _) = self.lstm(packed)

        features = torch.cat(
            [hidden[-2], hidden[-1]],
            dim=1
        )

        return self.fc(self.dropout(features))


# ------------------------------------------------------------
# 4. Создаём модель
# ------------------------------------------------------------

model_14 = BiLSTMFastText14(
    embedding_matrix=fasttext_matrix_14,
    hidden_dim=128,
    num_layers=1,
    dropout=0.5,
    num_classes=3
).to(DEVICE_NEW)

criterion_14 = nn.CrossEntropyLoss()

optimizer_14 = torch.optim.AdamW(
    model_14.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

print(
    "Количество параметров:",
    f"{sum(p.numel() for p in model_14.parameters()):,}"
)

# ------------------------------------------------------------
# 5. Обучение
# ------------------------------------------------------------

def train_epoch_14(model, loader, optimizer, criterion):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_NEW, non_blocking=True)
        labels = labels.to(DEVICE_NEW, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        logits = model(inputs)
        loss = criterion(logits, labels)

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ------------------------------------------------------------
# 6. Валидация
# ------------------------------------------------------------

@torch.no_grad()
def evaluate_14(model, loader, criterion):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    y_true = []
    y_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(DEVICE_NEW, non_blocking=True)
        labels = labels.to(DEVICE_NEW, non_blocking=True)

        logits = model(inputs)
        loss = criterion(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        y_true.extend(labels.cpu().numpy())
        y_pred.extend(predictions.cpu().numpy())

    return (
        total_loss / total_samples,
        f1_score(
            y_true, y_pred,
            average="macro",
            zero_division=0
        ),
        accuracy_score(y_true, y_pred),
        y_true,
        y_pred
    )


# ------------------------------------------------------------
# 7. Цикл обучения + сохранение на Drive
# ------------------------------------------------------------

EPOCHS_14 = 20
PATIENCE_14 = 5

best_f1_14 = -1.0
best_epoch_14 = 0
patience_counter_14 = 0
history_14 = []

start_time_14 = time.time()

for epoch in range(1, EPOCHS_14 + 1):

    epoch_start = time.time()

    train_loss = train_epoch_14(
        model_14,
        train_loader_new,
        optimizer_14,
        criterion_14
    )

    val_loss, val_f1, val_acc, _, _ = evaluate_14(
        model_14,
        val_loader_new,
        criterion_14
    )

    history_14.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": val_f1,
        "accuracy": val_acc
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {val_f1:.4f} | "
        f"Accuracy: {val_acc:.4f} | "
        f"Time: {time.time() - epoch_start:.1f}s",
        flush=True
    )

    # Сохраняем историю после каждой эпохи
    pd.DataFrame(history_14).to_csv(
        HISTORY_PATH_14,
        index=False
    )

    if val_f1 > best_f1_14:

        best_f1_14 = val_f1
        best_epoch_14 = epoch
        patience_counter_14 = 0

        # Временный файл защищает от неполной записи
        temp_path = BEST_PATH_14 + ".tmp"

        torch.save(
            {
                "model_state_dict": {
                    key: value.detach().cpu()
                    for key, value in model_14.state_dict().items()
                },
                "best_epoch": best_epoch_14,
                "macro_f1": best_f1_14,
                "accuracy": val_acc
            },
            temp_path
        )

        os.replace(temp_path, BEST_PATH_14)

        print(
            "  -> Новая лучшая модель сохранена на Drive",
            flush=True
        )

    else:

        patience_counter_14 += 1

        if patience_counter_14 >= PATIENCE_14:
            print("\nEarly stopping")
            break

# ------------------------------------------------------------
# 8. Восстанавливаем лучшую модель
# ------------------------------------------------------------

checkpoint_14 = torch.load(
    BEST_PATH_14,
    map_location="cpu",
    weights_only=True
)

model_14.load_state_dict(
    checkpoint_14["model_state_dict"]
)

del checkpoint_14
gc.collect()

# ------------------------------------------------------------
# 9. Итоговая оценка
# ------------------------------------------------------------

val_loss, val_f1, val_acc, y_true, y_pred = evaluate_14(
    model_14,
    val_loader_new,
    criterion_14
)

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 14 — BiLSTM + FASTTEXT")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_14)
print(f"Macro-F1: {val_f1:.4f}")
print(f"Accuracy: {val_acc:.4f}")

print(
    f"Время обучения: "
    f"{(time.time() - start_time_14) / 60:.1f} мин"
)

print("\nClassification Report:\n")

print(
    classification_report(
        y_true,
        y_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_true,
        y_pred,
        labels=[0, 1, 2]
    )
)

print("\nФайлы сохранены на Google Диске:")
print(BEST_PATH_14)
print(HISTORY_PATH_14)

## Эксперимент 15. BiLSTM + FastText + Mean Pooling

In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 15. BILSTM + FASTTEXT + MASKED MEAN POOLING
# Архитектура и настройки как в эксперименте № 7
# ============================================================

import os
import time
import random
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required_exp15 = [
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "DEVICE_NEW",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW",
    "PAD_IDX_NEW"
]

missing_exp15 = [
    name for name in required_exp15
    if name not in globals()
]

if missing_exp15:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing_exp15)
    )

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

MATRIX_PATH_EXP15 = os.path.join(
    PROJECT_DIR,
    "fasttext_matrix_new.npy"
)

BEST_PATH_EXP15 = os.path.join(
    PROJECT_DIR,
    "experiment_15_bilstm_fasttext_meanpool_best.pt"
)

HISTORY_PATH_EXP15 = os.path.join(
    PROJECT_DIR,
    "experiment_15_bilstm_fasttext_meanpool_history.csv"
)

if not os.path.isfile(MATRIX_PATH_EXP15):
    raise FileNotFoundError(
        "Матрица FastText не найдена на Google Диске."
    )

device_exp15 = torch.device(DEVICE_NEW)

print("Устройство:", device_exp15)
print("Матрица:", MATRIX_PATH_EXP15)

# ============================================================
# 2. SEED
# ============================================================

SEED_EXP15 = 42

random.seed(SEED_EXP15)
np.random.seed(SEED_EXP15)
torch.manual_seed(SEED_EXP15)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_EXP15)

# ============================================================
# 3. ЗАГРУЗКА FASTTEXT
# ============================================================

EMBED_DIM_EXP15 = 300
PAD_IDX_EXP15 = PAD_IDX_NEW

fasttext_matrix_exp15 = np.load(
    MATRIX_PATH_EXP15,
    mmap_mode="r"
)

expected_shape_exp15 = (
    VOCAB_SIZE_NEW,
    EMBED_DIM_EXP15
)

if fasttext_matrix_exp15.shape != expected_shape_exp15:
    raise ValueError(
        "Размер матрицы FastText не совпадает со словарём: "
        f"{fasttext_matrix_exp15.shape}"
    )

print("Размер матрицы:", fasttext_matrix_exp15.shape)

# ============================================================
# 4. АРХИТЕКТУРА BILSTM + FASTTEXT
# ============================================================

class BiLSTMFastTextExp15(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        hidden_size=128,
        dropout=0.4,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX_EXP15
        )

        # Загружаем FastText без лишней полной копии матрицы
        with torch.no_grad():

            for start in range(0, vocab_size, 10000):

                end = min(start + 10000, vocab_size)

                chunk = np.array(
                    embedding_matrix[start:end],
                    copy=True
                )

                self.embedding.weight[start:end].copy_(
                    torch.from_numpy(chunk)
                )

            self.embedding.weight[PAD_IDX_EXP15].zero_()

        # FastText дообучается
        self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_size * 2,
            num_classes
        )

    def forward(self, input_ids):

        # Маска непустых токенов
        mask = (input_ids != PAD_IDX_EXP15)

        embedded = self.embedding(input_ids)

        # Повторяем способ обработки из эксперимента № 7
        lstm_output, _ = self.lstm(embedded)

        # Masked Mean Pooling
        mask_float = mask.unsqueeze(-1).to(
            dtype=lstm_output.dtype
        )

        masked_output = lstm_output * mask_float

        summed = masked_output.sum(dim=1)

        lengths = mask_float.sum(dim=1).clamp(min=1.0)

        pooled = summed / lengths

        pooled = self.dropout(pooled)

        return self.fc(pooled)


# ============================================================
# 5. СОЗДАНИЕ МОДЕЛИ
# ============================================================

model_exp15 = BiLSTMFastTextExp15(
    embedding_matrix=fasttext_matrix_exp15,
    hidden_size=128,
    dropout=0.4,
    num_classes=3
).to(device_exp15)

criterion_exp15 = nn.CrossEntropyLoss()

optimizer_exp15 = torch.optim.AdamW(
    model_exp15.parameters(),
    lr=5e-4,
    weight_decay=1e-4
)

print("\nМодель BiLSTM + FastText + Mean Pooling создана!")

print(
    "Количество параметров:",
    f"{sum(p.numel() for p in model_exp15.parameters()):,}"
)

print("Функция потерь: CrossEntropyLoss")
print("Оптимизатор: AdamW")
print("Learning rate: 0.0005")
print("Dropout: 0.4")
print("Pooling: Masked Mean Pooling")

# ============================================================
# 6. ФУНКЦИЯ ОБУЧЕНИЯ
# ============================================================

def train_epoch_exp15(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(device_exp15)
        labels = labels.to(device_exp15)

        optimizer_exp15.zero_grad()

        logits = model(inputs)

        loss = criterion_exp15(logits, labels)

        loss.backward()

        # Без Gradient Clipping — как в эксперименте № 7
        optimizer_exp15.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 7. ФУНКЦИЯ ВАЛИДАЦИИ
# ============================================================

@torch.no_grad()
def evaluate_exp15(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(device_exp15)
        labels = labels.to(device_exp15)

        logits = model(inputs)

        loss = criterion_exp15(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )


# ============================================================
# 8. НАСТРОЙКИ ОБУЧЕНИЯ
# ============================================================

EPOCHS_EXP15 = 20
PATIENCE_EXP15 = 5

best_f1_exp15 = -1.0
best_epoch_exp15 = 0
best_accuracy_exp15 = 0.0

patience_counter_exp15 = 0
history_exp15 = []

start_time_exp15 = time.time()

print("\n" + "=" * 55)
print("НАЧИНАЕМ ЭКСПЕРИМЕНТ 15")
print("BiLSTM + FastText + Mean Pooling")
print("=" * 55)

# ============================================================
# 9. ОБУЧЕНИЕ
# ============================================================

for epoch in range(1, EPOCHS_EXP15 + 1):

    epoch_start = time.time()

    train_loss = train_epoch_exp15(
        model_exp15,
        train_loader_new
    )

    val_loss, y_true, y_pred = evaluate_exp15(
        model_exp15,
        val_loader_new
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_exp15.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f} | "
        f"Time: {time.time() - epoch_start:.1f}s",
        flush=True
    )

    # Сохраняем историю на Google Диск
    pd.DataFrame(history_exp15).to_csv(
        HISTORY_PATH_EXP15,
        index=False
    )

    if macro_f1 > best_f1_exp15:

        best_f1_exp15 = macro_f1
        best_epoch_exp15 = epoch
        best_accuracy_exp15 = accuracy

        patience_counter_exp15 = 0

        temp_path = BEST_PATH_EXP15 + ".tmp"

        torch.save(
            {
                "model_state_dict": {
                    key: value.detach().cpu().clone()
                    for key, value in model_exp15.state_dict().items()
                },
                "best_epoch": best_epoch_exp15,
                "macro_f1": best_f1_exp15,
                "accuracy": best_accuracy_exp15
            },
            temp_path
        )

        os.replace(temp_path, BEST_PATH_EXP15)

        print(
            "  -> Новая лучшая модель сохранена на Drive",
            flush=True
        )

    else:

        patience_counter_exp15 += 1

        if patience_counter_exp15 >= PATIENCE_EXP15:

            print("\nEarly stopping")
            break

# ============================================================
# 10. ВОССТАНОВЛЕНИЕ ЛУЧШЕЙ МОДЕЛИ
# ============================================================

checkpoint_exp15 = torch.load(
    BEST_PATH_EXP15,
    map_location="cpu",
    weights_only=True
)

model_exp15.load_state_dict(
    checkpoint_exp15["model_state_dict"]
)

del checkpoint_exp15
gc.collect()

model_exp15.eval()

# ============================================================
# 11. ФИНАЛЬНАЯ ОЦЕНКА
# ============================================================

_, y_true_exp15, y_pred_exp15 = evaluate_exp15(
    model_exp15,
    val_loader_new
)

final_f1_exp15 = f1_score(
    y_true_exp15,
    y_pred_exp15,
    average="macro",
    zero_division=0
)

final_accuracy_exp15 = accuracy_score(
    y_true_exp15,
    y_pred_exp15
)

elapsed_exp15 = (
    time.time() - start_time_exp15
) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 15 — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_exp15)
print(f"Macro-F1: {final_f1_exp15:.4f}")
print(f"Accuracy: {final_accuracy_exp15:.4f}")
print(f"Время обучения: {elapsed_exp15:.1f} мин")

print("\nClassification Report:\n")

print(
    classification_report(
        y_true_exp15,
        y_pred_exp15,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_true_exp15,
        y_pred_exp15,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 12. СРАВНЕНИЕ С ЭКСПЕРИМЕНТОМ 7
# ============================================================

f1_exp7_reference = 0.7650

difference_exp15 = final_f1_exp15 - f1_exp7_reference

print("\n" + "=" * 55)
print("СРАВНЕНИЕ С BILSTM + NAVEC")
print("=" * 55)

print(f"BiLSTM + Navec:    {f1_exp7_reference:.4f}")
print(f"BiLSTM + FastText: {final_f1_exp15:.4f}")
print(f"Разница:          {difference_exp15:+.4f}")

print("\nСохранённые файлы:")
print("Модель:", BEST_PATH_EXP15)
print("История:", HISTORY_PATH_EXP15)


In [ ]:
import os
import torch

required = [
    "word2idx_new",
    "train_loader_new",
    "val_loader_new",
    "DEVICE_NEW",
    "navec_new"
]

print("Проверка переменных:")

for name in required:
    print(
        f"{'✅' if name in globals() else '❌'} {name}"
    )

print("\nGPU доступен:", torch.cuda.is_available())

project_dir = "/content/drive/MyDrive/sentiment_research"

print("\nGoogle Диск подключён:", os.path.isdir(project_dir))

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
# ============================================================
# ШАГ 9. ВОССТАНОВЛЕНИЕ ДАТАСЕТА
# ============================================================

import os
import re
import random
import numpy as np
import pandas as pd
import torch

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

DATA_PATH = os.path.join(
    PROJECT_DIR,
    "merged_sentiment_dataset.csv.gz"
)

# Проверяем наличие файла
if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError("Датасет не найден на Google Диске!")

# Загружаем данные
df_new = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    low_memory=False
)

# Токенизация
def tokenize_new(text):
    return re.findall(r"[а-яёa-z0-9]+", str(text).lower())

# Train и Validation
train_new = df_new[df_new["split"] == "train"].copy()
val_new = df_new[df_new["split"] == "validation"].copy()

for data in (train_new, val_new):
    data["text"] = data["text"].fillna("").astype(str)

train_new = train_new[
    train_new["text"].apply(
        lambda x: len(tokenize_new(x)) > 0
    )
].copy()

val_new = val_new[
    val_new["text"].apply(
        lambda x: len(tokenize_new(x)) > 0
    )
].copy()

X_train_new = train_new["text"].tolist()
y_train_new = train_new["sentiment"].astype(int).to_numpy()

X_val_new = val_new["text"].tolist()
y_val_new = val_new["sentiment"].astype(int).to_numpy()

CLASS_NAMES_NEW = ["Neutral", "Positive", "Negative"]
MAX_LEN_NEW = 256

DEVICE_NEW = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# Проверка
print("Размер датасета:", df_new.shape)
print("Train:", len(X_train_new))
print("Validation:", len(X_val_new))
print("Устройство:", DEVICE_NEW)

In [ ]:
# ============================================================
# ШАГ 10. ВОССТАНОВЛЕНИЕ СЛОВАРЯ И DATALOADER
# ============================================================

import random
from collections import Counter

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Настройки
SEED_NEW = 42
MIN_FREQ_NEW = 2
MAX_LEN_NEW = 256
BATCH_SIZE_NEW = 64

PAD_IDX_NEW = 0
UNK_IDX_NEW = 1

random.seed(SEED_NEW)
np.random.seed(SEED_NEW)
torch.manual_seed(SEED_NEW)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_NEW)

# 2. Создаём словарь по обучающей выборке
word_counts_new = Counter()

for text in X_train_new:
    word_counts_new.update(tokenize_new(text))

word2idx_new = {
    "<PAD>": PAD_IDX_NEW,
    "<UNK>": UNK_IDX_NEW
}

for word, count in word_counts_new.items():
    if count >= MIN_FREQ_NEW:
        word2idx_new[word] = len(word2idx_new)

VOCAB_SIZE_NEW = len(word2idx_new)

# 3. Dataset
class SentimentDatasetNew(Dataset):

    def __init__(self, texts, labels):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.encoded = []

        for text in texts:
            tokens = tokenize_new(text)[:MAX_LEN_NEW]

            ids = [
                word2idx_new.get(token, UNK_IDX_NEW)
                for token in tokens
            ]

            self.encoded.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], int(self.labels[idx])


# 4. Padding внутри каждого батча
def collate_new(batch):
    texts, labels = zip(*batch)

    max_batch_len = min(
        max(len(text) for text in texts),
        MAX_LEN_NEW
    )

    padded = torch.full(
        (len(texts), max_batch_len),
        PAD_IDX_NEW,
        dtype=torch.long
    )

    for i, ids in enumerate(texts):
        padded[i, :len(ids)] = torch.tensor(
            ids, dtype=torch.long
        )

    return (
        padded,
        torch.tensor(labels, dtype=torch.long)
    )


# 5. Создаём DataLoader
train_dataset_new = SentimentDatasetNew(
    X_train_new,
    y_train_new
)

val_dataset_new = SentimentDatasetNew(
    X_val_new,
    y_val_new
)

train_loader_new = DataLoader(
    train_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=True,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)

val_loader_new = DataLoader(
    val_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=False,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)

# 6. Проверка
print("Устройство:", DEVICE_NEW)
print("Размер словаря:", VOCAB_SIZE_NEW)
print("Train batches:", len(train_loader_new))
print("Validation batches:", len(val_loader_new))

X_batch, y_batch = next(iter(train_loader_new))

print("Форма текстов:", tuple(X_batch.shape))
print("Форма меток:", tuple(y_batch.shape))

In [ ]:
# ============================================================
# ШАГ 11. NAVEC — ПОДГОТОВКА И СОХРАНЕНИЕ НА GOOGLE DRIVE
# ============================================================

import os
import sys
import subprocess
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

NAVEC_MATRIX_PATH_NEW = os.path.join(
    PROJECT_DIR,
    "navec_matrix_new.npy"
)

NAVEC_ARCHIVE_PATH_NEW = os.path.join(
    PROJECT_DIR,
    "navec_hudlit_v1_12B_500K_300d_100q.tar"
)

NAVEC_URL_NEW = (
    "https://storage.yandexcloud.net/navec/"
    "navec_hudlit_v1_12B_500K_300d_100q.tar"
)

EMBED_DIM_NEW = 300

os.makedirs(PROJECT_DIR, exist_ok=True)

# 1. Проверяем, есть ли готовая матрица
if os.path.isfile(NAVEC_MATRIX_PATH_NEW):

    matrix = np.load(
        NAVEC_MATRIX_PATH_NEW,
        mmap_mode="r"
    )

    if matrix.shape != (VOCAB_SIZE_NEW, EMBED_DIM_NEW):
        raise ValueError(
            f"Неверная форма матрицы: {matrix.shape}"
        )

    print("✅ Матрица Navec уже сохранена!")
    print("Форма:", matrix.shape)

    del matrix

else:

    # 2. Устанавливаем библиотеку Navec
    try:
        from navec import Navec
    except ImportError:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q", "navec"],
            check=True
        )
        from navec import Navec

    # 3. Скачиваем архив, если его ещё нет
    if not os.path.isfile(NAVEC_ARCHIVE_PATH_NEW):

        print("Скачиваем Navec...")

        subprocess.run(
            [
                "wget",
                "-q",
                "--show-progress",
                "-O",
                NAVEC_ARCHIVE_PATH_NEW,
                NAVEC_URL_NEW
            ],
            check=True
        )

    print("Загружаем Navec...")

    navec_new = Navec.load(NAVEC_ARCHIVE_PATH_NEW)

    # 4. Создаём матрицу
    rng = np.random.default_rng(42)

    navec_matrix_new = rng.normal(
        loc=0.0,
        scale=0.05,
        size=(VOCAB_SIZE_NEW, EMBED_DIM_NEW)
    ).astype(np.float32)

    navec_matrix_new[PAD_IDX_NEW] = 0.0

    found = 0

    for word, idx in word2idx_new.items():

        if idx == PAD_IDX_NEW:
            continue

        if word in navec_new:
            navec_matrix_new[idx] = navec_new[word]
            found += 1

    coverage = found / VOCAB_SIZE_NEW * 100

    # 5. Сохраняем на Google Диск
    temp_path = NAVEC_MATRIX_PATH_NEW + ".tmp.npy"

    np.save(temp_path, navec_matrix_new)

    check = np.load(temp_path, mmap_mode="r")

    assert check.shape == (VOCAB_SIZE_NEW, EMBED_DIM_NEW)

    del check

    os.replace(temp_path, NAVEC_MATRIX_PATH_NEW)

    print("\n✅ NAVEC ГОТОВ!")
    print("Найдено слов:", found)
    print(f"Покрытие словаря: {coverage:.2f}%")

    del navec_matrix_new

print("\n" + "=" * 50)
print("МАТРИЦА NAVEC СОХРАНЕНА")
print("=" * 50)

print("Путь:", NAVEC_MATRIX_PATH_NEW)
print(
    "Размер:",
    round(
        os.path.getsize(NAVEC_MATRIX_PATH_NEW) / 1024**2,
        1
    ),
    "МБ"
)

In [ ]:
import requests

url = (
    "https://storage.yandexcloud.net/natasha-navec/"
    "navec_hudlit_v1_12B_500K_300d_100q.tar"
)

try:
    response = requests.get(
        url,
        stream=True,
        timeout=30,
        allow_redirects=True
    )

    print("HTTP-статус:", response.status_code)
    print("Размер файла:", response.headers.get("Content-Length", "не указан"))
    print("Тип содержимого:", response.headers.get("Content-Type", "не указан"))

    response.close()

except Exception as e:
    print("Ошибка подключения:", str(e))

In [ ]:
import os
import requests

url = (
    "https://storage.yandexcloud.net/natasha-navec/packs/"
    "navec_hudlit_v1_12B_500K_300d_100q.tar"
)

path = (
    "/content/drive/MyDrive/sentiment_research/"
    "navec_hudlit_v1_12B_500K_300d_100q.tar"
)

# Скачиваем сначала во временный файл
temp_path = path + ".download"

with requests.get(url, stream=True, timeout=60) as response:
    response.raise_for_status()

    with open(temp_path, "wb") as file:
        for chunk in response.iter_content(chunk_size=1024 * 1024):
            if chunk:
                file.write(chunk)

os.replace(temp_path, path)

print("✅ Navec скачан!")
print("Размер:", round(os.path.getsize(path) / 1024**2, 2), "МБ")
print("Путь:", path)

In [ ]:
# ============================================================
# ШАГ 11.3. СОЗДАНИЕ МАТРИЦЫ NAVEC
# ============================================================

import os
import sys
import subprocess
import numpy as np

# 1. Устанавливаем Navec, если необходимо
try:
    from navec import Navec
except ImportError:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "navec"],
        check=True
    )
    from navec import Navec

# 2. Пути к файлам
PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

NAVEC_ARCHIVE_PATH_NEW = os.path.join(
    PROJECT_DIR,
    "navec_hudlit_v1_12B_500K_300d_100q.tar"
)

NAVEC_MATRIX_PATH_NEW = os.path.join(
    PROJECT_DIR,
    "navec_matrix_new.npy"
)

# 3. Проверяем необходимые переменные
required = ["word2idx_new", "VOCAB_SIZE_NEW", "PAD_IDX_NEW"]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing)
    )

if not os.path.isfile(NAVEC_ARCHIVE_PATH_NEW):
    raise FileNotFoundError("Архив Navec не найден!")

# 4. Если матрица уже существует — проверяем её
if os.path.isfile(NAVEC_MATRIX_PATH_NEW):

    matrix = np.load(NAVEC_MATRIX_PATH_NEW, mmap_mode="r")

    if matrix.shape != (VOCAB_SIZE_NEW, 300):
        raise ValueError(
            f"Неверная форма матрицы: {matrix.shape}"
        )

    print("✅ Матрица Navec уже существует!")
    print("Форма:", matrix.shape)

    del matrix

else:

    # 5. Загружаем Navec
    print("Загружаем архив Navec...")

    navec_new = Navec.load(NAVEC_ARCHIVE_PATH_NEW)

    # 6. Создаём матрицу
    rng = np.random.default_rng(42)

    navec_matrix_new = rng.normal(
        loc=0.0,
        scale=0.05,
        size=(VOCAB_SIZE_NEW, 300)
    ).astype(np.float32)

    navec_matrix_new[PAD_IDX_NEW] = 0.0

    found = 0

    print("Заполняем матрицу векторами Navec...")

    for word, idx in word2idx_new.items():

        if idx == PAD_IDX_NEW:
            continue

        if word in navec_new:
            navec_matrix_new[idx] = navec_new[word]
            found += 1

    coverage = found / VOCAB_SIZE_NEW * 100

    # 7. Сохраняем во временный файл
    temp_path = NAVEC_MATRIX_PATH_NEW + ".tmp.npy"

    np.save(temp_path, navec_matrix_new)

    # 8. Проверяем файл
    check_matrix = np.load(temp_path, mmap_mode="r")

    assert check_matrix.shape == (VOCAB_SIZE_NEW, 300)

    del check_matrix

    # 9. Переименовываем после успешного сохранения
    os.replace(temp_path, NAVEC_MATRIX_PATH_NEW)

    print("\n✅ NAVEC ГОТОВ!")
    print("Найдено слов:", found)
    print(f"Покрытие словаря: {coverage:.2f}%")

    del navec_matrix_new

# 10. Итог
print("\n" + "=" * 50)
print("МАТРИЦА NAVEC СОХРАНЕНА")
print("=" * 50)

print("Путь:", NAVEC_MATRIX_PATH_NEW)
print(
    "Размер:",
    round(
        os.path.getsize(NAVEC_MATRIX_PATH_NEW) / 1024**2,
        1
    ),
    "МБ"
)


## Эксперимент 16. BiLSTM + Navec + усиленная регуляризация

In [ ]:

# ============================================================
# ЭКСПЕРИМЕНТ 16. BILSTM + NAVEC + УСИЛЕННАЯ РЕГУЛЯРИЗАЦИЯ
# ============================================================

import os
import time
import random
import gc

import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report,
    confusion_matrix
)

# ============================================================
# 1. ПРОВЕРКА ДАННЫХ
# ============================================================

required_exp16 = [
    "word2idx_new",
    "VOCAB_SIZE_NEW",
    "PAD_IDX_NEW",
    "DEVICE_NEW",
    "train_loader_new",
    "val_loader_new",
    "CLASS_NAMES_NEW"
]

missing_exp16 = [
    name for name in required_exp16
    if name not in globals()
]

if missing_exp16:
    raise RuntimeError(
        "Не найдены переменные: " + ", ".join(missing_exp16)
    )

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

MATRIX_PATH_EXP16 = os.path.join(
    PROJECT_DIR,
    "navec_matrix_new.npy"
)

BEST_PATH_EXP16 = os.path.join(
    PROJECT_DIR,
    "experiment_16_bilstm_navec_regularized_best.pt"
)

HISTORY_PATH_EXP16 = os.path.join(
    PROJECT_DIR,
    "experiment_16_bilstm_navec_regularized_history.csv"
)

if not os.path.isfile(MATRIX_PATH_EXP16):
    raise FileNotFoundError(
        "Матрица Navec не найдена на Google Диске!"
    )

device_exp16 = torch.device(DEVICE_NEW)

# ============================================================
# 2. ФИКСИРУЕМ SEED
# ============================================================

SEED_EXP16 = 42

random.seed(SEED_EXP16)
np.random.seed(SEED_EXP16)
torch.manual_seed(SEED_EXP16)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_EXP16)

# ============================================================
# 3. ЗАГРУЖАЕМ МАТРИЦУ NAVEC
# ============================================================

navec_matrix_exp16 = np.load(
    MATRIX_PATH_EXP16,
    mmap_mode="r"
)

if navec_matrix_exp16.shape != (VOCAB_SIZE_NEW, 300):
    raise ValueError(
        f"Неверная форма матрицы: {navec_matrix_exp16.shape}"
    )

print("Устройство:", device_exp16)
print("Матрица Navec:", navec_matrix_exp16.shape)

# ============================================================
# 4. АРХИТЕКТУРА BILSTM
# ============================================================

class BiLSTMNavecExp16(nn.Module):

    def __init__(
        self,
        embedding_matrix,
        hidden_size=128,
        dropout=0.6,
        num_classes=3
    ):
        super().__init__()

        vocab_size, embedding_dim = embedding_matrix.shape

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=PAD_IDX_NEW
        )

        with torch.no_grad():

            for start in range(0, vocab_size, 10000):

                end = min(start + 10000, vocab_size)

                chunk = np.array(
                    embedding_matrix[start:end],
                    copy=True
                )

                self.embedding.weight[start:end].copy_(
                    torch.from_numpy(chunk)
                )

            self.embedding.weight[PAD_IDX_NEW].zero_()

        # Эмбеддинги дообучаются
        self.embedding.weight.requires_grad = True

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_size * 2,
            num_classes
        )

    def forward(self, input_ids):

        mask = (input_ids != PAD_IDX_NEW)

        embedded = self.embedding(input_ids)

        lstm_output, _ = self.lstm(embedded)

        # Masked Mean Pooling
        mask_float = mask.unsqueeze(-1).to(
            dtype=lstm_output.dtype
        )

        masked_output = lstm_output * mask_float

        summed = masked_output.sum(dim=1)

        lengths = mask_float.sum(dim=1).clamp(min=1.0)

        pooled = summed / lengths

        pooled = self.dropout(pooled)

        return self.fc(pooled)


# ============================================================
# 5. СОЗДАЁМ МОДЕЛЬ
# ============================================================

model_exp16 = BiLSTMNavecExp16(
    embedding_matrix=navec_matrix_exp16,
    hidden_size=128,
    dropout=0.6,
    num_classes=3
).to(device_exp16)

criterion_exp16 = nn.CrossEntropyLoss()

optimizer_exp16 = torch.optim.AdamW(
    model_exp16.parameters(),
    lr=5e-4,
    weight_decay=1e-3
)

print("\nМодель создана!")
print(
    "Количество параметров:",
    f"{sum(p.numel() for p in model_exp16.parameters()):,}"
)
print("Dropout: 0.6")
print("Weight Decay: 0.001")

# ============================================================
# 6. ОБУЧЕНИЕ
# ============================================================

def train_epoch_exp16(model, loader):

    model.train()

    total_loss = 0.0
    total_samples = 0

    for inputs, labels in loader:

        inputs = inputs.to(device_exp16)
        labels = labels.to(device_exp16)

        optimizer_exp16.zero_grad()

        logits = model(inputs)

        loss = criterion_exp16(logits, labels)

        loss.backward()
        optimizer_exp16.step()

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

    return total_loss / total_samples


# ============================================================
# 7. ВАЛИДАЦИЯ
# ============================================================

@torch.no_grad()
def evaluate_exp16(model, loader):

    model.eval()

    total_loss = 0.0
    total_samples = 0

    all_true = []
    all_pred = []

    for inputs, labels in loader:

        inputs = inputs.to(device_exp16)
        labels = labels.to(device_exp16)

        logits = model(inputs)

        loss = criterion_exp16(logits, labels)

        predictions = logits.argmax(dim=1)

        batch_size = labels.size(0)

        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_true.extend(labels.cpu().tolist())
        all_pred.extend(predictions.cpu().tolist())

    return (
        total_loss / total_samples,
        np.asarray(all_true),
        np.asarray(all_pred)
    )


# ============================================================
# 8. НАСТРОЙКИ
# ============================================================

EPOCHS_EXP16 = 20
PATIENCE_EXP16 = 5

best_f1_exp16 = -1.0
best_epoch_exp16 = 0
best_accuracy_exp16 = 0.0

patience_counter_exp16 = 0
history_exp16 = []

start_time_exp16 = time.time()

print("\n" + "=" * 55)
print("НАЧИНАЕМ ЭКСПЕРИМЕНТ 16")
print("BiLSTM + Navec + Regularization")
print("=" * 55)

# ============================================================
# 9. ЦИКЛ ОБУЧЕНИЯ
# ============================================================

for epoch in range(1, EPOCHS_EXP16 + 1):

    epoch_start = time.time()

    train_loss = train_epoch_exp16(
        model_exp16,
        train_loader_new
    )

    val_loss, y_true, y_pred = evaluate_exp16(
        model_exp16,
        val_loader_new
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        average="macro",
        zero_division=0
    )

    accuracy = accuracy_score(
        y_true,
        y_pred
    )

    history_exp16.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "macro_f1": macro_f1,
        "accuracy": accuracy
    })

    print(
        f"Epoch {epoch:02d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Macro-F1: {macro_f1:.4f} | "
        f"Accuracy: {accuracy:.4f} | "
        f"Time: {time.time() - epoch_start:.1f}s",
        flush=True
    )

    # История сохраняется после каждой эпохи
    pd.DataFrame(history_exp16).to_csv(
        HISTORY_PATH_EXP16,
        index=False
    )

    if macro_f1 > best_f1_exp16:

        best_f1_exp16 = macro_f1
        best_epoch_exp16 = epoch
        best_accuracy_exp16 = accuracy

        patience_counter_exp16 = 0

        temp_path = BEST_PATH_EXP16 + ".tmp"

        torch.save(
            {
                "model_state_dict": {
                    key: value.detach().cpu().clone()
                    for key, value in model_exp16.state_dict().items()
                },
                "best_epoch": best_epoch_exp16,
                "macro_f1": best_f1_exp16,
                "accuracy": best_accuracy_exp16
            },
            temp_path
        )

        os.replace(temp_path, BEST_PATH_EXP16)

        print(
            "  -> Лучшая модель сохранена на Google Диск",
            flush=True
        )

    else:

        patience_counter_exp16 += 1

        if patience_counter_exp16 >= PATIENCE_EXP16:
            print("\nEarly stopping")
            break

# ============================================================
# 10. ВОССТАНАВЛИВАЕМ ЛУЧШУЮ МОДЕЛЬ
# ============================================================

checkpoint_exp16 = torch.load(
    BEST_PATH_EXP16,
    map_location="cpu",
    weights_only=True
)

model_exp16.load_state_dict(
    checkpoint_exp16["model_state_dict"]
)

del checkpoint_exp16
gc.collect()

model_exp16.eval()

# ============================================================
# 11. ИТОГОВАЯ ОЦЕНКА
# ============================================================

_, y_true_exp16, y_pred_exp16 = evaluate_exp16(
    model_exp16,
    val_loader_new
)

final_f1_exp16 = f1_score(
    y_true_exp16,
    y_pred_exp16,
    average="macro",
    zero_division=0
)

final_accuracy_exp16 = accuracy_score(
    y_true_exp16,
    y_pred_exp16
)

elapsed_exp16 = (
    time.time() - start_time_exp16
) / 60

print("\n" + "=" * 55)
print("ЭКСПЕРИМЕНТ 16 — ИТОГИ")
print("=" * 55)

print("Лучшая эпоха:", best_epoch_exp16)
print(f"Macro-F1: {final_f1_exp16:.4f}")
print(f"Accuracy: {final_accuracy_exp16:.4f}")
print(f"Время обучения: {elapsed_exp16:.1f} мин")

print("\nClassification Report:\n")

print(
    classification_report(
        y_true_exp16,
        y_pred_exp16,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

print("\nConfusion Matrix:\n")

print(
    confusion_matrix(
        y_true_exp16,
        y_pred_exp16,
        labels=[0, 1, 2]
    )
)

# ============================================================
# 12. СРАВНЕНИЕ С ЭКСПЕРИМЕНТОМ 7
# ============================================================

f1_exp7_reference = 0.7650

difference_exp16 = final_f1_exp16 - f1_exp7_reference

print("\n" + "=" * 55)
print("СРАВНЕНИЕ С БАЗОВОЙ BILSTM + NAVEC")
print("=" * 55)

print(f"Эксперимент 7:  {f1_exp7_reference:.4f}")
print(f"Эксперимент 16: {final_f1_exp16:.4f}")
print(f"Разница:        {difference_exp16:+.4f}")

print("\nСохранённые файлы:")
print("Модель:", BEST_PATH_EXP16)
print("История:", HISTORY_PATH_EXP16)


In [ ]:
import os
import torch

print("GPU:", torch.cuda.is_available())

print(
    "Google Диск:",
    os.path.isdir("/content/drive/MyDrive/sentiment_research")
)

print("\nПеременные:")

for name in [
    "df_new",
    "X_train_new",
    "word2idx_new",
    "train_loader_new",
    "val_loader_new"
]:
    print(name, "✅" if name in globals() else "❌")

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

import os

project_dir = "/content/drive/MyDrive/sentiment_research"

print("\nПапка проекта найдена:", os.path.isdir(project_dir))

if os.path.isdir(project_dir):
    print("\nФайлы проекта:")
    for filename in sorted(os.listdir(project_dir)):
        print("✓", filename)

In [ ]:
# ============================================================
# ПРОВЕРКА СОХРАНЁННОГО ЭКСПЕРИМЕНТА № 16
# ============================================================

import os
import pandas as pd
import torch

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

history_path = os.path.join(
    PROJECT_DIR,
    "experiment_16_bilstm_navec_regularized_history.csv"
)

model_path = os.path.join(
    PROJECT_DIR,
    "experiment_16_bilstm_navec_regularized_best.pt"
)

# 1. Читаем историю обучения
history = pd.read_csv(history_path)

print("=" * 55)
print("ЭКСПЕРИМЕНТ № 16 — ИСТОРИЯ")
print("=" * 55)

print("\nКоличество завершённых эпох:", len(history))

print("\nРезультаты по эпохам:")
print(history.to_string(index=False))

# 2. Находим лучший результат
best_row = history.loc[history["macro_f1"].idxmax()]

print("\n" + "=" * 55)
print("ЛУЧШИЙ РЕЗУЛЬТАТ")
print("=" * 55)

print("Эпоха:", int(best_row["epoch"]))
print(f"Macro-F1: {best_row['macro_f1']:.4f}")
print(f"Accuracy: {best_row['accuracy']:.4f}")

# 3. Проверяем сохранённую модель
checkpoint = torch.load(
    model_path,
    map_location="cpu",
    weights_only=True
)

print("\nСохранённая модель:")
print("Лучшая эпоха:", checkpoint["best_epoch"])
print(f"Macro-F1: {checkpoint['macro_f1']:.4f}")
print(f"Accuracy: {checkpoint['accuracy']:.4f}")

# 4. Сравниваем с экспериментом № 7
print("\n" + "=" * 55)
print("СРАВНЕНИЕ С ЭКСПЕРИМЕНТОМ № 7")
print("=" * 55)

print("Эксперимент 7:  0.7650")
print(f"Эксперимент 16: {checkpoint['macro_f1']:.4f}")
print(f"Разница: {checkpoint['macro_f1'] - 0.7650:+.4f}")

print("\n✅ Проверка завершена!")

In [ ]:
import os
import torch

path = (
    "/content/drive/MyDrive/sentiment_research/"
    "experiment_16_bilstm_navec_regularized_best.pt"
)

checkpoint = torch.load(
    path,
    map_location="cpu",
    weights_only=True
)

print("Лучшая эпоха:", checkpoint["best_epoch"])
print(f"Macro-F1: {checkpoint['macro_f1']:.4f}")
print(f"Accuracy: {checkpoint['accuracy']:.4f}")
print("Модель сохранена:", os.path.getsize(path) > 0)

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ ОКРУЖЕНИЯ — ШАГ 1
# ДИАГНОСТИКА СОХРАНЁННОГО ДАТАСЕТА
# ============================================================

import os
import pandas as pd

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

DATA_PATH = os.path.join(
    PROJECT_DIR,
    "merged_sentiment_dataset.csv.gz"
)

print("=" * 60)
print("1. ПРОВЕРКА ФАЙЛА")
print("=" * 60)

if not os.path.isfile(DATA_PATH):
    raise FileNotFoundError(
        f"Датасет не найден: {DATA_PATH}"
    )

print("Файл найден: ✅")
print(
    "Размер:",
    round(os.path.getsize(DATA_PATH) / 1024**2, 2),
    "МБ"
)

print("\n" + "=" * 60)
print("2. ЗАГРУЗКА ДАТАСЕТА")
print("=" * 60)

df_recovery = pd.read_csv(
    DATA_PATH,
    compression="gzip"
)

print("Размерность:", df_recovery.shape)

print("\nНазвания колонок:")
for col in df_recovery.columns:
    print("-", repr(col))

print("\nТипы данных:")
print(df_recovery.dtypes.to_string())

print("\n" + "=" * 60)
print("3. ПОИСК РАЗДЕЛЕНИЯ TRAIN / VALIDATION")
print("=" * 60)

for col in df_recovery.columns:
    if any(
        word in col.lower()
        for word in ["split", "subset", "partition", "set"]
    ):
        print(f"\nКолонка: {col}")
        print(
            df_recovery[col]
            .value_counts(dropna=False)
            .head(20)
            .to_string()
        )

print("\n" + "=" * 60)
print("4. ПОИСК КОЛОНОК С МЕТКАМИ")
print("=" * 60)

for col in df_recovery.columns:
    if any(
        word in col.lower()
        for word in ["label", "sentiment", "class", "target"]
    ):
        print(f"\nКолонка: {col}")
        print(
            df_recovery[col]
            .value_counts(dropna=False)
            .head(15)
            .to_string()
        )

print("\n" + "=" * 60)
print("5. ПРИМЕРЫ ТЕКСТОВ")
print("=" * 60)

text_columns = [
    col for col in df_recovery.columns
    if any(
        word in col.lower()
        for word in ["text", "review", "content"]
    )
]

for col in text_columns[:3]:
    print(f"\nКолонка: {col}")
    for value in df_recovery[col].dropna().head(3):
        print("-", str(value)[:180])

print("\n" + "=" * 60)
print("6. ПРОВЕРКА СОХРАНЁННЫХ ФАЙЛОВ")
print("=" * 60)

files_to_check = [
    "navec_matrix_new.npy",
    "experiment_16_bilstm_navec_regularized_best.pt",
    "experiment_16_bilstm_navec_regularized_history.csv"
]

for filename in files_to_check:
    exists = os.path.isfile(
        os.path.join(PROJECT_DIR, filename)
    )
    print(
        "✅" if exists else "❌",
        filename
    )

print("\nДиагностика завершена!")

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ — ШАГ 2
# ПРОВЕРКА TRAIN И ПРОПУСКОВ
# ============================================================

import pandas as pd

df = df_recovery.copy()

train = df[df["split"] == "train"].copy()
val = df[df["split"] == "validation"].copy()

print("=" * 55)
print("1. ИСХОДНЫЕ РАЗМЕРЫ")
print("=" * 55)

print("Train:", len(train))
print("Validation:", len(val))

print("\n" + "=" * 55)
print("2. ПРОВЕРКА ПУСТЫХ ТЕКСТОВ")
print("=" * 55)

for name, data in [
    ("Train", train),
    ("Validation", val)
]:

    texts = data["text"]

    missing = texts.isna().sum()

    empty = (
        texts.fillna("")
        .astype(str)
        .str.strip()
        .eq("")
        .sum()
    )

    print(f"\n{name}:")
    print("Пропуски NaN:", missing)
    print("Пустые тексты:", empty)

print("\n" + "=" * 55)
print("3. ПРОВЕРКА ДЛИНЫ ТЕКСТОВ")
print("=" * 55)

train_lengths = (
    train["text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.len()
)

print("Текстов длиной 0:", (train_lengths == 0).sum())
print("Текстов длиной 1:", (train_lengths == 1).sum())
print("Текстов длиной 2:", (train_lengths == 2).sum())
print("Текстов короче 3 символов:", (train_lengths < 3).sum())
print("Текстов короче 5 символов:", (train_lengths < 5).sum())

print("\n" + "=" * 55)
print("4. ПРИМЕРЫ КОРОТКИХ ТЕКСТОВ")
print("=" * 55)

short_rows = train.loc[
    train_lengths < 5,
    ["text", "sentiment", "source_dataset"]
]

print(short_rows.head(25).to_string(index=False))

print("\n" + "=" * 55)
print("5. ПРОВЕРКА КЛАССОВ")
print("=" * 55)

print("\nTrain:")
print(train["sentiment"].value_counts().sort_index())

print("\nValidation:")
print(val["sentiment"].value_counts().sort_index())

print("\n✅ Диагностика завершена")

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ — ШАГ 3
# ПОИСК ПРАВИЛА ФИЛЬТРАЦИИ TRAIN
# ============================================================

import re
import pandas as pd

train_check = df_recovery.loc[
    df_recovery["split"] == "train"
].copy()

texts = train_check["text"].astype(str)

# Варианты фильтрации
checks = {
    "Исходный Train": pd.Series(True, index=texts.index),

    "Не менее 2 символов":
        texts.str.strip().str.len() >= 2,

    "Не менее 3 символов":
        texts.str.strip().str.len() >= 3,

    "Не менее 4 символов":
        texts.str.strip().str.len() >= 4,

    "Не менее 5 символов":
        texts.str.strip().str.len() >= 5,

    "Есть хотя бы одна буква":
        texts.str.contains(r"[а-яА-ЯёЁa-zA-Z]", regex=True),

    "Есть хотя бы одно слово из 2+ букв":
        texts.str.contains(
            r"[а-яА-ЯёЁa-zA-Z]{2,}",
            regex=True
        )
}

print("=" * 60)
print("ПРОВЕРКА ВАРИАНТОВ ОЧИСТКИ")
print("=" * 60)

for name, mask in checks.items():

    remaining = int(mask.sum())
    removed = len(train_check) - remaining

    print(
        f"{name:38s} "
        f"Осталось: {remaining:6d} | "
        f"Удалено: {removed:3d}"
    )

print("\nЦелевой размер Train: 203357")
print("Исходный размер Train:", len(train_check))
print("Искомое количество исключений:", len(train_check) - 203357)

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ — ШАГ 4
# ПРОВЕРКА ПУСТЫХ ТЕКСТОВ ПОСЛЕ ТОКЕНИЗАЦИИ
# ============================================================

import re

train_check = df_recovery.loc[
    df_recovery["split"] == "train"
].copy()

texts = train_check["text"].astype(str)

patterns = {
    "Русские слова (1+ символ)":
        r"[а-яё]+",

    "Русские слова (2+ символа)":
        r"[а-яё]{2,}",

    "Русские и английские слова":
        r"[а-яёa-z]+",

    "Русские и английские слова (2+)":
        r"[а-яёa-z]{2,}",

    "Слова и числа":
        r"\w+"
}

print("=" * 60)
print("ПРОВЕРКА ТОКЕНИЗАЦИИ")
print("=" * 60)

for name, pattern in patterns.items():

    empty_mask = texts.map(
        lambda text: len(
            re.findall(pattern, text.lower())
        ) == 0
    )

    removed = int(empty_mask.sum())
    remaining = len(texts) - removed

    print(
        f"{name:40s} "
        f"Осталось: {remaining:6d} | "
        f"Без токенов: {removed:3d}"
    )

print("\nЦелевой размер Train: 203357")
print("Искомое количество исключений: 17")

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ — ШАГ 5
# ПРОВЕРКА ГИПОТЕЗЫ О ТОКЕНИЗАЦИИ И СЛОВАРЕ
# ============================================================

import re
from collections import Counter

# Гипотеза о первоначальной токенизации
def tokenize_candidate(text):
    return re.findall(r"\w+", str(text).lower())

# Берём только обучающую часть
train_candidate = df_recovery.loc[
    df_recovery["split"] == "train"
].copy()

# Сохраняем исходный порядок строк
train_candidate["tokens"] = (
    train_candidate["text"].map(tokenize_candidate)
)

# Исключаем записи без токенов
train_candidate = train_candidate.loc[
    train_candidate["tokens"].map(len) > 0
].copy()

print("Количество Train:", len(train_candidate))

# Строим словарь по правилам предыдущего эксперимента
word_counts_candidate = Counter()

for tokens in train_candidate["tokens"]:
    word_counts_candidate.update(tokens)

word2idx_candidate = {
    "<PAD>": 0,
    "<UNK>": 1
}

for word, count in word_counts_candidate.items():
    if count >= 2:
        word2idx_candidate[word] = len(word2idx_candidate)

print("Размер восстановленного словаря:", len(word2idx_candidate))

print("\n" + "=" * 55)
print("СРАВНЕНИЕ С ПРЕДЫДУЩИМИ ЭКСПЕРИМЕНТАМИ")
print("=" * 55)

expected_train = 203357
expected_vocab = 251795

train_ok = len(train_candidate) == expected_train
vocab_ok = len(word2idx_candidate) == expected_vocab

print("Train совпадает:", "✅" if train_ok else "❌")
print("Словарь совпадает:", "✅" if vocab_ok else "❌")

print("\nПервые 15 элементов словаря:")
for word, idx in list(word2idx_candidate.items())[:15]:
    print(idx, repr(word))

if train_ok and vocab_ok:
    print(
        "\n✅ Оба размера совпали. "
        "Гипотеза о токенизации получила дополнительное подтверждение."
    )
else:
    print(
        "\n⚠️ Есть расхождения. "
        "Потребуется проверить исходную предобработку."
    )

print(
    "\nВажно: совпадение размеров не гарантирует "
    "совпадения индексов каждого слова."
)

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ — ШАГ 6
# ПОИСК ИСХОДНОГО ВАРИАНТА ТОКЕНИЗАЦИИ
# ============================================================

import re
from collections import Counter

texts = train_candidate["text"].astype(str).tolist()

variants = {
    "w_plus": r"\w+",
    "w_plus_2": r"\w{2,}",
    "letters_digits_no_underscore": r"[^\W_]+",
    "letters_only": r"[^\W\d_]+",
    "letters_digits_hyphen": r"[^\W_]+(?:-[^\W_]+)*",
}

EXPECTED_VOCAB = 251795
EXPECTED_TRAIN = 203357

results = []

for name, pattern in variants.items():

    counts = Counter()
    nonempty = 0

    for text in texts:
        tokens = re.findall(pattern, text.lower())

        if tokens:
            nonempty += 1
            counts.update(tokens)

    vocab_size = 2 + sum(
        count >= 2
        for count in counts.values()
    )

    results.append({
        "variant": name,
        "train": nonempty,
        "vocab": vocab_size,
        "vocab_difference": vocab_size - EXPECTED_VOCAB
    })

print("=" * 65)
print("СРАВНЕНИЕ ВАРИАНТОВ ТОКЕНИЗАЦИИ")
print("=" * 65)

for result in results:
    print(
        f"{result['variant']:30s} "
        f"Train: {result['train']:6d} | "
        f"Vocab: {result['vocab']:6d} | "
        f"Разница: {result['vocab_difference']:+d}"
    )

print("\nОжидаемый Train:", EXPECTED_TRAIN)
print("Ожидаемый словарь:", EXPECTED_VOCAB)

print(
    "\nВажно: совпадение размеров само по себе "
    "не подтверждает идентичность исходного словаря."
)

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ — ШАГ 7
# ПОИСК ИСХОДНОЙ ПРЕДОБРАБОТКИ В NOTEBOOK
# ============================================================

import os
import json
from pathlib import Path

ROOT = Path("/content/drive/MyDrive")

KEYWORDS = [
    "def tokenize_new",
    "X_train_new =",
    "X_val_new =",
    "MAX_LEN_NEW =",
    "word2idx_new ="
]

print("=" * 65)
print("ПОИСК НУЖНЫХ ЯЧЕЕК В СОХРАНЁННЫХ NOTEBOOK")
print("=" * 65)

notebooks = list(ROOT.rglob("*.ipynb"))

print("Найдено ноутбуков:", len(notebooks))

found = 0

for path in notebooks:

    try:
        with open(path, "r", encoding="utf-8") as f:
            notebook = json.load(f)
    except Exception:
        continue

    for index, cell in enumerate(notebook.get("cells", [])):

        if cell.get("cell_type") != "code":
            continue

        source = cell.get("source", [])
        code = "".join(source) if isinstance(source, list) else source

        matched = [key for key in KEYWORDS if key in code]

        if not matched:
            continue

        found += 1

        print("\n" + "=" * 65)
        print("ФАЙЛ:", path.name)
        print("ЯЧЕЙКА:", index + 1)
        print("НАЙДЕНО:", ", ".join(matched))
        print("=" * 65)

        # Ограничиваем длину вывода одной ячейки
        print(code[:7000])

        if len(code) > 7000:
            print("\n... [ОСТАЛЬНОЙ КОД СКРЫТ] ...")

print("\n" + "=" * 65)
print("Всего подходящих ячеек:", found)

if found == 0:
    print(
        "Исходные определения не найдены в доступных "
        "ноутбуках Google Drive."
    )

In [ ]:
# ============================================================
# ВОССТАНОВЛЕНИЕ — ШАГ 8
# ИСХОДНАЯ ТОКЕНИЗАЦИЯ, TRAIN, VALIDATION, СЛОВАРЬ
# ============================================================

import os
import re
import json
import random
import hashlib
from collections import Counter

import numpy as np
import pandas as pd
import torch

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"
DATA_PATH = os.path.join(
    PROJECT_DIR, "merged_sentiment_dataset.csv.gz"
)

SEED_NEW = 42
MAX_LEN_NEW = 256
MIN_FREQ_NEW = 2
BATCH_SIZE_NEW = 64
PAD_IDX_NEW = 0
UNK_IDX_NEW = 1

random.seed(SEED_NEW)
np.random.seed(SEED_NEW)
torch.manual_seed(SEED_NEW)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED_NEW)

DEVICE_NEW = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 1. Исходная функция из ноутбука
def tokenize_new(text):
    return re.findall(r"[а-яёa-z0-9]+", str(text).lower())

# 2. Загружаем данные
df_new = pd.read_csv(
    DATA_PATH,
    compression="gzip",
    low_memory=False
)

train_new = df_new.loc[df_new["split"] == "train"].copy()
val_new = df_new.loc[df_new["split"] == "validation"].copy()

for data in (train_new, val_new):
    data["text"] = data["text"].fillna("").astype(str)

# 3. Исключаем тексты без токенов
train_new = train_new.loc[
    train_new["text"].map(lambda x: len(tokenize_new(x)) > 0)
].copy()

val_new = val_new.loc[
    val_new["text"].map(lambda x: len(tokenize_new(x)) > 0)
].copy()

X_train_new = train_new["text"].tolist()
y_train_new = train_new["sentiment"].astype(int).to_numpy()

X_val_new = val_new["text"].tolist()
y_val_new = val_new["sentiment"].astype(int).to_numpy()

CLASS_NAMES_NEW = ["Neutral", "Positive", "Negative"]

# 4. Восстанавливаем словарь в исходном порядке
word_counts_new = Counter()

for text in X_train_new:
    word_counts_new.update(tokenize_new(text))

word2idx_new = {
    "<PAD>": PAD_IDX_NEW,
    "<UNK>": UNK_IDX_NEW
}

for word, count in word_counts_new.items():
    if count >= MIN_FREQ_NEW:
        word2idx_new[word] = len(word2idx_new)

VOCAB_SIZE_NEW = len(word2idx_new)

# 5. Проверяем результат
print("=" * 60)
print("ВОССТАНОВЛЕНИЕ ИСХОДНОГО ОКРУЖЕНИЯ")
print("=" * 60)

print("Train:", len(X_train_new))
print("Validation:", len(X_val_new))
print("Размер словаря:", VOCAB_SIZE_NEW)
print("MAX_LEN:", MAX_LEN_NEW)
print("Устройство:", DEVICE_NEW)

train_ok = len(X_train_new) == 203357
val_ok = len(X_val_new) == 20890
vocab_ok = VOCAB_SIZE_NEW == 251795

print("\nПРОВЕРКА:")
print("Train:", "✅" if train_ok else "❌")
print("Validation:", "✅" if val_ok else "❌")
print("Словарь:", "✅" if vocab_ok else "❌")

# 6. Дополнительная проверка по сохранённым весам
CHECKPOINT_PATH = os.path.join(
    PROJECT_DIR,
    "experiment_16_bilstm_navec_regularized_best.pt"
)

if os.path.isfile(CHECKPOINT_PATH):
    checkpoint = torch.load(
        CHECKPOINT_PATH,
        map_location="cpu",
        weights_only=True
    )

    state = checkpoint.get("model_state_dict", checkpoint)

    embedding_shape = tuple(
        state["embedding.weight"].shape
    )

    print("\nРазмер embedding в модели:", embedding_shape)

    shape_ok = embedding_shape[0] == VOCAB_SIZE_NEW
    print(
        "Совпадение размера embedding:",
        "✅" if shape_ok else "❌"
    )
else:
    shape_ok = False
    print("\n⚠️ Файл модели не найден")

# 7. Сохраняем словарь только при успешных проверках
if train_ok and val_ok and vocab_ok and shape_ok:

    vocab_path = os.path.join(
        PROJECT_DIR, "word2idx_new_recovered.json"
    )

    # Не перезаписываем существующий файл
    if not os.path.exists(vocab_path):
        with open(vocab_path, "w", encoding="utf-8") as f:
            json.dump(
                word2idx_new,
                f,
                ensure_ascii=False
            )

        print("\n✅ Восстановленный словарь сохранён:")
        print(vocab_path)
    else:
        print("\n⚠️ Файл словаря уже существует, не перезаписываем")

    print(
        "\nВсе проверки размеров пройдены. "
        "Порядок индексов восстановлен по исходному коду, "
        "но идентичность с обучением ещё не доказана."
    )

else:
    print(
        "\n⚠️ Есть несовпадения. "
        "Словарь не сохранён."
    )

In [ ]:

# ============================================================
# ШАГ 9. ВОССТАНОВЛЕНИЕ DATASET И DATALOADER
# ============================================================

import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

# 1. Проверяем восстановленные переменные
required = [
    "tokenize_new", "word2idx_new",
    "X_train_new", "y_train_new",
    "X_val_new", "y_val_new",
    "MAX_LEN_NEW", "BATCH_SIZE_NEW",
    "PAD_IDX_NEW", "UNK_IDX_NEW",
    "DEVICE_NEW", "VOCAB_SIZE_NEW"
]

missing = [name for name in required if name not in globals()]

if missing:
    raise RuntimeError(
        "Отсутствуют переменные: " + ", ".join(missing)
    )

assert VOCAB_SIZE_NEW == 251795
assert len(X_train_new) == 203357
assert len(X_val_new) == 20890

# 2. Исходный Dataset
class SentimentDatasetNew(Dataset):

    def __init__(self, texts, labels):
        self.labels = np.asarray(labels, dtype=np.int64)
        self.encoded = []

        for text in texts:
            tokens = tokenize_new(text)[:MAX_LEN_NEW]

            ids = [
                word2idx_new.get(token, UNK_IDX_NEW)
                for token in tokens
            ]

            self.encoded.append(ids)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.encoded[idx], int(self.labels[idx])


# 3. Динамический padding
def collate_new(batch):

    texts, labels = zip(*batch)

    max_batch_len = min(
        max(len(text) for text in texts),
        MAX_LEN_NEW
    )

    padded = torch.full(
        (len(texts), max_batch_len),
        PAD_IDX_NEW,
        dtype=torch.long
    )

    for i, ids in enumerate(texts):
        padded[i, :len(ids)] = torch.tensor(
            ids, dtype=torch.long
        )

    return (
        padded,
        torch.tensor(labels, dtype=torch.long)
    )


# 4. Создаём Dataset
train_dataset_new = SentimentDatasetNew(
    X_train_new,
    y_train_new
)

val_dataset_new = SentimentDatasetNew(
    X_val_new,
    y_val_new
)


# 5. Создаём DataLoader
train_loader_new = DataLoader(
    train_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=True,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)

val_loader_new = DataLoader(
    val_dataset_new,
    batch_size=BATCH_SIZE_NEW,
    shuffle=False,
    collate_fn=collate_new,
    num_workers=0,
    pin_memory=(DEVICE_NEW.type == "cuda")
)


# 6. Проверяем результат
X_batch, y_batch = next(iter(val_loader_new))

print("=" * 60)
print("DATASET И DATALOADER ВОССТАНОВЛЕНЫ")
print("=" * 60)

print("Устройство:", DEVICE_NEW)
print("Размер словаря:", VOCAB_SIZE_NEW)
print("Train samples:", len(train_dataset_new))
print("Validation samples:", len(val_dataset_new))
print("Train batches:", len(train_loader_new))
print("Validation batches:", len(val_loader_new))
print("Форма текстов:", tuple(X_batch.shape))
print("Форма меток:", tuple(y_batch.shape))
print("Метки в батче:", torch.unique(y_batch).tolist())

assert len(train_loader_new) == 3178
assert len(val_loader_new) == 327
assert X_batch.shape[0] == y_batch.shape[0]
assert X_batch.shape[1] <= MAX_LEN_NEW
assert X_batch.max().item() < VOCAB_SIZE_NEW

print("\n✅ Все проверки пройдены!")


In [ ]:

# ============================================================
# ШАГ 10. ПРОВЕРКА CHECKPOINT ЭКСПЕРИМЕНТА № 16
# BiLSTM + Navec + усиленная регуляризация
# ============================================================

import os
import gc
import numpy as np
import torch
import torch.nn as nn

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    classification_report
)

PROJECT_DIR = "/content/drive/MyDrive/sentiment_research"

CHECKPOINT_PATH = os.path.join(
    PROJECT_DIR,
    "experiment_16_bilstm_navec_regularized_best.pt"
)

assert os.path.isfile(CHECKPOINT_PATH), (
    "Checkpoint эксперимента № 16 не найден"
)

assert len(word2idx_new) == 251795
assert len(val_loader_new.dataset) == 20890

# 1. Архитектура модели № 16
class BiLSTMNavecExp16(nn.Module):

    def __init__(
        self,
        vocab_size,
        embedding_dim=300,
        hidden_size=128,
        dropout=0.6,
        num_classes=3
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            vocab_size,
            embedding_dim,
            padding_idx=PAD_IDX_NEW
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)

        self.fc = nn.Linear(
            hidden_size * 2,
            num_classes
        )

    def forward(self, input_ids):

        mask = input_ids != PAD_IDX_NEW

        embedded = self.embedding(input_ids)

        lstm_output, _ = self.lstm(embedded)

        mask_float = mask.unsqueeze(-1).to(
            dtype=lstm_output.dtype
        )

        pooled = (
            (lstm_output * mask_float).sum(dim=1)
            / mask_float.sum(dim=1).clamp(min=1.0)
        )

        return self.fc(self.dropout(pooled))


# 2. Загружаем сохранённые веса
checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=True
)

state = checkpoint.get("model_state_dict", checkpoint)

model_16 = BiLSTMNavecExp16(
    vocab_size=VOCAB_SIZE_NEW
)

model_16.load_state_dict(state, strict=True)

del checkpoint, state
gc.collect()

model_16 = model_16.to(DEVICE_NEW)
model_16.eval()

# 3. Оцениваем модель на Validation
all_true = []
all_pred = []

print("Проверяем модель № 16 на Validation...")

with torch.inference_mode():

    for batch_idx, (input_ids, labels) in enumerate(
        val_loader_new, start=1
    ):

        input_ids = input_ids.to(DEVICE_NEW)

        logits = model_16(input_ids)
        predictions = logits.argmax(dim=1)

        all_true.extend(labels.numpy().tolist())

        all_pred.extend(
            predictions.cpu().numpy().tolist()
        )

        if batch_idx % 50 == 0:
            print(
                f"Обработано батчей: "
                f"{batch_idx}/{len(val_loader_new)}"
            )

# 4. Метрики
macro_f1 = f1_score(
    all_true,
    all_pred,
    average="macro"
)

accuracy = accuracy_score(
    all_true,
    all_pred
)

print("\n" + "=" * 60)
print("ЭКСПЕРИМЕНТ № 16 — ПРОВЕРКА")
print("=" * 60)

print(f"Macro-F1: {macro_f1:.6f}")
print(f"Accuracy: {accuracy:.6f}")

print("\nClassification report:")
print(
    classification_report(
        all_true,
        all_pred,
        labels=[0, 1, 2],
        target_names=CLASS_NAMES_NEW,
        digits=4,
        zero_division=0
    )
)

# 5. Сравниваем с прежними результатами
EXPECTED_F1 = 0.763593
EXPECTED_ACCURACY = 0.777262

print("\n" + "=" * 60)
print("СРАВНЕНИЕ С СОХРАНЁННЫМИ РЕЗУЛЬТАТАМИ")
print("=" * 60)

print(f"Macro-F1 ранее: {EXPECTED_F1:.6f}")
print(f"Macro-F1 сейчас: {macro_f1:.6f}")

print(f"\nAccuracy ранее: {EXPECTED_ACCURACY:.6f}")
print(f"Accuracy сейчас: {accuracy:.6f}")

f1_ok = abs(macro_f1 - EXPECTED_F1) < 0.00001
acc_ok = abs(accuracy - EXPECTED_ACCURACY) < 0.00001

if f1_ok and acc_ok:
    print(
        "\n✅ Метрики совпали! "
        "Checkpoint успешно воспроизводит прежний результат."
    )
else:
    print(
        "\n⚠️ Метрики отличаются. "
        "Нужно проверить соответствие словаря, "
        "валидационных данных и исходной модели."
    )


## Дополнительная архитектура CNN Kim
Зависит от конфигурации второй части исходного исследования.

In [ ]:
# ============================================================
# Часть II. Импорты, конфиг, утилиты
# ============================================================
import os, time, pickle, random, warnings, subprocess, gc
from pathlib import Path
from functools import lru_cache
from collections import Counter

import numpy as np
import pandas as pd
import scipy.sparse as sp

import optuna
from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import (f1_score, accuracy_score,
                             classification_report, confusion_matrix,
                             precision_recall_fscore_support)

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# --- Безопасные fallbacks ---
SEED        = globals().get("SEED", 42)
CLASS_MAP   = globals().get("CLASS_MAP",   {0: "Нейтральный", 1: "Позитивный", 2: "Негативный"})
CLASS_ORDER = globals().get("CLASS_ORDER", [0, 1, 2])
CLASS_NAMES = globals().get("CLASS_NAMES", [CLASS_MAP[i] for i in CLASS_ORDER])
LABEL_TO_ID = {v: k for k, v in CLASS_MAP.items()}
LEADERBOARD = globals().get("LEADERBOARD", [])

# ============================================================
# Единый конфиг
# ============================================================
CONFIG = {
    # --- Общий TF-IDF для сравнимых экспериментов ---
    "min_df": 3,
    "max_features": 100_000,

    # --- Optuna LogReg ---
    "optuna_trials": 15,
    "optuna_sub": 30_000,
    "cv_folds": 3,

    # --- LightGBM ---
    "lgb_sub": 60_000,
    "lgb_svd_dim": 150,
    "lgb_optuna_trials": 12,

    # --- fastText ---
    "ft_epochs": 10,
    "ft_dim": 100,
    "ft_lr": 0.5,

    # --- CNN ---
    "cnn_sub": 60_000,
    "cnn_epochs": 3,
    "cnn_max_vocab": 50_000,
    "cnn_min_freq": 2,
    "cnn_max_len": 64,
    "cnn_emb_dim": 100,
}

print("CONFIG:", CONFIG)


def report(name, y_true, y_pred, train_time=0.0, group="—"):
    """Считает macro-F1 и accuracy, добавляет строку в LEADERBOARD."""
    f1  = f1_score(y_true, y_pred, average="macro")
    acc = accuracy_score(y_true, y_pred)
    LEADERBOARD.append({
        "группа": group, "модель": name,
        "macro-F1": round(float(f1), 4),
        "accuracy": round(float(acc), 4),
        "обучение, с": round(float(train_time), 1),
    })
    print(f"{name:50s}  macro-F1={f1:.4f}  acc={acc:.4f}  ({train_time:.1f} с)")
    return f1


for _name in ["train_df", "test_df", "y_train", "y_test"]:
    assert _name in globals(), f"Не найдено: {_name} (выполните Part I)"
assert "text_clean" in train_df.columns, "train_df должен содержать text_clean"
assert "text_clean" in test_df.columns,  "test_df должен содержать text_clean"
print(f"\nTrain: {len(train_df):,} | Validation: {len(test_df):,}")
print(f"Классы: {CLASS_NAMES}")

## 5. CNN Кима (2014)

Из вебинарa 4 (слайды 84–86): параллельные свёртки с k = 3, 4, 5 →
ReLU → max-over-time pooling → concat → dropout → линейный слой.

Ключевой рецепт из лекции: **инициализация эмбеддингов решает**.
Обучаем fastText unsupervised на train и используем его векторы как
начальную точку. У преподавателя на семинаре 5 это дало +3 п.п.
macro-F1 относительно случайной инициализации.

Обучение на подвыборке 60k, 3 эпохи. Сохраняем лучшую по val.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
# CNN Kim v4 — цель 0.75+
# Фиксы v3 → v4:
#   • Differential LR: эмбеддинги обучаются с LR=1e-4, голова с LR=2e-3
#   • Голова из двух FC-слоёв (hidden=512) — больше ёмкости
#   • n_filters 200 → 250
#   • dropout 0.6/0.3 → 0.5/0.2 (были недоучивание, не переобучение)
#   • weight_decay только для головы, эмбеддинги без WD
#   • Усреднение топ-5 чекпоинтов
#   • fastText из .bin кеша
# ═══════════════════════════════════════════════════════════════════════════

import os, time, warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm
from collections import Counter

warnings.filterwarnings("ignore")
os.environ["PYTHONWARNINGS"] = "ignore"

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.manual_seed(SEED)
if DEVICE.type == "cuda":
    torch.backends.cudnn.benchmark = True
    torch.cuda.manual_seed_all(SEED)
print(f"▶ Устройство: {DEVICE}", flush=True)


# ───────────────────────────────────────────────────────────────────────────
# ШАГ 1. Словарь
# ───────────────────────────────────────────────────────────────────────────
t0 = time.perf_counter()
counter = Counter(tok for t in train_df["text_clean"] for tok in t.split())
top_tokens = [w for w, c in counter.most_common(CONFIG["cnn_max_vocab"])
              if c >= CONFIG["cnn_min_freq"]]
PAD, UNK = "<pad>", "<unk>"
itos = [PAD, UNK] + top_tokens
stoi = {w: i for i, w in enumerate(itos)}

lengths_all = np.array([len(t.split()) for t in train_df["text_clean"]])
p50, p95, p99 = np.percentile(lengths_all, [50, 95, 99])
print(f"▶ [1/6] Словарь: {len(itos):,}  |  "
      f"длины p50={p50:.0f} p95={p95:.0f} p99={p99:.0f}  "
      f"({time.perf_counter()-t0:.1f} с)", flush=True)

HARD_CAP = 250
if CONFIG["cnn_max_len"] > HARD_CAP:
    CONFIG["cnn_max_len"] = HARD_CAP
    print(f"   cnn_max_len → {HARD_CAP}", flush=True)


def encode(text):
    ids = [stoi.get(tok, 1) for tok in text.split()][:CONFIG["cnn_max_len"]]
    return ids if ids else [1]


# ───────────────────────────────────────────────────────────────────────────
# ШАГ 2. fastText — из кеша
# ───────────────────────────────────────────────────────────────────────────
try:
    HAS_FASTTEXT
except NameError:
    try:
        import fasttext
        HAS_FASTTEXT = True
    except ImportError:
        HAS_FASTTEXT = False

emb_dim = CONFIG["cnn_emb_dim"]
emb_matrix = np.random.randn(len(itos), emb_dim).astype(np.float32) * 0.1
FT_PATH = "ft_unsup.bin"

if HAS_FASTTEXT:
    if os.path.exists(FT_PATH):
        t0 = time.perf_counter()
        ft_unsup = fasttext.load_model(FT_PATH)
        print(f"▶ [2/6] fastText из кеша ({time.perf_counter()-t0:.1f} с)",
              flush=True)
    else:
        t0 = time.perf_counter()
        FT_CORPUS_SIZE = min(150_000, len(train_df))
        ft_texts = train_df["text_clean"].sample(FT_CORPUS_SIZE,
                                                 random_state=SEED).tolist()
        with open("ft_corpus.txt", "w", encoding="utf-8") as f:
            f.write("\n".join(ft_texts))
        print(f"▶ [2/6] Корпус {FT_CORPUS_SIZE:,} текстов "
              f"({time.perf_counter()-t0:.1f} с)", flush=True)

        t0 = time.perf_counter()
        print("   Обучаю fastText (skipgram, 5 эпох):", flush=True)
        ft_unsup = fasttext.train_unsupervised(
            "ft_corpus.txt", model="skipgram", dim=emb_dim,
            epoch=5, minCount=3, minn=3, maxn=5,
            thread=4, verbose=1,
        )
        ft_unsup.save_model(FT_PATH)
        print(f"   fastText: {time.perf_counter()-t0:.0f} с → {FT_PATH}",
              flush=True)

    t0 = time.perf_counter()
    for i, w in enumerate(itos[2:], start=2):
        emb_matrix[i] = ft_unsup.get_word_vector(w)
    emb_matrix[1] = emb_matrix[2:].mean(0)
    emb_matrix[0] = 0
    print(f"   Эмбеддинги инициализированы ({time.perf_counter()-t0:.1f} с)",
          flush=True)
else:
    emb_matrix[1] = emb_matrix[2:].mean(0)
    emb_matrix[0] = 0
    print("▶ [2/6] fastText недоступен — случайная инициализация", flush=True)


# ───────────────────────────────────────────────────────────────────────────
# ШАГ 3. Dataset
# ───────────────────────────────────────────────────────────────────────────
class TextDataset(Dataset):
    def __init__(self, texts, labels):
        self.x = [encode(t) for t in texts]
        self.y = list(labels)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.x[i], self.y[i]


def collate(batch, min_len=5):
    xs, ys = zip(*batch)
    lengths = torch.tensor([len(x) for x in xs])
    L = max(int(lengths.max()), min_len)
    X = torch.zeros(len(xs), L, dtype=torch.long)
    for i, x in enumerate(xs):
        X[i, :len(x)] = torch.tensor(x)
    return X, lengths, torch.tensor(ys)


# ───────────────────────────────────────────────────────────────────────────
# ШАГ 4. Модель v4 — 2 FC слоя
# ───────────────────────────────────────────────────────────────────────────
class KimCNNv4(nn.Module):
    """Kim CNN с двумя FC-слоями. Эмбеддинги обучаются с маленьким LR."""
    def __init__(self, vocab_size, emb_dim=200, n_filters=250,
                 kernel_sizes=(2, 3, 4, 5), n_classes=3,
                 dropout=0.5, emb_dropout=0.2, hidden=512,
                 pretrained=None):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        if pretrained is not None:
            self.emb.weight.data.copy_(torch.as_tensor(pretrained))
        self.emb_drop = nn.Dropout(emb_dropout)
        self.convs = nn.ModuleList([
            nn.Conv1d(emb_dim, n_filters, k, padding=k // 2)
            for k in kernel_sizes
        ])
        self.bn = nn.BatchNorm1d(n_filters * len(kernel_sizes))
        self.drop = nn.Dropout(dropout)
        self.fc1 = nn.Linear(n_filters * len(kernel_sizes), hidden)
        self.fc2 = nn.Linear(hidden, n_classes)

    def forward(self, X):
        e = self.emb_drop(self.emb(X)).transpose(1, 2)
        pooled = [F.relu(conv(e)).max(dim=2).values for conv in self.convs]
        h = torch.cat(pooled, dim=1)
        h = self.drop(self.bn(h))
        h = F.relu(self.fc1(h))
        return self.fc2(self.drop(h))


# ───────────────────────────────────────────────────────────────────────────
# ШАГ 5. DataLoader'ы (num_workers=0 — без спама)
# ───────────────────────────────────────────────────────────────────────────
t0 = time.perf_counter()

CONFIG["cnn_sub"] = max(CONFIG["cnn_sub"], 150_000)
cnn_sub = train_df.sample(min(CONFIG["cnn_sub"], len(train_df)),
                          random_state=SEED)

train_dl = DataLoader(
    TextDataset(cnn_sub["text_clean"].values,
                y_train.loc[cnn_sub.index].values),
    batch_size=256, shuffle=True, collate_fn=collate, num_workers=0,
)

val_sub = test_df.sample(min(15_000, len(test_df)), random_state=SEED)
val_dl = DataLoader(
    TextDataset(val_sub["text_clean"].values,
                y_test.loc[val_sub.index].values),
    batch_size=512, collate_fn=collate, num_workers=0,
)
y_val_sub = y_test.loc[val_sub.index].values

test_dl = DataLoader(
    TextDataset(test_df["text_clean"].values, y_test.values),
    batch_size=512, collate_fn=collate, num_workers=0,
)
print(f"▶ [3/6] DataLoader: train={len(cnn_sub):,}, "
      f"val={len(val_sub):,}, test={len(test_df):,}  "
      f"({time.perf_counter()-t0:.1f} с)", flush=True)


# ───────────────────────────────────────────────────────────────────────────
# ШАГ 6. Обучение
# ───────────────────────────────────────────────────────────────────────────
@torch.no_grad()
def predict_proba(model, dl):
    model.eval()
    out = []
    for X, _, _ in dl:
        X = X.to(DEVICE, non_blocking=True)
        out.append(F.softmax(model(X), dim=1).cpu().numpy())
    return np.concatenate(out, axis=0)


@torch.no_grad()
def predict(model, dl):
    return predict_proba(model, dl).argmax(1)


# ── Гиперпараметры v4 ──
EPOCHS       = 20
PATIENCE     = 5
LABEL_SMOOTH = 0.05
LR_HEAD      = 2e-3      # голова
LR_EMB       = 1e-4      # эмбеддинги — в 20 раз меньше
WD_HEAD      = 5e-4      # weight decay для головы
WD_EMB       = 0.0       # без WD для эмбеддингов
MAX_GRAD     = 1.0
N_BEST       = 5

model = KimCNNv4(
    len(itos),
    emb_dim=emb_dim,
    n_filters=250,
    kernel_sizes=(2, 3, 4, 5),
    dropout=0.5,
    emb_dropout=0.2,
    hidden=512,
    pretrained=emb_matrix,
).to(DEVICE)

# ── Differential LR ──
head_params = [p for n, p in model.named_parameters() if not n.startswith("emb")]
opt = torch.optim.AdamW([
    {"params": [model.emb.weight], "lr": LR_EMB,  "weight_decay": WD_EMB},
    {"params": head_params,        "lr": LR_HEAD, "weight_decay": WD_HEAD},
])

sched = torch.optim.lr_scheduler.CosineAnnealingLR(
    opt, T_max=EPOCHS, eta_min=1e-5)
loss_fn = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTH)

total = sum(p.numel() for p in model.parameters())
n_emb = model.emb.weight.numel()
n_head = total - n_emb
print(f"▶ [4/6] Модель: {total:,} параметров "
      f"(emb={n_emb:,} lr={LR_EMB}, head={n_head:,} lr={LR_HEAD}), "
      f"эпох={EPOCHS}, patience={PATIENCE}", flush=True)

best_f1 = -1.0
best_states = []
patience_left = PATIENCE
t_train = time.perf_counter()

for ep in range(1, EPOCHS + 1):
    ep_t0 = time.perf_counter()

    model.train()
    losses = []
    pbar = tqdm(train_dl, desc=f"эпоха {ep:2d}/{EPOCHS}",
                leave=False, unit="batch")
    for X, _, y in pbar:
        X, y = X.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
        opt.zero_grad()
        loss = loss_fn(model(X), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD)
        opt.step()
        losses.append(loss.item())
        pbar.set_postfix(loss=f"{np.mean(losses[-20:]):.4f}")

    val_f1 = f1_score(y_val_sub, predict(model, val_dl), average="macro")
    ep_time = time.perf_counter() - ep_t0
    lr_now = opt.param_groups[1]["lr"]   # head LR

    marker = ""
    if val_f1 > best_f1:
        best_f1 = val_f1
        best_states.append({k: v.detach().cpu().clone()
                            for k, v in model.state_dict().items()})
        best_states = best_states[-N_BEST:]
        patience_left = PATIENCE
        marker = " ★"
    else:
        patience_left -= 1

    print(f"   эпоха {ep:2d}: loss={np.mean(losses):.4f}  "
          f"val_F1={val_f1:.4f}  lr={lr_now:.1e}  "
          f"({ep_time:.0f} с){marker}", flush=True)

    sched.step()

    if patience_left <= 0:
        print(f"   early stopping после {ep} эпох", flush=True)
        break

total_time = time.perf_counter() - t_train
print(f"▶ [5/6] Обучение: {total_time:.0f} с, best val_F1={best_f1:.4f}",
      flush=True)


# ───────────────────────────────────────────────────────────────────────────
# ШАГ 7. Усреднение топ-N чекпоинтов и финальный инференс
# ───────────────────────────────────────────────────────────────────────────
print(f"   Усредняю {len(best_states)} лучших чекпоинтов...", flush=True)
avg_state = {
    k: torch.stack([s[k].float() for s in best_states]).mean(0)
    for k in best_states[0]
}
model.load_state_dict(avg_state)

t0 = time.perf_counter()
y_pred_cnn  = predict(model, test_dl)
y_proba_cnn = predict_proba(model, test_dl)
print(f"▶ [6/6] Инференс: {time.perf_counter()-t0:.0f} с", flush=True)

report("CNN Kim v4 (fastText-init, differential LR)",
       y_test, y_pred_cnn, total_time, group="5. нейросеть")